# How Far Can Classical NLP Go? From Bag-of-Words to Stacking

**Task.** Given a sentence from a 19th-century horror story, predict its author: Edgar Allan Poe (EAP), Mary Shelley (MWS), or H.P. Lovecraft (HPL). This is the [Kaggle *Spooky Author Identification*](https://www.kaggle.com/competitions/spooky-author-identification) competition, where the official metric is **multi-class log loss**. Lower is better.

>The official Kaggle metric is **multi-class log loss**, defined as:
>
>$$
>\text{LogLoss}=-\frac{1}{N}\sum_{i=1}^{N}\sum_{j=1}^{M} y_{ij}\log(p_{ij})
>$$
>
>where $N$ is the number of sentences, $M=3$ is the number of authors, $y_{ij}$ is 1 if sentence $i$ belongs to author $j$ and 0 otherwise, and $p_{ij}$ is the predicted probability for that author.

<details>
<summary><span style="color:red">About this formula:</span></summary>

This formula is not “created randomly.” Multi-class log loss comes from **maximum likelihood estimation** for a classification problem.

Imagine the model predicts probabilities for 3 authors:

$$
p_i = [p_{i,EAP}, p_{i,MWS}, p_{i,HPL}]
$$

For one sentence $i$, the true label is one-hot encoded. Example:

$$
y_i = [1,0,0]
$$

means the true author is EAP.


## 1. Probability of the correct class

For one sentence, only the correct class matters.

If the true author is EAP, the probability assigned to the correct answer is:

$$
p_{i,EAP}
$$

If the true author is MWS, it is:

$$
p_{i,MWS}
$$

If the true author is HPL, it is:

$$
p_{i,HPL}
$$

Using one-hot notation, we can write this compactly as:
$$
P(y_i \mid x_i)=\prod_{j=1}^{M} p_{ij}^{y_{ij}}
$$

Meaning:

* $p_{ij}$ = model’s predicted probability that sentence $i$ belongs to class $j$

* $y_{ij}$ = 1 if class $j$ is the true class, otherwise 0

* $M$ = number of classes, here $3$

For our project:

$$
P(y_i \mid x_i)
=
p_{i,EAP}^{y_{i,EAP}}
p_{i,MWS}^{y_{i,MWS}}
p_{i,HPL}^{y_{i,HPL}}
$$

Why does this work?

Suppose the true label is EAP:

$$
y_i = [1,0,0]
$$

Then:

$$
P(y_i \mid x_i)
=
p_{i,EAP}^{1}
p_{i,MWS}^{0}
p_{i,HPL}^{0}
$$

Anything to the power of 0 becomes 1, so:

$$
P(y_i \mid x_i)=p_{i,EAP}
$$

So this formula simply selects the probability of the correct class.


## 2. Likelihood of the whole dataset

For $N$ independent samples, the likelihood is the product of all correct-class probabilities:

$$
L = \prod_{i=1}^{N}\prod_{j=1}^{M} p_{ij}^{y_{ij}}
$$

This means:

> How likely is the whole dataset under the probabilities predicted by the model?

A better model gives higher probability to the correct class, so it has a larger likelihood.


## 3. Take the log

Products are hard to optimize, so we take the log:

$$
\log L
=
\log \left(
\prod_{i=1}^{N}\prod_{j=1}^{M} p_{ij}^{y_{ij}}
\right)
$$

Using:

$$
\log(a b)=\log a+\log b
$$

we get:

$$
\log L
=
\sum_{i=1}^{N}\sum_{j=1}^{M} y_{ij}\log(p_{ij})
$$

This is the **log-likelihood**.

A good model should **maximize** this.

## 4. Turn it into a loss

In machine learning, we usually minimize losses instead of maximizing scores.

So we multiply by $-1$:

Then we divide by $N$ to get the average loss per sample:
$$
\text{LogLoss}
=
-\frac{1}{N}
\sum_{i=1}^{N}
\sum_{j=1}^{M}
y_{ij}\log(p_{ij})
$$

That is the multi-class log loss formula.


## Why it penalizes confident wrong predictions

Because for each row, only the correct class matters:

$$
\text{loss for one sample} = -\log(p_{\text{correct}})
$$

Examples:

If the model gives the correct author probability $0.90$:

$$
-\log(0.90)=0.105
$$

Small loss.

If it gives the correct author probability $0.50$:

$$
-\log(0.50)=0.693
$$

Bigger loss.

If it gives the correct author probability $0.01$:

$$
-\log(0.01)=4.605
$$

Very large loss.

So log loss rewards confident correct predictions and strongly punishes confident wrong predictions.

In our project, because there are 3 authors:

$$
M=3
$$

So the formula becomes:

$$
\text{LogLoss}
=
-\frac{1}{N}
\sum_{i=1}^{N}
\left[
y_{i,EAP}\log(p_{i,EAP})
+
y_{i,MWS}\log(p_{i,MWS})
+
y_{i,HPL}\log(p_{i,HPL})
\right]
$$

But since only one $y$ is 1 for each sentence, this reduces to:

$$
\text{LogLoss for one sentence}=-\log(\text{probability assigned to the true author})
$$


</details>

**Approach.** I explore how strong *classical, linear* NLP methods can be on this authorship-classification task. The project is organized into three main stages, with each stage evaluated step by step:

1. **Vowpal Wabbit:** fast online linear models using One-Against-All classification and the hashing trick, with features engineered for *authorship style*.
2. **Classical TF-IDF ensemble:** a second sparse-text pipeline using word-level and character-level TF-IDF features.
3. **Stacked generalization with fold averaging:** the strongest classical setup, where a meta-learner combines out-of-fold base-model predictions, and fold-trained models are averaged to make more stable test-set predictions.


<details>
<summary><span style="color:red">More on this:</span></summary>

**CV-bagging** means **cross-validation bagging**.

>Strictly speaking, classic bagging means:
>
>Train models on bootstrap samples, meaning samples drawn with replacement.
>
>But in this notebook, with cross-validation folds, we are not sampling with replacement. We split the data into folds and train each model on different fold combinations. So technically, calling it CV-bagging is a bit loose.
>
>A more accurate name would be:
>
>cross-validation ensembling


In simple terms, instead of training one final model only once, we train several versions of the same model on different cross-validation folds, then average their predictions for the test set.

In a 5-fold setup, this looks like:

```text
Model 1: trained on folds 2,3,4,5
Model 2: trained on folds 1,3,4,5
Model 3: trained on folds 1,2,4,5
Model 4: trained on folds 1,2,3,5
Model 5: trained on folds 1,2,3,4
```

There are two different cases to keep separate.

## 1. Training data: out-of-fold predictions

For training data, we do **not** average all fold models.

If a sentence belongs to Fold 1, we only use the model that did **not** train on Fold 1:

```text
Sentence in Fold 1 → predicted by Model 1 only
```

For example, one base model such as Logistic Regression may produce:

```text
LogReg OOF prediction: [0.70, 0.20, 0.10]
```

This vector becomes part of the **meta-learner training set**.

If we have several base models, each one gives an out-of-fold probability vector for the same training sentence:

```text
LogReg OOF:       [0.70, 0.20, 0.10]
NB-SVM OOF:       [0.65, 0.25, 0.10]
CompNB OOF:       [0.75, 0.15, 0.10]
SGD OOF:          [0.60, 0.30, 0.10]
```

These vectors are joined into one meta-feature row:

```text
[0.70, 0.20, 0.10, 0.65, 0.25, 0.10, 0.75, 0.15, 0.10, 0.60, 0.30, 0.10]
```

That row is used to train the meta-learner.

So for training data:

```text
OOF predictions create the blending/meta training set.
Each training sentence is predicted only by models that did not train on it.
```

## 2. Test data: CV-bagged predictions

For a real Kaggle test sentence, none of the fold-trained models has seen it.

So all fold-trained versions of the same base model can predict the test sentence:

```text
LogReg fold model 1: [0.70, 0.20, 0.10]
LogReg fold model 2: [0.65, 0.25, 0.10]
LogReg fold model 3: [0.72, 0.18, 0.10]
LogReg fold model 4: [0.68, 0.22, 0.10]
LogReg fold model 5: [0.69, 0.21, 0.10]
```

Then we average them:

```text
LogReg test prediction: [0.688, 0.212, 0.100]
```

This is **one base model's CV-bagged test prediction**. It is not the final stacked prediction yet.

We repeat the same process for the other base models:

```text
LogReg test prediction:   [0.688, 0.212, 0.100]
NB-SVM test prediction:   [0.740, 0.180, 0.080]
CompNB test prediction:   [0.630, 0.250, 0.120]
SGD test prediction:      [0.700, 0.190, 0.110]
```

Then these vectors are joined into one meta-feature row for the test sentence:

```text
[0.688, 0.212, 0.100, 0.740, 0.180, 0.080, 0.630, 0.250, 0.120, 0.700, 0.190, 0.110]
```

The already-trained meta-learner uses this row to produce the **final stacked prediction**.

So the key difference is:

```text
Training data:
Use out-of-fold predictions to create the meta-learner training set.

Test data:
Average fold-model predictions, then pass the averaged base-model outputs to the meta-learner.
```

In this project, **stacked generalization with CV-bagging** means:

> A meta-learner combines out-of-fold probability predictions from several base models, and cross-validation bagging is used to make the final test-set predictions more stable.

</details>


**Result.** The final stacked model reaches a level-2 OOF estimate of **0.3167 log loss**, **0.8830 accuracy**, and **0.8834 macro-F1** on the full training data. On Kaggle, it scores **0.30414 private** and **0.33621 public**. This level-2 OOF estimate is a sanity check on the meta-learner, not a full nested cross-validation estimate of the entire pipeline, so the close agreement with the leaderboard is reassuring rather than a guarantee.


<details>
<summary><span style="color:red">About "level-2":</span></summary>

“Level-2” means the **second layer** in a stacked ensemble.

In stacking, we usually have two levels:

### Level 1: base models

These are the original models trained on the text features.

In our notebook, examples are things like:

```text
Logistic Regression
NB-SVM
Complement Naive Bayes
Multinomial Naive Bayes
SGD
```

Each of them predicts probabilities like:

```text
EAP: 0.70
MWS: 0.20
HPL: 0.10
```

These are **level-1 predictions**.

### Level 2: meta-learner

The **level-2 model** does not look directly at the original text.

Instead, it looks at the predictions from the level-1 models and learns how to combine them.

For example, imagine one sentence gets these predictions:

```text
Model 1: [0.70, 0.20, 0.10]
Model 2: [0.60, 0.25, 0.15]
Model 3: [0.80, 0.10, 0.10]
```

The level-2 model takes these probability outputs as its input and learns the final prediction.

So in our case:

```text
Level 1 = classical text models
Level 2 = logistic regression meta-learner
```

That is why I called it a **level-2 OOF estimate**.

OOF, or **out-of-fold**, means that each training example is predicted by a model that **did not train on that example**. So:

> “level-2 OOF estimate” means the cross-validated estimate of the second-layer meta-learner, using out-of-fold predictions from the base models.

For a Medium article, this phrase may be too technical. You can write it more simply:

```md
The stacked ensemble reached about **0.87 accuracy** and **0.340 log loss** in an out-of-fold estimate of the meta-learner on the training data.
```

Or even smoother:

```md
The stacked ensemble reached about **0.87 accuracy** and **0.340 log loss** when I evaluated the meta-learner using out-of-fold predictions on the training data.
```

That is easier for readers than “level-2.”


----


**Out-of-fold**, or **OOF**, means that each training example is predicted by a model that **did not train on that example**.

For example, suppose we split the training data into 5 folds:

```text
Fold 1 | Fold 2 | Fold 3 | Fold 4 | Fold 5
```

For each round:

```text
Round 1: train on Folds 2–5, predict Fold 1
Round 2: train on Folds 1,3,4,5, predict Fold 2
Round 3: train on Folds 1,2,4,5, predict Fold 3
Round 4: train on Folds 1,2,3,5, predict Fold 4
Round 5: train on Folds 1–4, predict Fold 5
```

At the end, every training sentence has a prediction, but every prediction came from a model that **had not seen that sentence during training**.

That is why OOF predictions are useful. They give you training-set predictions that are much more honest than predictions made by a model on the same data it trained on.

In stacking, this matters because the second model, the **meta-learner**, needs predictions from the base models. If we gave it base-model predictions made on the same data the base models trained on, the meta-learner could learn from overly optimistic predictions and overfit.

So this sentence:

> “level-2 OOF estimate” means the cross-validated estimate of the second-layer meta-learner, using out-of-fold predictions from the base models.

means:

> The stacked model was evaluated by first creating honest out-of-fold predictions from the base models, then training/evaluating the second-layer model on those predictions through cross-validation. This gives a more realistic estimate than letting the stacker learn from predictions made on examples the base models already trained on.

In other words:

In stacking, the second-layer model should not learn from base-model predictions made on examples those base models already saw during training. To avoid that leakage, I used out-of-fold predictions: each training sentence was predicted only by base models trained on the other folds. Those out-of-fold probability predictions then became the input features for the meta-learner.

And the result sentence can become:

```md
The stacked ensemble reached about **0.87 accuracy** and **0.340 log loss** when the meta-learner was evaluated using cross-validation on out-of-fold base-model predictions from the training data.
```

</details>

>A **stacked ensemble** means we combine several models using another model.
>
>Simple idea:
>
>```text
>Base models make predictions
>↓
>A second model learns how to combine those predictions
>↓
>Final prediction
>```
>
>In our project:
>
>```text
>Level 1: Logistic Regression, NB-SVM, Naive Bayes, SGD
>Level 2: Logistic Regression meta-learner
>```
>
>So instead of manually averaging model outputs, the meta-learner learns which models to trust more.

**Key takeaway.** Several classical linear methods reached around **0.86 holdout accuracy**, and the strongest stacked submission reached roughly **0.30 to 0.35 log loss** across the holdout and Kaggle evaluations. Classical sparse-text models performed very strongly in this setup. A natural next step would be comparing this pipeline with a contextual, non-linear model, such as a fine-tuned transformer.

> *Built on the Vowpal Wabbit material from [mlcourse.ai](https://mlcourse.ai), Topic 8, debugged and extended into a complete, submitted Kaggle pipeline.*

## Setup & data

~19,600 labelled sentences (train) and ~8,400 unlabelled (test). Authors are
encoded as integers because Vowpal Wabbit's One-Against-All mode expects
**1-based** labels: `EAP=1, MWS=2, HPL=3`.

In [56]:
import os, re, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp

from sklearn.model_selection import train_test_split, StratifiedKFold, ParameterGrid
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, log_loss, confusion_matrix

# Targeted only: silence one third-party deprecation notice raised by importing
# vowpalwabbit. We do NOT suppress warnings globally, so genuine warnings surface.
warnings.filterwarnings("ignore", message="pkg_resources is deprecated")
from vowpalwabbit import Workspace

DATA_DIR = Path("data")            # input CSVs (not committed; see README)
OUTPUT_DIR = Path("outputs")       # generated VW files, models, submission
OUTPUT_DIR.mkdir(exist_ok=True)

<details>
<summary><span style="color:red">

More on `LogisticRegression`, `SGDClassifier`, `ComplementNB`, and `MultinomialNB`:</span></summary>

We’ll explain the 4 models in the context of our notebook:

```python
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import ComplementNB, MultinomialNB
```

Assume each sentence becomes a feature vector:

$$
x_i = [x_{i1}, x_{i2}, ..., x_{id}]
$$

where $d$ is the number of text features, such as words, bigrams, or character n-grams.

The label is one of three authors:

$$
y_i \in {\text{EAP}, \text{MWS}, \text{HPL}}
$$

So we have:

$$
K = 3
$$

classes.


# 1. Logistic Regression

Despite the name, **Logistic Regression is a classification model**.

In text classification, it learns one weight vector per class.

For class $c$, the model computes a score:

$$
z_c = w_c^T x + b_c
$$

where:

* $x$ = feature vector of the sentence
* $w_c$ = weights for class $c$
* $b_c$ = bias/intercept for class $c$

For multiclass classification, these scores are converted into probabilities using **softmax**:

$$
P(y=c \mid x)
=

\frac{e^{z_c}}
{\sum_{k=1}^{K} e^{z_k}}
$$

So for our project:

$$
P(\text{EAP} \mid x),\quad
P(\text{MWS} \mid x),\quad
P(\text{HPL} \mid x)
$$

The predicted class is the one with the highest probability:

$$
\hat{y}
=

\arg\max_c P(y=c \mid x)
$$

## Loss function

Logistic Regression is trained by minimizing **cross-entropy loss**, which is the same basic idea as multiclass log loss:

$$
L
=

-\frac{1}{N}
\sum_{i=1}^{N}
\sum_{c=1}^{K}
y_{ic}\log P(y=c \mid x_i)
$$

With L2 regularization, the objective becomes:

$$
J(W)
=

-\frac{1}{N}
\sum_{i=1}^{N}
\sum_{c=1}^{K}
y_{ic}\log P(y=c \mid x_i)
+
\lambda \sum_{c=1}^{K} \lVert w_c \rVert_2^2
$$

In sklearn, the parameter `C` controls regularization:

$$
C \uparrow \Rightarrow \text{weaker regularization}
$$

$$
C \downarrow \Rightarrow \text{stronger regularization}
$$

## Intuition

Logistic Regression learns which words or character patterns push a sentence toward Poe, Shelley, or Lovecraft.

For example, if a feature is very associated with HPL, its HPL weight becomes large.


# 2. SGDClassifier

`SGDClassifier` is not one specific model like `LogisticRegression` or` MultinomialNB`. It is a flexible linear classifier whose exact model depends on the chosen loss function, and it is trained using `stochastic gradient descent`.

The model still has a linear score:

$$
z_c = w_c^T x + b_c
$$

But instead of solving the optimization problem all at once, SGD updates the weights little by little using one example or one mini-batch at a time.

General objective:

$$
J(w)
=
\frac{1}{N}
\sum_{i=1}^{N}
\ell(y_i, f(x_i))
+
\lambda \lVert w \rVert_2^2
$$

where:

* $\ell$ = loss function
* $f(x_i)=w^Tx_i+b$
* $\lambda$ = regularization strength

The SGD update is:

$$
w
\leftarrow
w
-

\eta
\nabla_w J(w)
$$

where:

* $\eta$ = learning rate
* $\nabla_w J(w)$ = gradient of the loss with respect to the weights

## If `loss="log_loss"`

Then `SGDClassifier` behaves like Logistic Regression trained with SGD.

Binary logistic loss:

$$
\ell(y, z)
=
\log(1+e^{-yz})
$$

where:

$$
y \in {-1,+1}
$$

For multiclass, sklearn usually handles this with one-vs-rest classifiers.

> For multiclass classification, `SGDClassifier`uses one-vs-rest. `LogisticRegression` usually uses multinomial softmax with solvers like `lbfgs`, `sag`, `saga`, or `newton-cg`, but with `liblinear` it uses one-vs-rest.

The probability for one class can be written as (using sigmoid):

$$
P(y=1 \mid x)
=
\frac{1}{1+e^{-z}}
$$ 

where:

$$
z = w^Tx+b
$$

## If `loss="hinge"`

Then `SGDClassifier` behaves like a linear SVM.

The hinge loss is:

$$
\ell(y,z)
=
\max(0, 1-yz)
$$

This does not naturally produce probabilities. It focuses on separating classes with a margin.

## Intuition

`SGDClassifier` is useful for text because text matrices are usually huge and sparse. SGD can train fast without loading everything into dense memory.

In our notebook, it acts as another linear model whose predictions can be included in the stacked ensemble.


# 3. Multinomial Naive Bayes

`MultinomialNB` is a probabilistic model often used for text classification.

It assumes that each document is generated by a class-specific word distribution.

For each class $c$, it estimates:

$$
P(y=c)
$$

and:

$$
P(x_j \mid y=c)
$$

where $x_j$ is feature $j$, such as a word or n-gram.

Using Bayes’ rule:

$$
P(y=c \mid x)
=
\frac{P(y=c)P(x \mid y=c)}
{P(x)}
$$

* $x$: the input sentence, represented as features such as word counts or TF-IDF values. So $x$ is the sentence after it has been converted into numerical text features.

* $y=c$: the event that the sentence belongs to class/author $c$, for example EAP, MWS, or HPL.

* $P(y=c \mid x)$: the probability that the author is $c$ **after seeing the sentence**. This is what we want to predict.

* $P(y=c)$: the prior probability of class $c$, before seeing the sentence. Example: how common EAP is in the training set.

* $P(x \mid y=c)$: the likelihood. It means: how likely this sentence/features would be if the author were $c$.

* $P(x)$: the evidence. It means the overall probability of seeing this sentence/features. For classification, it is usually ignored because it is the same for all classes.

So for choosing the class:

$$
P(y=c \mid x) \propto P(y=c)P(x \mid y=c)
$$

because $P(x)$ does not change across authors.



Naive Bayes assumes features are conditionally independent given the class:

$$
P(x \mid y=c)
=

\prod_{j=1}^{d}
P(x_j \mid y=c)^{x_j}
$$

So:

$$
P(y=c \mid x)
\propto
P(y=c)
\prod_{j=1}^{d}
\theta_{cj}^{x_j}
$$

where:

$$
\theta_{cj}=P(x_j \mid y=c)
$$

In practice, we use logs:

$$
\log P(y=c \mid x)
=
\log P(y=c)
+
\sum_{j=1}^{d}
x_j \log \theta_{cj}
+
\text{constant}
$$

The predicted class is:

$$
\hat{y}
=
\arg\max_c
\left[
\log P(y=c)
+
\sum_{j=1}^{d}
x_j \log \theta_{cj}
\right]
$$

## Parameter estimation

The class prior is estimated as:

$$
P(y=c)
=
\frac{N_c}{N}
$$

where:

* $N_c$ = number of training examples in class $c$
* $N$ = total number of training examples

The feature probability is estimated with smoothing:

$$
\theta_{cj}
=
\frac{N_{cj}+\alpha}
{\sum_{k=1}^{d}N_{ck}+\alpha d}
$$

where:

* $N_{cj}$ = total count of feature $j$ in class $c$
* $\alpha$ = smoothing parameter
* $d$ = number of features

## Why smoothing?

Without smoothing, if a word never appears in a class, we get:

$$
\theta_{cj}=0
$$

Then the whole probability becomes zero.

Smoothing prevents this.

## Intuition

MultinomialNB asks:

> Which author’s word distribution makes this sentence most likely?

It is simple, fast, and often surprisingly strong for text classification.

# 4. Complement Naive Bayes

`ComplementNB` is a variation of Naive Bayes designed to work better for text classification, especially when classes are imbalanced.

Instead of estimating feature weights from class $c$, it estimates them from the **complement of class $c$**.

That means:

$$
\text{complement of class } c
=
\text{all documents not in class } c
$$

For example:

$$
\text{Complement of EAP}
=
\text{MWS + HPL documents}
$$

## Complement feature counts

For class $c$, define complement count:

$$
\tilde{N}_{cj}
=
\alpha
+
\sum_{i:y_i \neq c}
x_{ij}
$$

This counts feature $j$ in all documents **not belonging to class $c$**.

Then normalize:

$$
\tilde{\theta}_{cj}
=
\frac{\tilde{N}_{cj}}
{\sum_{k=1}^{d}\tilde{N}_{ck}}
$$

Then the complement weight is usually based on the negative log:

$$
w_{cj}
=
-\log \tilde{\theta}_{cj}
$$

So if a word is common in the complement of class $c$, it gets a smaller weight for class $c$.

If a word is rare outside class $c$, it gets a larger weight for class $c$.

The class score is:

$$
s_c(x)
=
\log P(y=c)
+
\sum_{j=1}^{d}
x_j w_{cj}
$$

The predicted class is:

$$
\hat{y}
=
\arg\max_c s_c(x)
$$

Some versions also normalize the weight vector:
$$
w_{cj}
\leftarrow
\frac{w_{cj}}
{\sum_{k=1}^{d}|w_{ck}|}
$$

## Intuition

MultinomialNB asks:

> Which class makes this sentence likely?

ComplementNB asks something closer to:

> Which class’s opposite does this sentence least look like?

For text data, this can work better because it reduces some problems caused by class imbalance and unstable word counts.


# Quick comparison

| Model                | Type                                | Main idea                              | Formula core                                                |
| -------------------- | ----------------------------------- | -------------------------------------- | ----------------------------------------------------------- |
| `LogisticRegression` | Discriminative linear classifier    | Learns direct class probabilities      | $P(y=c\mid x)=\frac{e^{w_c^Tx+b_c}}{\sum_k e^{w_k^Tx+b_k}}$ |
| `SGDClassifier`      | Linear model trained with SGD       | Optimizes a chosen loss step by step   | $w \leftarrow w-\eta\nabla J(w)$                            |
| `MultinomialNB`      | Generative probabilistic classifier | Uses class-specific word probabilities | $P(y=c\mid x)\propto P(y=c)\prod_j\theta_{cj}^{x_j}$        |
| `ComplementNB`       | Naive Bayes variant                 | Uses statistics from all other classes | $w_{cj}=-\log\tilde{\theta}_{cj}$                           |


In our project:  

These models are useful together because they make different kinds of mistakes.

* `LogisticRegression` is a strong linear baseline.
* `SGDClassifier` gives another fast linear view of the data.
* `MultinomialNB` is simple and probabilistic.
* `ComplementNB` is often stronger than plain Naive Bayes for text.

Then stacking combines their probability outputs so the meta-learner can decide how much to trust each one.


</details>


In [57]:
train_texts = pd.read_csv(DATA_DIR / "train.csv", index_col="id")
test_texts = pd.read_csv(DATA_DIR / "test.csv", index_col="id")

AUTHOR_CODE = {"EAP": 1, "MWS": 2, "HPL": 3}            # VW OAA needs 1-based labels
train_texts["author_code"] = train_texts["author"].map(AUTHOR_CODE)

print(f"Train: {len(train_texts)} sentences   Test: {len(test_texts)} sentences")
print("\nClass balance:")
print(train_texts["author"].value_counts(normalize=True).round(3))
train_texts.head()

Train: 19579 sentences   Test: 8392 sentences

Class balance:
author
EAP    0.403
MWS    0.309
HPL    0.288
Name: proportion, dtype: float64


,text,author,author_code
id,,,
id26305,"This process, however, afforded me no means of...",EAP,1
id17569,It never once occurred to me that the fumbling...,HPL,3
id11008,"In his left hand was a gold snuff box, from wh...",EAP,1
id27763,How lovely is spring As we looked from Windsor...,MWS,2
id12958,"Finding nothing else, not even gold, the Super...",HPL,3


In [58]:
train_texts.iloc[0, 0]

'This process, however, afforded me no means of ascertaining the dimensions of my dungeon; as I might make its circuit, and return to the point whence I set out, without being aware of the fact; so perfectly uniform seemed the wall.'

In [59]:
train_texts['text'].iloc[0]   # inspect the first training sentence


'This process, however, afforded me no means of ascertaining the dimensions of my dungeon; as I might make its circuit, and return to the point whence I set out, without being aware of the fact; so perfectly uniform seemed the wall.'

### Stratified train / validation split

A single stratified 70/30 split keeps class proportions stable and gives every
model the **same** held-out set, so their scores are directly comparable. 

> Optionally, cross-validation can be used for more robust performance estimation, but we prioritize a simple split due to the large-scale and streaming nature of Vowpal Wabbit.

In [60]:
train_texts_part, valid_texts = train_test_split(
    train_texts, test_size=0.3, random_state=17, stratify=train_texts["author_code"]
)
y_part = train_texts_part["author_code"].values # (13705,)
y_valid = valid_texts["author_code"].values # (5874,)

## 1. Vowpal Wabbit: online linear learning + the hashing trick

Vowpal Wabbit trains linear models by **online SGD** and hashes features into a
fixed space (no vocabulary to store), which makes it very fast and a natural fit
for sparse text. We use One-Against-All (`oaa=3`) with logistic loss.

>Vowpal Wabbit, often shortened to VW, is a fast machine-learning library designed for large-scale sparse learning. In this project, I used it as a linear text classifier: the text features are hashed into a fixed feature space, and the model learns through online updates.

> Vowpal Wabbit works well for this kind of task because text can be represented as a large sparse feature space. Each sentence activates only a small subset of word, n-gram, punctuation, and character features, and a linear model can learn useful weights for those signals efficiently. VW is especially fast because it uses online learning and feature hashing, so it can handle many sparse text features without building a huge explicit vocabulary.

>VW is mainly used as a fast sparse linear learner, but nonlinear behavior can still be introduced through feature engineering, such as n-grams, character n-grams, and feature interactions. In that sense, the model remains linear in the transformed feature space, but the features themselves can capture richer patterns from the original text.

### Baseline: word features

Lowercase, keep words of 3+ characters, one `|text` namespace.

In [61]:
def to_vw_words(df, is_train=True):
    """VW line: '<label> |text <words of 3+ chars>'."""
    lines = []
    for i in range(len(df)):
        label = df["author_code"].iloc[i] if is_train else 1
        text = df["text"].iloc[i].lower().replace("|", "").replace(":", "")
        words = " ".join(re.findall(r"\w{3,}", text))
        lines.append(f"{label} |text {words}\n")
    return lines

for fname, part in [("train_words.vw", train_texts_part), ("valid_words.vw", valid_texts)]:
    with open(f"{OUTPUT_DIR}/{fname}", "w") as f:
        f.writelines(to_vw_words(part))

<details>
<summary><span style="color:red">About this cell:</span></summary>

This code converts our dataframe into **Vowpal Wabbit format** and saves two `.vw` text files: one for training and one for validation.

Vowpal Wabbit expects each training example in this style:

```text
<label> |namespace feature1 feature2 feature3
```

For example:

```text
1 |text dark night house fear
```

Here:

* `1` is the class label
* `|text` is the feature namespace
* `dark night house fear` are the text features


## Function explanation

```python
def to_vw_words(df, is_train=True):
```

This defines a function that takes a dataframe and converts each row into a VW-formatted line.

* `df` = dataframe containing text and labels
* `is_train=True` = if it is training/validation data, use real labels
* if `is_train=False`, use a dummy label


```python
"""VW line: '<label> |text <words of 3+ chars>'."""
```

This comment says the output line will look like:

```text
<label> |text words
```

and it only keeps words with **3 or more characters**.


```python
lines = []
```

Creates an empty list to store all VW lines.


```python
for i in range(len(df)):
```

Loops through every row in the dataframe.


```python
label = df["author_code"].iloc[i] if is_train else 1
```

If this is training data, it gets the real label from:

```python
df["author_code"]
```

For example:

```text
EAP → 1
MWS → 2
HPL → 3
```

If this is test data, there is no true label, so it uses dummy label `1`.

VW still expects a label in the file format, even for prediction data.


```python
text = df["text"].iloc[i].lower().replace("|", "").replace(":", "")
```

This gets the sentence text and cleans it a little:

* `.lower()` makes everything lowercase
* `.replace("|", "")` removes `|`
* `.replace(":", "")` removes `:`

This matters because in VW format, symbols like `|` and `:` have special meanings. Removing them prevents formatting problems.


```python
words = " ".join(re.findall(r"\w{3,}", text))
```

This extracts words with **at least 3 word characters**.

The regex:

```python
r"\w{3,}"
```

means:

```text
find sequences of letters/numbers/underscore with length 3 or more
```

So from:

```text
"It was a dark night."
```

it keeps:

```text
was dark night
```

It removes:

```text
it
a
```

because they are shorter than 3 characters.

Then `" ".join(...)` turns the list of words back into one string.


```python
lines.append(f"{label} |text {words}\n")
```

This creates one VW line.

Example:

```text
1 |text dark night house fear
```

Then it adds that line to the `lines` list.


```python
return lines
```

Returns all VW-formatted lines.


## File-writing part

```python
for fname, part in [("train_words.vw", train_texts_part), ("valid_words.vw", valid_texts)]:
```

This loops over two datasets:

```text
train_texts_part → train_words.vw
valid_texts → valid_words.vw
```

So it will create:

```text
outputs/train_words.vw
outputs/valid_words.vw
```


```python
with open(f"{OUTPUT_DIR}/{fname}", "w") as f:
```

Opens a file for writing.

For example:

```python
outputs/train_words.vw
```

```python
f.writelines(to_vw_words(part))
```

Converts the dataframe to VW format and writes all lines into the file.


## In simple words

This code takes our text data like this:

```text
author_code = 1
text = "The dark night was silent."
```

and converts it into this:

```text
1 |text the dark night was silent
```

Then it saves it into a `.vw` file so Vowpal Wabbit can train or validate on it.

</details>


In [62]:
# Gotcha: VW honours `passes`/`cache` only when it reads the data file itself.
# With the streaming Python API you must loop yourself for real multiple passes,
# otherwise VW silently saves an empty model that just predicts the majority class.
N_PASSES = 10

vw = Workspace(oaa=3, loss_function="logistic", ngram=2, b=28, quiet=True,
               final_regressor=f"{OUTPUT_DIR}/spooky_words.vw")
for _ in range(N_PASSES):
    with open(f"{OUTPUT_DIR}/train_words.vw") as f:
        for line in f:
            vw.learn(line)
vw.finish() 

vw = Workspace(quiet=True, i=f"{OUTPUT_DIR}/spooky_words.vw")
pred_words = [vw.predict(line) for line in open(f"{OUTPUT_DIR}/valid_words.vw")]
vw.finish()   # free VW's internal resources


print(f"word-only Vowpal Wabbit baseline accuracy: {accuracy_score(y_valid, pred_words):.4f}"
      f"   macro-F1: {f1_score(y_valid, pred_words, average='macro'):.4f}")


word-only Vowpal Wabbit baseline accuracy: 0.8332   macro-F1: 0.8323


<details>
<summary><span style="color:red">About this cell:</span></summary>

This code trains a **baseline Vowpal Wabbit text classifier** using only word features, then evaluates it on the validation set.

## Code explanation

```python
N_PASSES = 10
```

We set the number of training passes to 10. That means VW will see the training data 10 times.

This matters because, when using the Python streaming API, VW does **not automatically repeat the data**. We must manually loop over the file.


```python
vw = Workspace(
    oaa=3,
    loss_function="logistic",
    ngram=2,
    b=28,
    quiet=True,
    final_regressor=f"{OUTPUT_DIR}/spooky_words.vw"
)
```

This creates a Vowpal Wabbit model.

* `oaa=3` means **one-against-all classification** with 3 classes: EAP, MWS, HPL.
* `loss_function="logistic"` means VW uses logistic loss.
* `ngram=2` means it uses unigrams and bigrams (single words + two-word combinations).
* `b=28` controls the size of the hashing space. VW stores features using hashing, so this gives it a large (hashed) feature space.
* `quiet=True` hides VW training logs.
* `final_regressor=...` tells VW where to save the trained model.


```python
for _ in range(N_PASSES):
    with open(f"{OUTPUT_DIR}/train_words.vw") as f:
        for line in f:
            vw.learn(line)
```

This is the actual training loop.

Each line in `train_words.vw` looks like:

```text
<label> |text word1 word2 word3
```

For each pass, the code opens the training file and feeds every line to VW using:

```python
vw.learn(line)
```

Because this loop runs 10 times, the model trains for 10 passes.


```python
vw.finish()
```

This finishes training and saves the model to:

```text
spooky_words.vw
```

```python
vw = Workspace(quiet=True, i=f"{OUTPUT_DIR}/spooky_words.vw")
```

This loads the saved VW model for prediction.

The `i=...` argument means “load this existing model file.”


```python
pred_words = [vw.predict(line) for line in open(f"{OUTPUT_DIR}/valid_words.vw")]
```

This makes predictions on the validation set.

Each validation sentence is converted into VW format, passed to the model, and VW predicts one of the three author labels.


```python
vw.finish()
```

This closes the VW workspace after prediction.

> It is good practice because VW uses internal resources, and `finish()` tells VW that we are done using the model so it can free those resources.  


```python
print(f"Baseline VW (words only)   accuracy: {accuracy_score(y_valid, pred_words):.4f}")
```

This compares VW’s predicted labels with the true validation labels and prints the accuracy.

Our result was:

```text
Baseline VW (words only) accuracy: 0.8332
```

So the model correctly classified about:

83.32%

of the validation sentences.

</details>

The word-only Vowpal Wabbit baseline reached **0.8332 accuracy**, which is a strong result for a fast linear model using only simple word features. This suggests that even basic word and bigram patterns contain useful authorship information.

However, as we will see, this result is still below the richer models in the notebook, especially the versions that add character n-grams, punctuation, TF-IDF features, or stacking. The main takeaway is:

> Word-only VW is a strong and fast baseline, but authorship identification benefits from richer style-based features beyond plain words.

### Authorship-aware features

Authorship is about **style**, not topic, so we add exactly the signals
stylometry relies on, across three namespaces:

- `|w`: **all** words, *including* short function words (`the`, `of`, `I`...),
  the classic stylometric fingerprint;
- `|p`: **punctuation** marks (comma / semicolon / dash habits);
- `|c`: **character n-grams** (2–4) on case- and punctuation-preserving text.

VW adds word bigrams on top (`ngram=2`).



In [63]:
def char_ngrams(text, ns=(2, 3, 4)):
    """Boundary-aware character n-grams; whitespace/edges become '_'."""
    t = "_" + re.sub(r"\s+", "_", text.strip()) + "_"
    return [t[i:i + n] for n in ns for i in range(len(t) - n + 1)]

def to_vw_rich(df, is_train=True, char_ns=(2, 3, 4)):
    """Three namespaces: |w words (incl. function words), |p punctuation, |c char n-grams."""
    lines = []
    texts = df["text"].values
    labels = df["author_code"].values if is_train else None
    for i in range(len(texts)):
        label = labels[i] if is_train else 1
        safe = str(texts[i]).replace("|", " ").replace(":", " ")
        words = re.findall(r"[a-z']+", safe.lower())
        puncts = ["P" + p for p in re.findall(r"[^\w\s]", safe)]   # prefix "P" keeps punctuation features distinct
        chars = char_ngrams(safe, char_ns)
        lines.append(f"{label} |w {' '.join(words)} |p {' '.join(puncts)} |c {' '.join(chars)}\n")
    return lines

for fname, part in [("train_rich.vw", train_texts_part), ("valid_rich.vw", valid_texts)]:
    with open(f"{OUTPUT_DIR}/{fname}", "w") as f:
        f.writelines(to_vw_rich(part))

In [64]:
N_PASSES = 15

vw = Workspace(oaa=3, loss_function="logistic", ngram=2, b=29, quiet=True,
               final_regressor=f"{OUTPUT_DIR}/spooky_rich.vw")
for _ in range(N_PASSES):
    with open(f"{OUTPUT_DIR}/train_rich.vw") as f:
        for line in f:
            vw.learn(line)
vw.finish()

vw = Workspace(quiet=True, i=f"{OUTPUT_DIR}/spooky_rich.vw")
pred_rich = [vw.predict(line) for line in open(f"{OUTPUT_DIR}/valid_rich.vw")]
vw.finish()

print(f"Rich VW (words+punct+char) accuracy: {accuracy_score(y_valid, pred_rich):.4f}"
      f"   macro-F1: {f1_score(y_valid, pred_rich, average='macro'):.4f}")

Rich VW (words+punct+char) accuracy: 0.8553   macro-F1: 0.8552


<details>
<summary><span style="color:red">About this cell:</span></summary>

This code creates a **richer Vowpal Wabbit model** by adding style-based features, not just ordinary words.

The goal is:

```text
word features + punctuation features + character n-gram features
```

This is useful because authorship identification is strongly related to **writing style**.

> In the rich VW model, word features include all lowercase word tokens, including short function words. This differs from the baseline VW model, which kept only words with at least three characters.

## 1. Character n-gram function

```python
def char_ngrams(text, ns=(2, 3, 4)):
    """Boundary-aware character n-grams; whitespace/edges become '_'."""
    t = "_" + re.sub(r"\s+", "_", text.strip()) + "_"
    return [t[i:i + n] for n in ns for i in range(len(t) - n + 1)]
```

This function creates character n-grams of length 2, 3, and 4.

For example, if the text is:

```text
dark night
```

it first becomes:

```text
_dark_night_
```

The underscores show word boundaries and spaces.

Then 2-character n-grams include:

```text
_d
da
ar
rk
k_
_n
ni
...
```

3-character n-grams include:

```text
_da
dar
ark
rk_
_n
...
```

Why this matters:

Character n-grams can capture small stylistic patterns such as:

```text
word endings
spelling patterns
punctuation habits
spacing patterns
capitalization traces
```

These are useful for authorship classification.

## 2. Rich VW formatter

```python
def to_vw_rich(df, is_train=True, char_ns=(2, 3, 4)):
```

This function converts each row of our dataframe into **Vowpal Wabbit format**.

VW format looks like:

```text
<label> |w word features |p punctuation features |c character features
```

The three parts are called **namespaces**.


```python
texts = df["text"].values
labels = df["author_code"].values if is_train else None
```

This extracts:

* the text sentences
* the author labels, if this is training data

If it is test data, labels are not available.


```python
label = labels[i] if is_train else 1
```

If it is training or validation data, use the real label.

If it is test data, use dummy label `1`, because VW still expects a label in the input format.

```python
safe = str(texts[i]).replace("|", " ").replace(":", " ")
```

This removes characters that have special meaning in VW.

In VW:

* `|` starts a namespace
* `:` can be used for feature values

So replacing them avoids formatting problems.


## 3. Word features

```python
words = re.findall(r"[a-z']+", safe.lower())
```

This extracts lowercase word tokens.

Unlike the earlier word-only version, this keeps short words too.

For example:

```text
I was in the house
```

keeps:

```text
i
was
in
the
house
```

This matters because short function words like:

```text
I
of
to
the
and
in
```

are important stylometric signals. Authors often differ in how they use small function words.

## 4. Punctuation features

```python
puncts = ["P" + p for p in re.findall(r"[^\w\s]", safe)]
```

This extracts punctuation marks.

For example:

```text
"Wait; what happened?"
```

punctuation features become something like:

```text
P"
P;
P?
P"
```

The `"P"` prefix separates punctuation features from normal word features.

This is useful because authors may differ in their use of:

```text
commas
semicolons
dashes
quotation marks
exclamation marks
```


## 5. Character n-gram features

```python
chars = char_ngrams(safe, char_ns)
```

This creates character n-grams of length 2, 3, and 4 from the original text.

These features capture lower-level writing style, such as:

```text
suffixes
prefixes
spacing
punctuation context
word shapes
```

## 6. Building one VW line

```python
lines.append(f"{label} |w {' '.join(words)} |p {' '.join(puncts)} |c {' '.join(chars)}\n")
```

This creates one training example in VW format.

Example:

```text
1 |w i heard the sound |p P. |c _I I_ _h he ea ar rd d_ _t th he _I_ I_h _he hea ear ard ...
```

```
2-grams: _I, I_, _h, he, ea, ar, rd, d_, _t, th, he, ...
3-grams: _I_, I_h, _he, hea, ear, ard, rd_, ...
4-grams: _I_h, I_he, _hea, hear, eard, ...
```

Meaning:

* `1` is the author label
* `|w` contains word features
* `|p` contains punctuation features
* `|c` contains character n-gram features

This is richer than the word-only version because it gives VW multiple views of the same sentence.


## 7. Saving train and validation files

```python
for fname, part in [("train_rich.vw", train_texts_part), ("valid_rich.vw", valid_texts)]:
    with open(f"{OUTPUT_DIR}/{fname}", "w") as f:
        f.writelines(to_vw_rich(part))
```

This writes two VW files:

```text
train_rich.vw
valid_rich.vw
```

These are the rich-feature versions of our training and validation data.


## 8. Training the rich VW model

```python
N_PASSES = 15
```

The model will see the training data 15 times.


```python
vw = Workspace(oaa=3, loss_function="logistic", ngram=2, b=29, quiet=True,
               final_regressor=f"{OUTPUT_DIR}/spooky_rich.vw")
```

This creates a VW model.

Important parts:

* `oaa=3`: one-against-all classification for 3 authors
* `loss_function="logistic"`: uses logistic loss
* `ngram=2`: adds bigram features on top of the provided features. The `|c` namespace already contains manually created character n-grams. The `ngram=2` option is used mainly to let VW add adjacent token bigrams, especially word bigrams in the `|w` namespace, giving the model short phrase-level features on top of the manually engineered style features.

* `b=29`: uses a larger hashing space than before
* `final_regressor=...`: saves the model file

<details>
<summary><span style="color:red">

When I set `ngram=2` in Vowpal Wabbit, where are the automatically generated bigram features added? Are they added inside the existing `|w` namespace, or does VW create them separately? Also, are these bigrams written into the `.vw` file, or are they generated internally during training?:</span></summary>

They are **not written back into the VW text file**. Our file still looks like:

```text
<label> |w words... |p punct... |c chars...
```

When VW reads the line, `ngram=2` makes VW **internally create extra hashed bigram features** during training.

So for this input:

```text
1 |w i heard the sound |p P. |c _i i_ _h he ea ...
```

VW internally sees features like:

```text
|w i
|w heard
|w the
|w sound
```

and also creates word bigrams like:

```text
|w i_heard
|w heard_the
|w the_sound
```

So yes, conceptually the word bigrams are added **inside the `|w` namespace**, but they are added **internally by VW**, not physically into our `.vw` file.

Important nuance: with global `ngram=2`, VW may also create adjacent bigrams inside other namespaces, like `|p` and `|c`, not only `|w`.

So the clean explanation is:

> The `.vw` file only contains the original namespace features. The `ngram=2` option tells VW to internally create additional adjacent-token bigram features, mainly useful for the `|w` word namespace. These bigrams are hashed and used by the model, but they do not appear explicitly in the saved `.vw` text file.

</details>

```python
for _ in range(N_PASSES):
    with open(f"{OUTPUT_DIR}/train_rich.vw") as f:
        for line in f:
            vw.learn(line)
```

This manually trains VW for 15 passes.

This is necessary because, when using the Python streaming API, VW does not automatically repeat the file unless you loop yourself.


```python
vw.finish()
```

This finishes training and saves the model.


## 9. Prediction

```python
vw = Workspace(quiet=True, i=f"{OUTPUT_DIR}/spooky_rich.vw")
```

This reloads the trained model.


```python
pred_rich = [vw.predict(line) for line in open(f"{OUTPUT_DIR}/valid_rich.vw")]
```

This predicts the author for every validation sentence.


```python
print(f"Rich VW (words+punct+char) accuracy: {accuracy_score(y_valid, pred_rich):.4f}"
      f"   macro-F1: {f1_score(y_valid, pred_rich, average='macro'):.4f}")
```

This prints two metrics:

* **accuracy**: overall percentage of correct predictions
* **macro-F1**: average F1 score across the three authors, treating each author equally

Macro-F1 is useful because the dataset is slightly imbalanced.


## Brief conclusion

This model is stronger than the word-only VW baseline because it adds **authorship-aware style features**:

```text
words
function words
punctuation
character n-grams
word bigrams
```

So the model is no longer relying only on topic words. It can also learn stylistic habits, such as punctuation usage, short word patterns, and character-level writing patterns.
</details>

The rich VW model improves on the word-only baseline by adding features that better match the nature of the task. Since authorship identification depends heavily on style, punctuation and character n-grams give the model extra signals beyond plain word choice.


## 2. Classical TF-IDF ensemble: testing another sparse-text pipeline

Is ~0.85 accuracy specific to VW, or do other classical sparse-text models reach a similar range? To check this, I build a second pipeline using TF-IDF features with word 1-2 grams, character 2-5 grams, and sublinear TF. Then I average three complementary models: **Logistic Regression**, **NB-SVM-style Logistic Regression** based on Wang and Manning (2012), and **Complement Naive Bayes**. For a fairer setup, I tune the regularization strength separately for plain Logistic Regression and the NB-SVM-style model using inner cross-validation on the training split only.


In [65]:
CLASSES = np.array([1, 2, 3])  # 1=EAP, 2=MWS, 3=HPL

def build_tfidf(fit_texts):
    """
    Fit two TF-IDF vectorizers on the training text only:
    - word-level TF-IDF with unigrams and bigrams
    - character-level TF-IDF with 2-5 character n-grams inside word boundaries
    """
    word_vectorizer = TfidfVectorizer(
        sublinear_tf=True,
        ngram_range=(1, 2),
        min_df=2
    ).fit(fit_texts)

    char_vectorizer = TfidfVectorizer(
        sublinear_tf=True,
        analyzer="char_wb",
        ngram_range=(2, 5),
        min_df=2
    ).fit(fit_texts)

    return word_vectorizer, char_vectorizer


def tfidf_features(word_vectorizer, char_vectorizer, texts):
    """
    Transform texts into one sparse matrix by joining word-level and
    character-level TF-IDF features.
    """
    word_features = word_vectorizer.transform(texts)
    char_features = char_vectorizer.transform(texts)

    return sp.hstack([word_features, char_features]).tocsr()


def nbsvm_proba(X_tr, y_tr, X_te, C=10):
    """
    NB-SVM style classifier.

    For each class, compute Naive-Bayes-style log-count-ratio weights,
    multiply the TF-IDF features by those weights, then train a binary
    Logistic Regression classifier for that class.

    This is not a pure SVM here. In this implementation, the final classifier
    is Logistic Regression trained on NB-weighted features.
    """
    y_tr = np.asarray(y_tr)
    out = np.zeros((X_te.shape[0], len(CLASSES)))

    for k, c in enumerate(CLASSES):
        y_bin = (y_tr == c).astype(int)

        # Feature totals inside the current class and outside the current class.
        p = X_tr[y_bin == 1].sum(axis=0) + 1
        q = X_tr[y_bin == 0].sum(axis=0) + 1

        # Log-count ratio: positive values favor the current class;
        # negative values favor the other classes.
        r = np.asarray(np.log((p / p.sum()) / (q / q.sum()))).ravel()

        model = LogisticRegression(C=C, max_iter=3000)
        model.fit(X_tr.multiply(r), y_bin)

        # Column 1 is the probability of "current class" in the one-vs-rest setup.
        out[:, k] = model.predict_proba(X_te.multiply(r))[:, 1]

    # Normalize one-vs-rest probabilities so each row sums to 1.
    return out / out.sum(axis=1, keepdims=True)


def tune_nbsvm_C(X, y, C_grid=(0.1, 0.3, 1, 3, 10, 30), n_splits=5, random_state=42):
    """
    Tune the Logistic Regression C parameter inside NB-SVM using inner CV
    on the training split only.
    """
    y = np.asarray(y)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    scores = []

    for C in C_grid:
        oof = np.zeros((X.shape[0], len(CLASSES)))

        for tr_idx, va_idx in cv.split(X, y):
            X_inner_tr = X[tr_idx]
            X_inner_va = X[va_idx]
            y_inner_tr = y[tr_idx]

            oof[va_idx] = nbsvm_proba(
                X_inner_tr,
                y_inner_tr,
                X_inner_va,
                C=C
            )

        score = log_loss(y, oof, labels=CLASSES)
        scores.append((C, score))
        print(f"NB-SVM C={C:<5} inner-CV log loss: {score:.4f}")

    best_C, best_score = min(scores, key=lambda item: item[1])
    print(f"\nBest NB-SVM C: {best_C}   best inner-CV log loss: {best_score:.4f}")

    return best_C, pd.DataFrame(scores, columns=["C", "inner_cv_log_loss"])

In [66]:
'''
This code snippet tunes C separately for:
plain Logistic Regression on normal TF-IDF features
NB-SVM-style Logistic Regression on NB-weighted TF-IDF features
'''
def tune_lr_C(X, y, C_grid=(0.1, 0.3, 1, 3, 10, 30), n_splits=5, random_state=42):
    """
    Tune the C parameter for plain multiclass Logistic Regression
    using inner cross-validation on the training split only.
    """
    y = np.asarray(y)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    scores = []

    for C in C_grid:
        oof = np.zeros((X.shape[0], len(CLASSES)))

        for tr_idx, va_idx in cv.split(X, y):
            X_inner_tr = X[tr_idx]
            X_inner_va = X[va_idx]
            y_inner_tr = y[tr_idx]

            model = LogisticRegression(C=C, max_iter=3000)
            model.fit(X_inner_tr, y_inner_tr)

            oof[va_idx] = model.predict_proba(X_inner_va)

        score = log_loss(y, oof, labels=CLASSES)
        scores.append((C, score))
        print(f"LR C={C:<5} inner-CV log loss: {score:.4f}")

    best_C, best_score = min(scores, key=lambda item: item[1])
    print(f"\nBest LR C: {best_C}   best inner-CV log loss: {best_score:.4f}")

    return best_C, pd.DataFrame(scores, columns=["C", "inner_cv_log_loss"])


wv, cv = build_tfidf(train_texts_part["text"])

X_tr = tfidf_features(wv, cv, train_texts_part["text"])
X_va = tfidf_features(wv, cv, valid_texts["text"])

best_lr_C, lr_c_results = tune_lr_C(
    X_tr,
    y_part,
    C_grid=(0.1, 0.3, 1, 3, 10, 30),
    n_splits=5,
    random_state=42
)

best_nbsvm_C, nbsvm_c_results = tune_nbsvm_C(
    X_tr,
    y_part,
    C_grid=(0.1, 0.3, 1, 3, 10, 30),
    n_splits=5,
    random_state=42
)

proba_lr = LogisticRegression(C=best_lr_C, max_iter=3000).fit(X_tr, y_part).predict_proba(X_va)

proba_nbsvm = nbsvm_proba(
    X_tr,
    y_part,
    X_va,
    C=best_nbsvm_C
)

proba_cnb = ComplementNB().fit(X_tr, y_part).predict_proba(X_va)

proba_avg = (proba_lr + proba_nbsvm + proba_cnb) / 3

pred_avg = proba_avg.argmax(axis=1) + 1

print(
    f"TF-IDF 3-model average     accuracy: {accuracy_score(y_valid, pred_avg):.4f}"
    f"   macro-F1: {f1_score(y_valid, pred_avg, average='macro'):.4f}"
    f"   log loss: {log_loss(y_valid, proba_avg, labels=CLASSES):.4f}"
)

LR C=0.1   inner-CV log loss: 0.7509
LR C=0.3   inner-CV log loss: 0.6051
LR C=1     inner-CV log loss: 0.4850
LR C=3     inner-CV log loss: 0.4196
LR C=10    inner-CV log loss: 0.3924
LR C=30    inner-CV log loss: 0.3986

Best LR C: 10   best inner-CV log loss: 0.3924
NB-SVM C=0.1   inner-CV log loss: 0.8707
NB-SVM C=0.3   inner-CV log loss: 0.7364
NB-SVM C=1     inner-CV log loss: 0.5951
NB-SVM C=3     inner-CV log loss: 0.4925
NB-SVM C=10    inner-CV log loss: 0.4162
NB-SVM C=30    inner-CV log loss: 0.3853

Best NB-SVM C: 30   best inner-CV log loss: 0.3853
TF-IDF 3-model average     accuracy: 0.8601   macro-F1: 0.8596   log loss: 0.3843


The tuned TF-IDF 3-model average reached **0.8601 accuracy**, **0.8596 macro-F1**, and **0.3843 log loss** on the holdout set. Compared with the rich VW model, the accuracy gain is small, but the TF-IDF ensemble gives a stronger probability-based result. This is important because the Kaggle competition is evaluated with log loss, so the quality of the predicted probabilities matters, not only the final class prediction.

In this version, the regularization strength was tuned separately for plain Logistic Regression and for the NB-SVM-style Logistic Regression. For the NB-SVM component, inner cross-validation selected **C=30**, with the best inner-CV log loss of **0.3853** among the tested values. Since larger `C` means weaker regularization, this suggests that the NB-SVM-style model benefited from being allowed to fit the sparse TF-IDF features more flexibly within the tested range.

Overall, this experiment shows that combining word-level TF-IDF, character-level TF-IDF, NB-SVM-style feature weighting, Logistic Regression, and Complement Naive Bayes creates a strong classical sparse-text baseline. The main improvement is not a large jump in accuracy, but a more reliable probability model, which is especially valuable for a log-loss-based competition.


<details>
<summary><span style="color:red">Detailed explanation of the TF-IDF and NB-SVM code</span></summary>

This code does four main things:

1. Builds **word-level and character-level TF-IDF features**
2. Defines an **NB-SVM-style model**
3. Tunes the `C` parameter inside NB-SVM using inner cross-validation
4. Averages three different linear/probabilistic models: Logistic Regression, NB-SVM, and Complement Naive Bayes

<details>
<summary><span style="color:red">discriminative vs generative:</span></summary>

**Logistic Regression is a probabilistic discriminative model.**

It is **discriminative** because it models:

$$
P(y \mid x)
$$

directly, not:

$$
P(x \mid y)
$$

like Naive Bayes does.

It is **probabilistic** because it outputs class probabilities.

For binary classification:

$$
P(y=1 \mid x)=\frac{1}{1+e^{-(w^Tx+b)}}
$$

For multiclass classification, it can use softmax:

$$
P(y=c \mid x)=\frac{e^{w_c^Tx+b_c}}{\sum_k e^{w_k^Tx+b_k}}
$$

So in our notebook, Logistic Regression predicts probabilities such as:

```text
EAP: 0.72
MWS: 0.18
HPL: 0.10
```

Then the class with the highest probability is chosen.

**Discriminative** means the model learns the boundary between classes directly.

In classification, we have:

$$
x = \text{input features}
$$

$$
y = \text{class label}
$$

For our notebook:

$$
x = \text{TF-IDF features of a sentence}
$$

$$
y = \text{author: EAP, MWS, or HPL}
$$

A **discriminative model** learns:

$$
P(y \mid x)
$$

Meaning:

> Given this sentence’s features, what is the probability that the author is EAP, MWS, or HPL?

So Logistic Regression directly learns:

```text
sentence features → author probability
```

For example:

```text
P(EAP | sentence) = 0.70
P(MWS | sentence) = 0.20
P(HPL | sentence) = 0.10
```

It does **not** try to model how each author generates text.

That is different from a **generative model**, like Naive Bayes, which models:

$$
P(x \mid y)
$$

Meaning:

> If the author were EAP, how likely would this sentence/features be?

So:

```text
Logistic Regression = discriminative
It learns class boundaries directly.

Naive Bayes = generative
It models how features are generated by each class.
```

</details>

# 1. Class labels

```python
CLASSES = np.array([1, 2, 3])  # 1=EAP, 2=MWS, 3=HPL
```

This defines our three author labels:

```text
1 = Edgar Allan Poe
2 = Mary Shelley
3 = H. P. Lovecraft
```

We use this later when calculating probabilities and log loss.


# 2. `build_tfidf`

```python
def build_tfidf(fit_texts):
```

This function creates and fits two TF-IDF vectorizers.

<details>
<summary><span style="color:red">What a TF-IDF vectorizer is:</span></summary>

A **TF-IDF vectorizer** is a tool that converts raw text into numerical features using TF-IDF scores.

In ML, models cannot understand text like:

```text
the dark night was silent
```

So `TfidfVectorizer` turns the sentence into a vector of numbers:

```text
[0.0, 0.42, 0.0, 0.81, ...]
```

Each number represents how important a word, phrase, or character pattern is in that sentence.

TF-IDF means:

$$
\text{TF-IDF} = \text{Term Frequency} \times \text{Inverse Document Frequency}
$$

### 1. Term Frequency, TF

This measures how often a term appears in a document.

If the word **“night”** appears many times in one sentence/document, its TF is higher.

### 2. Inverse Document Frequency, IDF

This measures how rare or distinctive a term is across the whole dataset.

A common word like:

```text
the
and
of
```

gets a lower IDF.

A more distinctive word or pattern like:

```text
chamber
madness
eldritch
```

gets a higher IDF.

So TF-IDF gives high weight to terms that are:

```text
frequent in this sentence
but not too common everywhere
```

In this notebook, we use two TF-IDF vectorizers:

```python
wv = TfidfVectorizer(...)
cv = TfidfVectorizer(...)
```

* `wv` creates **word-level TF-IDF features**
* `cv` creates **character-level TF-IDF features**

So the model gets both:

```text
word/phrase information
+
small stylistic character patterns
```

That is very useful for authorship identification.


</details>

A **TF-IDF vectorizer** converts raw text into numerical features.

For example, a sentence like:

```text
the dark night was silent
```

becomes a sparse numeric vector like:

```text
[0.00, 0.41, 0.00, 0.72, ...]
```

Each column represents a word, phrase, or character pattern.

The basic TF-IDF idea is:

$$
\text{TF-IDF}(t,d)=\text{TF}(t,d)\times \text{IDF}(t)
$$

where:

$$
\text{TF}(t,d)
$$

means how often term $t$ appears in document $d$, and:

$$
\text{IDF}(t)
$$

means how rare or distinctive term $t$ is across the dataset.


In this dataset, each “document” is usually one sentence, not a long article. So for word-level TF-IDF, many words will appear either:

```
0 times, the word is not in the sentence
1 time, the word appears once
```

So for many word features:

$$TF(t,d) \in \{0,1\}$$

A common term like:

```text
the
and
of
```

usually gets a lower weight.

A more distinctive term like:

```text
chamber
horror
creature
```

can get a higher weight.


## 2.1 Word-level TF-IDF

```python
word_vectorizer = TfidfVectorizer(
    sublinear_tf=True,
    ngram_range=(1, 2),
    min_df=2
).fit(fit_texts)
```

This creates word-based TF-IDF features.

### `ngram_range=(1, 2)`

This means the vectorizer uses:

```text
unigrams = single words
bigrams = pairs of consecutive words
```

Example sentence:

```text
the dark night
```

Features can include:

```text
the
dark
night
the dark
dark night
```

This is useful because some stylistic patterns appear not only in single words, but also in short phrases.



### `sublinear_tf=True`

Normally, term frequency uses the raw count:

$$
tf
$$

With `sublinear_tf=True`, sklearn uses:

$$
1 + \log(tf)
$$  

This reduces the effect of words that appear many times.

For example, if a word appears 20 times, we do not want it to become 20 times more important. Sublinear scaling makes the increase gentler.


### `min_df=2`

This means:

> Ignore features that appear in fewer than 2 documents.

So extremely rare words or patterns are removed.

This helps reduce noise.


## 2.2 Character-level TF-IDF

```python
char_vectorizer = TfidfVectorizer(
    sublinear_tf=True,
    analyzer="char_wb",
    ngram_range=(2, 5),
    min_df=2
).fit(fit_texts)
```

This creates character n-gram features.

Instead of looking at full words, it looks at small character patterns.

For example, from:

```text
dark
```

it can create patterns like:

```text
da
dar
dark
ark
```

### `analyzer="char_wb"`

This means character n-grams are extracted **inside word boundaries**.

It can capture prefixes, suffixes, and spelling patterns, but it avoids noisy character patterns that cross too freely between words.

For authorship identification, character n-grams are very useful because they can capture style signals such as:

```text
spelling habits
suffixes
prefixes
punctuation-adjacent patterns
word shapes
```

This matters because authorship is not only about topic. It is also about style.


# 3. `tfidf_features`

```python
def tfidf_features(word_vectorizer, char_vectorizer, texts):
```

This function transforms texts into one combined feature matrix.

```python
word_features = word_vectorizer.transform(texts)
char_features = char_vectorizer.transform(texts)
```

These two lines create:

```text
word_features = word-level TF-IDF matrix
char_features = character-level TF-IDF matrix
```

Then:

```python
return sp.hstack([word_features, char_features]).tocsr()
```

joins them horizontally.

So the final feature vector looks like:

```text
[word TF-IDF features | character TF-IDF features]
```

Mathematically, if:

$$
X_{word}
$$

is the word feature matrix, and:

$$
X_{char}
$$

is the character feature matrix, then the combined matrix is:

[
X = [X_{word}, X_{char}]
]

`.tocsr()` converts it to CSR sparse format.

CSR is useful because TF-IDF matrices are mostly zeros. Most sentences contain only a tiny fraction of all possible words and character patterns.


# 4. `nbsvm_proba`

```python
def nbsvm_proba(X_tr, y_tr, X_te, C=10):
```

This function implements an **NB-SVM-style classifier**.

The name is a little confusing. In this implementation, it is not a pure SVM.

It is:

```text
Naive Bayes log-count-ratio feature weighting
+
Logistic Regression classifier
```

The idea is:

> First, find which features are unusually common for each author compared with the other authors. Then multiply the TF-IDF features by those class-specific weights. Then train Logistic Regression.


## 4.1 Convert labels to NumPy array

```python
y_tr = np.asarray(y_tr)
```

This makes sure the labels are in NumPy array form, which makes indexing easier.


## 4.2 Create an empty probability matrix

```python
out = np.zeros((X_te.shape[0], len(CLASSES)))
```

This creates a matrix for predicted probabilities.

Shape:

```text
number of validation/test samples × number of classes
```

Since we have three authors, each row will eventually look like:

```text
[P(EAP), P(MWS), P(HPL)]
```


## 4.3 Loop over classes

```python
for k, c in enumerate(CLASSES):
```

This trains one binary classifier per class.

So it trains:

```text
EAP vs not-EAP
MWS vs not-MWS
HPL vs not-HPL
```

This is called **one-vs-rest classification**.


## 4.4 Create binary labels

```python
y_bin = (y_tr == c).astype(int)
```

For the current class $c$, this turns the labels into 0 and 1.

Example for EAP:

```text
EAP     → 1
not EAP → 0
```

Mathematically:

$$
y_i^{(c)} =
\begin{cases}
1 & \text{if } y_i = c \\
0 & \text{if } y_i \neq c
\end{cases}
$$


# 5. Naive-Bayes-style feature weighting

This part is the core of NB-SVM.

```python
p = X_tr[y_bin == 1].sum(axis=0) + 1
q = X_tr[y_bin == 0].sum(axis=0) + 1
```

For the current class $c$:

$$
p_j = 1 + \sum_{i:y_i=c} X_{ij}
$$

$$
q_j = 1 + \sum_{i:y_i\neq c} X_{ij}
$$

where:

```text
p_j = total value of feature j inside the current class
q_j = total value of feature j outside the current class
```

The `+ 1` is smoothing.

It prevents zero values, which would cause division problems later.


## 5.1 Convert feature totals into normalized feature probabilities

The code uses:

```python
p / p.sum()
q / q.sum()
```

For feature $j$:

$$
P_j^{(c)} = \frac{p_j}{\sum_l p_l}
$$

$$
Q_j^{(c)} = \frac{q_j}{\sum_l q_l}
$$

Meaning:

$$
P_j^{(c)}
$$

is how common feature $j$ is inside class $c$, and:

$$
Q_j^{(c)}
$$

is how common feature $j$ is outside class $c$.


## 5.2 Compute log-count ratio

```python
r = np.asarray(np.log((p / p.sum()) / (q / q.sum()))).ravel()
```

This computes:

$$
r_j^{(c)}
=
\log
\left(
\frac{P_j^{(c)}}{Q_j^{(c)}}
\right)
$$  

or fully:

$$
r_j^{(c)}
=
\log
\left(
\frac{
\frac{p_j}{\sum_l p_l}
}{
\frac{q_j}{\sum_l q_l}
}
\right)
$$

This value tells us whether a feature is more associated with the current author or with the other authors.

If:

$$
r_j^{(c)} > 0
$$

then feature $j$ is more common in class $c$.

If:

$$
r_j^{(c)} < 0
$$

then feature $j$ is more common outside class $c$.

If:

$$
r_j^{(c)} \approx 0
$$

then the feature does not strongly separate class $c$ from the others.


# 6. Train Logistic Regression on NB-weighted features

```python
model = LogisticRegression(C=C, max_iter=3000)
model.fit(X_tr.multiply(r), y_bin)
```

Here:

```python
X_tr.multiply(r)
```

means each feature column is multiplied by its log-count-ratio weight.

For one sentence:

$$
\tilde{x}_{ij}^{(c)} = x_{ij} \cdot r_j^{(c)}
$$

So the original feature vector:

$$
x_i
$$

becomes:

$$
\tilde{x}_i^{(c)} = x_i \odot r^{(c)}
$$

where $\odot$ means element-wise multiplication.

Then Logistic Regression learns:

$$
P(y=c \mid x)
$$

For binary Logistic Regression:

$$
P(y=1 \mid x)=\frac{1}{1+e^{-(w^Tx+b)}}
$$

In this code, for each class, Logistic Regression learns:

> Given these NB-weighted features, is this sentence from the current author or not?

# 7. Predict current-class probability

```python
out[:, k] = model.predict_proba(X_te.multiply(r))[:, 1]
```

This line predicts probabilities for the validation/test set.

`model.predict_proba(...)` returns two columns because this is a binary classifier:

```text
column 0 = probability of not current class
column 1 = probability of current class
```

So:

```python
[:, 1]
```

takes the probability of the current author.

Then:

```python
out[:, k] = ...
```

stores those probabilities in the correct class column.

Example:

```text
k = 0 → EAP probability column
k = 1 → MWS probability column
k = 2 → HPL probability column
```


# 8. Normalize the one-vs-rest probabilities

```python
return out / out.sum(axis=1, keepdims=True)
```

Because each class was trained separately, the three probabilities may not naturally sum to 1.

Example before normalization:

```text
[0.70, 0.25, 0.20]
```

Sum:

```text
1.15
```

After normalization:

```text
[0.609, 0.217, 0.174]
```

Now:

$$
P(EAP)+P(MWS)+P(HPL)=1
$$

This is important because log loss expects valid probability distributions.


# 9. `tune_nbsvm_C`

```python
def tune_nbsvm_C(X, y, C_grid=(0.1, 0.3, 1, 3, 10, 30), n_splits=5, random_state=42):
```

This function tunes the `C` value inside NB-SVM.

`C` is the inverse regularization strength in Logistic Regression.

Important meaning:

```text
smaller C = stronger regularization
larger C  = weaker regularization
```

Regularization controls how flexible the model is.

If `C` is too small, the model may underfit.

If `C` is too large, the model may overfit.

So we test several values:

```python
C_grid=(0.1, 0.3, 1, 3, 10, 30)
```


## 9.1 Convert labels

```python
y = np.asarray(y)
```

Again, this makes indexing easier.


## 9.2 Create stratified cross-validation

```python
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
```

This creates 5 folds by default.

`StratifiedKFold` means each fold keeps roughly the same class distribution.

That matters because we have three authors. We do not want one fold to have too many EAP samples and too few HPL samples.


## 9.3 Store results

```python
scores = []
```

This list will store:

```text
(C value, inner-CV log loss)
```

## 9.4 Try each C value

```python
for C in C_grid:
```

For each candidate value of `C`, the function runs cross-validation.


## 9.5 Create OOF probability matrix

```python
oof = np.zeros((X.shape[0], len(CLASSES)))
```

OOF means **out-of-fold**.

This matrix will store predictions for every training sample.

But each sample will be predicted by a model that did **not** train on that sample.

This is important because it gives a fairer estimate of performance.


## 9.6 Inner cross-validation loop

```python
for tr_idx, va_idx in cv.split(X, y):
```

This splits the training data into inner training and inner validation folds.

For each fold:

```python
X_inner_tr = X[tr_idx]
X_inner_va = X[va_idx]
y_inner_tr = y[tr_idx]
```

This creates:

```text
X_inner_tr = inner training features
X_inner_va = inner validation features
y_inner_tr = inner training labels
```

Then:

```python
oof[va_idx] = nbsvm_proba(
    X_inner_tr,
    y_inner_tr,
    X_inner_va,
    C=C
)
```

This trains NB-SVM on the inner training fold and predicts probabilities for the inner validation fold.

The predictions are stored in the correct rows of `oof`.

After all folds finish, every training sample has an OOF prediction.


# 10. Score each C using log loss

```python
score = log_loss(y, oof, labels=CLASSES)
```

This calculates log loss for the OOF predictions.

Log loss measures probability quality.

For multiclass classification:

$$
\text{LogLoss}
=
-\frac{1}{N}
\sum_{i=1}^{N}
\sum_{j=1}^{M}
y_{ij}\log(p_{ij})
$$

where:

```text
N = number of samples
M = number of classes
y_ij = 1 if sample i belongs to class j, otherwise 0
p_ij = predicted probability for class j
```

Lower log loss is better.

This is especially important for Kaggle because the competition metric is log loss.


## 10.1 Save and print score

```python
scores.append((C, score))
print(f"NB-SVM C={C:<5} inner-CV log loss: {score:.4f}")
```

This records and prints the log loss for that `C`.


## 10.2 Pick the best C

```python
best_C, best_score = min(scores, key=lambda item: item[1])
```

This chooses the `C` with the lowest inner-CV log loss.

Then:

```python
return best_C, pd.DataFrame(scores, columns=["C", "inner_cv_log_loss"])
```

returns:

```text
best_C = best regularization value
nbsvm_c_results = table of all tested C values and their scores
```


# 11. Build final TF-IDF features

```python
wv, cv = build_tfidf(train_texts_part["text"])
```

This fits the TF-IDF vectorizers on the training split only.

Then:

```python
X_tr = tfidf_features(wv, cv, train_texts_part["text"])
X_va = tfidf_features(wv, cv, valid_texts["text"])
```

This creates:

```text
X_tr = training TF-IDF matrix
X_va = validation TF-IDF matrix
```

Important point:

> The vectorizers are fitted only on the training text, then used to transform validation text.

This avoids validation leakage.

# 12. Tune NB-SVM C

```python
best_nbsvm_C, nbsvm_c_results = tune_nbsvm_C(
    X_tr,
    y_part,
    C_grid=(0.1, 0.3, 1, 3, 10, 30),
    n_splits=5,
    random_state=42
)
```

This finds the best `C` for NB-SVM using only the training split.

The validation set is still untouched.

That is good practice.

The process is:

```text
training split
    inner cross-validation to choose C

validation split
    final evaluation only once
```

# 13. Train three models and get probabilities

```python
proba_lr = LogisticRegression(C=10, max_iter=3000).fit(X_tr, y_part).predict_proba(X_va)
```

This trains normal multiclass Logistic Regression on the TF-IDF features and predicts probabilities on the validation set.

Output shape:

```text
number of validation samples × 3
```

Each row looks like:

```text
[P(EAP), P(MWS), P(HPL)]
```

```python
proba_nbsvm = nbsvm_proba(X_tr, y_part, X_va, C=best_nbsvm_C)
```

This trains the NB-SVM-style classifier using the tuned `C`.

It also outputs probabilities:

```text
[P(EAP), P(MWS), P(HPL)]
```


```python
proba_cnb = ComplementNB().fit(X_tr, y_part).predict_proba(X_va)
```

This trains Complement Naive Bayes.

ComplementNB is a Naive Bayes variant that often works well for text classification, especially with sparse high-dimensional features.

It gives another probability estimate for each author.


# 14. Average the three models

```python
proba_avg = (proba_lr + proba_nbsvm + proba_cnb) / 3
```

This averages the predicted probabilities from the three models.

Mathematically:

$$
P_{avg}
=
\frac{
P_{LR}
+
P_{NB-SVM}
+
P_{CNB}
}{3}
$$

Why do this?

Because each model sees the same TF-IDF features differently:

```text
Logistic Regression:
learns discriminative linear boundaries

NB-SVM:
uses class-specific Naive Bayes feature weighting, then Logistic Regression

ComplementNB:
uses a probabilistic Naive Bayes approach
```

Averaging can make predictions more stable and reduce overconfident mistakes.

This is especially helpful for log loss.


# 15. Convert probabilities to class predictions

```python
pred_avg = proba_avg.argmax(axis=1) + 1
```

`argmax(axis=1)` finds the column with the highest probability for each row.

Example:

```text
[0.10, 0.75, 0.15]
```

The highest value is in column index 1.

Python indices are:

```text
0, 1, 2
```

But our labels are:

```text
1, 2, 3
```

So we add `+ 1`.

Column index 1 becomes class label 2, meaning MWS.


# 16. Print final metrics

```python
print(
    f"TF-IDF 3-model average     accuracy: {accuracy_score(y_valid, pred_avg):.4f}"
    f"   macro-F1: {f1_score(y_valid, pred_avg, average='macro'):.4f}"
    f"   log loss: {log_loss(y_valid, proba_avg, labels=CLASSES):.4f}"
)
```

This prints three metrics.

## Accuracy

$$
\text{Accuracy}
=
\frac{\text{correct predictions}}{\text{total predictions}}
$$

It tells us how often the final predicted author is correct.

## Macro-F1

Macro-F1 computes F1 for each author separately, then averages them.

This is useful because it checks whether the model performs reasonably across all three authors.

## Log loss

Log loss measures the quality of predicted probabilities.

A model can have good accuracy but bad log loss if it is too confident when wrong.

Since the Kaggle competition uses log loss, this metric is very important.


# Whole code in one simple explanation

This code builds a stronger TF-IDF ensemble for authorship classification.

First, it converts each sentence into sparse TF-IDF features using both word n-grams and character n-grams. The word-level vectorizer captures single words and short phrases, while the character-level vectorizer captures smaller stylistic patterns inside words.

Then, the code defines an NB-SVM-style classifier. In this implementation, NB-SVM means that Naive-Bayes-style log-count ratios are used to reweight the TF-IDF features, and then one-vs-rest Logistic Regression models are trained on those weighted features.

Next, the code tunes the regularization parameter `C` separately for two models: plain Logistic Regression on normal TF-IDF features, and NB-SVM-style Logistic Regression on NB-weighted TF-IDF features. Both tuning steps use inner cross-validation on the training split only, so the holdout validation set remains untouched.

Finally, the code trains three complementary models: tuned Logistic Regression, tuned NB-SVM-style Logistic Regression, and Complement Naive Bayes. Their predicted probabilities are averaged, and the final averaged predictions are evaluated on the holdout validation set using accuracy, macro-F1, and log loss.


</details>


## 3. Stacked generalization with fold averaging: the strongest classical model

A flat average gives every base model the same weight, but the models may not be equally useful for every class. To improve this, I use **stacked generalization**. Each base model first produces **out-of-fold** predictions on the training split, so every training sentence is predicted by a model that did not train on that sentence. These out-of-fold probabilities become the input features for a Logistic Regression meta-learner.

For the holdout set, each base model is trained across cross-validation folds and its holdout probabilities are averaged across folds. This is fold averaging, not classical bootstrap bagging. I use five base models: **Logistic Regression**, **NB-SVM-style Logistic Regression**, **Complement Naive Bayes**, **Multinomial Naive Bayes**, and **SGD Logistic Regression**. To make the experiment stronger, I tune the main hyperparameters of the base models and the meta-learner using cross-validation on the training split only, then evaluate the final stacked model once on the holdout validation set.

> It is called stacked generalization because the method “stacks” models in layers and tries to improve generalization to unseen data.

<details>
<summary><span style="color:red">More on "stacked generalization":</span></summary>  

It is called **stacked generalization** because the method “stacks” models in layers and tries to improve **generalization** to unseen data.

In our case, the structure is:

**Level 0 / base layer**
Our base models learn directly from the original text features:

```text
TF-IDF features → Logistic Regression
TF-IDF features → NB-SVM Logistic Regression
TF-IDF features → Complement NB
TF-IDF features → Multinomial NB
TF-IDF features → SGD Logistic Regression
```

Each model outputs class probabilities.

**Level 1 / meta layer**
Then another model, the meta-learner, learns from those predictions:

```text
Base-model probabilities → Logistic Regression meta-learner → final prediction
```

So the models are literally **stacked**: one layer of models produces predictions, and another model is trained on top of those predictions.

The word **generalization** means the goal is not just to fit the training data better, but to learn how to combine models in a way that performs better on unseen data. That is why we use **out-of-fold predictions**: the meta-learner should see predictions made on examples that the base model did not train on. This reduces leakage and makes the meta-learner learn realistic model behavior.

A simple way to say it:

**Stacked generalization means training a second-level model to learn how to combine the predictions of several first-level models, using out-of-fold predictions so the combination generalizes better to unseen data.**

In our paragraph, “stacked” refers to the layered model architecture, and “generalization” refers to improving holdout/test performance rather than only training accuracy.


</details>

In [67]:
# stacking helpers with hyperparameter search
BASE_MODELS = ["lr", "nbsvm", "cnb", "mnb", "sgd"]


BASE_PARAM_GRIDS = {
    "lr": {
        "C": [1, 3, 10, 30]
    },
    "nbsvm": {
        "C": [1, 3, 10, 30]
    },
    "cnb": {
        "alpha": [0.1, 0.3, 0.5, 1.0]
    },
    "mnb": {
        "alpha": [0.1, 0.3, 0.5, 1.0]
    },
    "sgd": {
        "alpha": [1e-6, 3e-6, 1e-5, 3e-5]}
}    

def align_proba(model, X):
    """
    Return predict_proba output aligned to CLASSES = [1, 2, 3].
    """
    raw = model.predict_proba(X)
    aligned = np.zeros((X.shape[0], len(CLASSES)))

    for model_col, label in enumerate(model.classes_):
        target_col = np.where(CLASSES == label)[0][0]
        aligned[:, target_col] = raw[:, model_col]

    return aligned


def base_proba(kind, X_tr, y_tr, X_te, params):
    """
    Train one base model and return class probabilities on X_te.
    """
    if kind == "lr":
        model = LogisticRegression(
            C=params["C"],
            max_iter=3000
        )
        model.fit(X_tr, y_tr)
        return align_proba(model, X_te)

    if kind == "nbsvm":
        return nbsvm_proba(
            X_tr,
            y_tr,
            X_te,
            C=params["C"]
        )

    if kind == "cnb":
        model = ComplementNB(
            alpha=params["alpha"]
        )
        model.fit(X_tr, y_tr)
        return align_proba(model, X_te)

    if kind == "mnb":
        model = MultinomialNB(
            alpha=params["alpha"]
        )
        model.fit(X_tr, y_tr)
        return align_proba(model, X_te)

    if kind == "sgd":
        model = SGDClassifier(
            loss="log_loss",
            alpha=params["alpha"],
            max_iter=100,
            tol=1e-4,
            random_state=17
        )
        model.fit(X_tr, y_tr)
        return align_proba(model, X_te)

    raise ValueError(f"Unknown base model: {kind}")


def tune_base_model(kind, X, y, param_grid, n_folds=5, seed=17):
    """
    Tune one base model using out-of-fold log loss on the training split.
    """
    y = np.asarray(y)
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)

    rows = []

    for params in ParameterGrid(param_grid):
        oof = np.zeros((X.shape[0], len(CLASSES)))

        for tr_idx, va_idx in skf.split(X, y):
            oof[va_idx] = base_proba(
                kind,
                X[tr_idx],
                y[tr_idx],
                X[va_idx],
                params
            )

        score = log_loss(y, oof, labels=CLASSES) # here y is multiclass labels for the training split, and oof contains the predicted probabilities for the validation fold in each CV iteration. We compute the log loss across all out-of-fold predictions to evaluate the performance of the current hyperparameters (params) for the base model specified by kind. oof.shape is (n_samples, n_classes) and y.shape is (n_samples,), so log_loss will compute the multiclass log loss across all samples using the predicted probabilities in oof and the true labels in y. The labels=CLASSES argument ensures that the log loss is computed with respect to the correct class order. The multiclass log loss formula is: -1/n * sum(sum(y_true * log(y_pred))) where y_true is the one-hot encoded true labels and y_pred is the predicted probabilities for each class. By passing the multiclass labels in CLASSES, we ensure that the log loss is computed correctly for our specific class labels (1, 2, 3) rather than assuming a default order. This is important because the predicted probabilities in oof must be aligned with the true labels in y for the log loss calculation to be meaningful.

        row = {"model": kind, **params, "inner_cv_log_loss": score}
        rows.append(row)

        print(f"{kind.upper():<6} params={params}   inner-CV log loss: {score:.4f}")

    results = pd.DataFrame(rows)
    best_row = results.loc[results["inner_cv_log_loss"].idxmin()]
    best_params = {
        col: best_row[col]
        for col in results.columns
        if col not in ["model", "inner_cv_log_loss"]
    }

    print(f"\nBest {kind.upper()} params: {best_params}   best inner-CV log loss: {best_row['inner_cv_log_loss']:.4f}\n")

    return best_params, results


def build_stack_features(X_train, y_train, X_test, best_params_by_model, n_folds=5, seed=17):
    """
    Build stacking features.

    For training data:
    - store out-of-fold probabilities from each base model.

    For holdout/test data:
    - average each base model's probabilities across folds.
    """
    y_train = np.asarray(y_train)
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)

    n_classes = len(CLASSES)
    n_models = len(BASE_MODELS)

    oof_stack = np.zeros((X_train.shape[0], n_classes * n_models)) # oof_stack is a 2D array initialized with zeros, where the number of rows is equal to the number of training samples (X_train.shape[0]) and the number of columns is equal to the total number of features generated by all base models (n_classes * n_models). Each base model will produce n_classes probability features for each sample, and since we have n_models base models, we multiply n_classes by n_models to get the total number of columns in oof_stack. This array will be filled with the out-of-fold predicted probabilities from each base model for each class, which will then be used as features for the meta-learner in the stacking ensemble.
    test_stack = np.zeros((X_test.shape[0], n_classes * n_models))

    feature_names = []

    for j, kind in enumerate(BASE_MODELS):
        start = j * n_classes
        end = start + n_classes

        params = best_params_by_model[kind]

        for label in CLASSES:
            feature_names.append(f"{kind}_p_{label}")

        for tr_idx, va_idx in skf.split(X_train, y_train):
            oof_stack[va_idx, start:end] = base_proba(
                kind,
                X_train[tr_idx],
                y_train[tr_idx],
                X_train[va_idx],
                params
            )

            test_stack[:, start:end] += base_proba(
                kind,
                X_train[tr_idx],
                y_train[tr_idx],
                X_test,
                params
            ) / n_folds

    return oof_stack, test_stack, feature_names # oof_stack.shape is (n_samples_train, n_classes * n_models) and test_stack.shape is (n_samples_test, n_classes * n_models). Each block of n_classes columns in oof_stack and test_stack corresponds to the predicted probabilities from one base model for each class. By concatenating these blocks horizontally, we create a new feature space for the meta-learner that captures the predictions of all base models. The out-of-fold predictions in oof_stack are used to train the meta-learner without data leakage, while the averaged predictions in test_stack are used for final evaluation or submission.


def tune_meta_C(oof_stack, y, C_grid=(0.03, 0.1, 0.3, 1, 3, 10, 30), n_folds=5, seed=17):
    """
    Tune the Logistic Regression meta-learner using CV on the stacking features.
    """
    y = np.asarray(y)
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)

    rows = []

    for C in C_grid:
        oof_meta = np.zeros((oof_stack.shape[0], len(CLASSES)))

        for tr_idx, va_idx in skf.split(oof_stack, y):
            meta = LogisticRegression(C=C, max_iter=3000)
            meta.fit(oof_stack[tr_idx], y[tr_idx])
            oof_meta[va_idx] = align_proba(meta, oof_stack[va_idx])

        score = log_loss(y, oof_meta, labels=CLASSES)
        rows.append({"meta_C": C, "inner_cv_log_loss": score})

        print(f"META LR C={C:<5} inner-CV log loss: {score:.4f}")

    results = pd.DataFrame(rows)
    best_row = results.loc[results["inner_cv_log_loss"].idxmin()]
    best_C = best_row["meta_C"]

    print(f"\nBest META LR C: {best_C}   best inner-CV log loss: {best_row['inner_cv_log_loss']:.4f}\n")

    return best_C, results

<details>
<summary><span style="color:red">about this cell:</span></summary>   

This code builds a **stacked ensemble classifier** and tunes its hyperparameters using cross-validation.

The main idea is:

```text
Original text features
        ↓
5 base models
        ↓
their predicted class probabilities
        ↓
Logistic Regression meta-model
        ↓
final prediction
```

Our classes are assumed to be:

```python
CLASSES = [1, 2, 3]
```

So each model gives 3 probability values:

```text
P(class 1), P(class 2), P(class 3)
```

Since we have 5 base models, the stacking feature vector has:

```text
5 models × 3 classes = 15 features
```
So the input matrix of the meta-learner is:

number of samples × 15


<details>
<summary><span style="color:red">More on this:</span></summary>  

The **input matrix of the meta-learner** is:

```text
number of samples × 15
```

For the training split, that matrix is:

```python
oof_stack.shape = (n_train_samples, 15)
```

For the holdout/test split, it is:

```python
test_stack.shape = (n_test_samples, 15)
```

Because you have:

```text
5 base models × 3 class probabilities = 15 columns
```

So one row in the meta-learner input looks like this:

```text
[
  lr_p_1, lr_p_2, lr_p_3,
  nbsvm_p_1, nbsvm_p_2, nbsvm_p_3,
  cnb_p_1, cnb_p_2, cnb_p_3,
  mnb_p_1, mnb_p_2, mnb_p_3,
  sgd_p_1, sgd_p_2, sgd_p_3
]
```

For example, for one sentence:

```text
[
  0.80, 0.15, 0.05,
  0.75, 0.20, 0.05,
  0.60, 0.30, 0.10,
  0.65, 0.25, 0.10,
  0.78, 0.17, 0.05
]
```

This 15-dimensional vector becomes the new feature representation for that sentence. Then the meta-learner, here Logistic Regression, learns from these 15 probability-based features.

So instead of this:

```text
TF-IDF features → final classifier
```

the meta-learner sees this:

```text
base-model probability features → final classifier
```

One small technical note: since the 3 probabilities from each model sum to 1, there is some redundancy. For example:

```text
p_3 = 1 - p_1 - p_2
```

So you could use only 2 probabilities per model, giving:

```text
5 models × 2 classes = 10 features
```

But using all 15 is common and usually fine, especially with Logistic Regression regularization.


</details>

## 1. Base model names

```python
BASE_MODELS = ["lr", "nbsvm", "cnb", "mnb", "sgd"]
```

This defines the five base classifiers used in the stack:

| Name    | Meaning                          |
| ------- | -------------------------------- |
| `lr`    | Logistic Regression              |
| `nbsvm` | NB-SVM-style Logistic Regression |
| `cnb`   | Complement Naive Bayes           |
| `mnb`   | Multinomial Naive Bayes          |
| `sgd`   | SGDClassifier with logistic loss |

These are the first-level models.


## 2. Hyperparameter grids

```python
BASE_PARAM_GRIDS = {
    "lr": {
        "C": [1, 3, 10, 30]
    },
    ...
}
```

This dictionary tells the code which hyperparameters to try for each base model.

For Logistic Regression:

```python
"C": [1, 3, 10, 30]
```

`C` controls regularization strength.

Higher `C` means weaker regularization, so the model is allowed to fit the training data more strongly.

Lower `C` means stronger regularization, so the model is more conservative.

For Naive Bayes models:

```python
"alpha": [0.1, 0.3, 0.5, 1.0]
```

`alpha` is smoothing.

For SGD Logistic Regression:

```python
"alpha": [1e-6, 3e-6, 1e-5, 3e-5]
```

Here `alpha` controls regularization strength.


## 3. `align_proba`

```python
def align_proba(model, X):
    """
    Return predict_proba output aligned to CLASSES = [1, 2, 3].
    """
```

This function makes sure the probability columns always follow the same class order.

For example, one model might return:

```text
column 0 = class 1
column 1 = class 2
column 2 = class 3
```

But another model might internally store classes in a slightly different order. So this function forces all outputs to match:

```text
[probability of class 1, probability of class 2, probability of class 3]
```

This matters a lot in stacking, because the meta-model must know exactly what each column means.

Example output shape:

```python
aligned.shape = (number_of_samples, 3)
```

So for 100 sentences:

```python
(100, 3)
```

## 4. `base_proba`

```python
def base_proba(kind, X_tr, y_tr, X_te, params):
    """
    Train one base model and return class probabilities on X_te.
    """
```

This function trains one base model and returns its predicted probabilities.

For example, if `kind == "lr"`:

```python
model = LogisticRegression(
    C=params["C"],
    max_iter=3000
)
model.fit(X_tr, y_tr)
return align_proba(model, X_te)
```

Meaning:

1. Create a Logistic Regression model.
2. Train it on `X_tr`, `y_tr`.
3. Predict probabilities on `X_te`.
4. Align the probability columns to `[1, 2, 3]`.

So this function is a helper that says:

```text
Given a model type and parameters, train the model and return probabilities.
```

For `nbsvm`, it calls our custom function:

```python
nbsvm_proba(...)
```
that we defined earlier.

## 5. `tune_base_model`

```python
def tune_base_model(kind, X, y, param_grid, n_folds=5, seed=17):
```

This function chooses the best hyperparameters for one base model.

For example, for Logistic Regression, it tests:

```python
C = 1
C = 3
C = 10
C = 30
```

It uses **Stratified K-Fold cross-validation**:

```python
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
```

`StratifiedKFold` keeps class proportions similar in every fold.

So if our dataset has class 1, class 2, and class 3, each fold should contain a similar balance of those classes.

### Important part: out-of-fold prediction

Inside this function:

```python
oof = np.zeros((X.shape[0], len(CLASSES)))
```

This creates an empty matrix to store out-of-fold probabilities.

For each parameter setting, the model is trained on 4 folds and predicts the remaining fold.

Example with 5 folds:

```text
Fold 1: train on folds 2,3,4,5 → predict fold 1
Fold 2: train on folds 1,3,4,5 → predict fold 2
Fold 3: train on folds 1,2,4,5 → predict fold 3
Fold 4: train on folds 1,2,3,5 → predict fold 4
Fold 5: train on folds 1,2,3,4 → predict fold 5
```

So every training sentence gets predicted by a model that did **not** train on it.

That is why this is safer than predicting on the same data used for training.


### Scoring

```python
score = log_loss(y, oof, labels=CLASSES)
```

The code evaluates the predicted probabilities using **log loss**.

Lower log loss is better.

Log loss is useful here because stacking depends on probability quality, not just final class labels.

So this function finds the parameter setting with the lowest inner-CV log loss.

It returns:

```python
return best_params, results
```

For example:

```python
best_params = {"C": 10}
```

and `results` is a DataFrame showing all tested settings and their scores.


## 6. `build_stack_features`

```python
def build_stack_features(X_train, y_train, X_test, best_params_by_model, n_folds=5, seed=17):
```

This is the central stacking function.

It creates two matrices:

```python
oof_stack
test_stack
```

### `oof_stack`

This contains out-of-fold predictions for the training data.

Shape:

```python
(number_of_training_samples, 15)
```

because:

```text
5 base models × 3 class probabilities = 15 stacking features
```

Example columns:

```text
lr_p_1, lr_p_2, lr_p_3,
nbsvm_p_1, nbsvm_p_2, nbsvm_p_3,
cnb_p_1, cnb_p_2, cnb_p_3,
mnb_p_1, mnb_p_2, mnb_p_3,
sgd_p_1, sgd_p_2, sgd_p_3
```

These are generated here:

```python
for label in CLASSES:
    feature_names.append(f"{kind}_p_{label}")
```

So the meta-model does not see the original TF-IDF features. It sees the base models’ probabilities.

### `test_stack`

This contains predictions for the holdout/test set.

Shape:

```python
(number_of_test_samples, 15)
```

But there is an important detail:

```python
test_stack[:, start:end] += base_proba(...) / n_folds
```

For each base model, the code trains 5 versions of that model, one per fold. Each version predicts the holdout/test set. Then the probabilities are averaged.

So for the holdout set, this is doing:

```text
test prediction = average of 5 fold-trained models
```

That is why we correctly described it as **fold averaging**, not classical bootstrap bagging.


## 7. Why out-of-fold predictions are necessary

This part is very important.

If we trained each base model on the full training set and then predicted the same training set, the meta-model would see overly optimistic predictions.

That would cause leakage.

The meta-model might learn from predictions that are too clean because the base models already saw those examples.

Instead, this code uses out-of-fold predictions:

```text
Each training example is predicted by a model that did not train on it.
```

That makes the meta-learner’s training data more realistic.  

## 8. `tune_meta_C`

```python
def tune_meta_C(oof_stack, y, C_grid=(0.03, 0.1, 0.3, 1, 3, 10, 30), n_folds=5, seed=17): # oof_stack.shape = (n_samples_train, n_classes * n_models) 
```

This function tunes the Logistic Regression meta-learner.

The meta-learner is the second-level model.

It does not train on raw text features. It trains on:

```python
oof_stack
```

That means it learns from the base models’ probability outputs.

For example, one row might look like:

```text
lr_p_1      = 0.70
lr_p_2      = 0.20
lr_p_3      = 0.10
nbsvm_p_1   = 0.65
nbsvm_p_2   = 0.25
nbsvm_p_3   = 0.10
...
```

The meta-model learns patterns like:

```text
When LR and NB-SVM strongly agree, trust them.
When MNB is uncertain but CNB is confident, maybe trust CNB.
For class 2, SGD may be more useful than MNB.
```

That is the main advantage of stacking over simple averaging.


## 9. Meta-model tuning process

Inside `tune_meta_C`, the code tries different values of `C`:

```python
C_grid=(0.03, 0.1, 0.3, 1, 3, 10, 30)
```

For each value, it cross-validates a Logistic Regression meta-model:

```python
meta = LogisticRegression(C=C, max_iter=3000)
meta.fit(oof_stack[tr_idx], y[tr_idx])
oof_meta[va_idx] = align_proba(meta, oof_stack[va_idx])
```

Then it calculates:

```python
score = log_loss(y, oof_meta, labels=CLASSES)
```

Again, lower log loss is better.

The function returns:

```python
return best_C, results
```

So we get the best regularization strength for the meta-model.


## 10. Overall workflow

The whole code works like this:

```text
Step 1:
Tune each base model using cross-validation on the training split.

Step 2:
Store the best hyperparameters for each base model.

Step 3:
Build stacking features:
- training features = out-of-fold base probabilities
- holdout features = fold-averaged base probabilities

Step 4:
Tune the Logistic Regression meta-model on the stacking features.

Step 5:
Train the final meta-model on all oof_stack features.

Step 6:
Evaluate once on the holdout validation set using test_stack.
```

The final training step would usually look like this:

```python
meta = LogisticRegression(C=best_meta_C, max_iter=3000)
meta.fit(oof_stack, y_train)

holdout_proba = align_proba(meta, test_stack)
holdout_pred = CLASSES[np.argmax(holdout_proba, axis=1)]
```


## Simple explanation

This code does three main things:

1. **Tunes each base model** using cross-validation and log loss.
2. **Creates stacking features** from the probability outputs of those base models.
3. **Tunes a Logistic Regression meta-model** that learns how to combine the base models.

So instead of saying:

```text
All models are equally important.
```

stacking says:

```text
Let another model learn how much to trust each base model for each class.
```

That is why this can be stronger than a flat average.


</details>

<details>
<summary><span style="color:red">More on these five base classifiers used in the stacking system:</span></summary>  

Below is a mathematical explanation of the five base models in our stacking system.

Assume:

```text
N = number of training samples
D = number of text features
K = number of classes = 3
x_i ∈ R^D = feature vector for sample i
y_i ∈ {1, 2, 3}
```

Usually `x_i` is a TF-IDF vector or count vector.


# 1. `lr`: Logistic Regression

Logistic Regression is a **linear probabilistic discriminative classifier**.

For multiclass classification, it usually uses the **softmax** function.

For each class `k`, the model learns a weight vector `w_k` and bias `b_k`.

The score for class `k` is:

```text
z_k = w_k^T x + b_k
```

Then the probability of class `k` is:

```text
P(y = k | x) = exp(z_k) / Σ_j exp(z_j)
```

For 3 classes:

```text
P(y = 1 | x), P(y = 2 | x), P(y = 3 | x)
```

The predicted class is:

```text
ŷ = argmax_k P(y = k | x)
```

## Loss function

Logistic Regression minimizes **cross-entropy loss**, also called **negative log-likelihood**:

```text
L = - Σ_i log P(y_i | x_i)
```

This means:

```text
If the correct class gets high probability → loss is low
If the correct class gets low probability → loss is high
```

## Regularization

In our code:

```python
LogisticRegression(C=params["C"], max_iter=3000)
```

By default, scikit-learn Logistic Regression uses **L2 regularization**.

The regularized objective is approximately:

$$
\min_W \left[-\sum_{i=1}^{N}\log P(y_i \mid x_i)+\lambda \lVert W\rVert_2^2\right]
$$

where:

```text
||W||² = sum of squared weights
```

In scikit-learn:

```text
C = 1 / λ
```

So:

```text
larger C  → weaker regularization → model can fit more strongly
smaller C → stronger regularization → model is more conservative
```

For example:

```python
C = [1, 3, 10, 30]
```

`C = 30` means weaker regularization than `C = 1`.


Why is cross-entropy loss an appropriate loss function for logistic regression?  
A short proof comes from **maximum likelihood estimation**.

Logistic Regression defines a conditional probability model:

$$
P(y_i \mid x_i; W)
$$

we read it as "the probability of the label $y_i$, given the input $x_i$, under model parameters $W$."

For the whole training set, assuming samples are independent, the probability of observing all labels is:

$$
P(y_1,\dots,y_N \mid x_1,\dots,x_N; W)
=
\prod_{i=1}^{N} P(y_i \mid x_i; W)
$$

This is called the **likelihood**:

$$
\mathcal{L}_{\text{likelihood}}(W)
=
\prod_{i=1}^{N} P(y_i \mid x_i; W)
$$

We want the model parameters $W$ to make the observed correct labels as probable as possible, so we maximize:

$$
\max_W \prod_{i=1}^{N} P(y_i \mid x_i; W)
$$

Products are inconvenient, so we take log. Since log is increasing, maximizing the likelihood is equivalent to maximizing the log-likelihood:

$$
\max_W \sum_{i=1}^{N} \log P(y_i \mid x_i; W)
$$

Machine learning usually writes training as **minimization**, so we multiply by $-1$:

$$
\min_W -\sum_{i=1}^{N} \log P(y_i \mid x_i; W)
$$

Therefore, the loss function is:

$$
\mathcal{L}(W)
=
-\sum_{i=1}^{N}\log P(y_i \mid x_i; W)
$$

So this loss is good because minimizing it is exactly the same as finding the parameters that assign maximum probability to the correct training labels.

It also has a useful penalty behavior:

If the model gives high probability to the correct class:

$$
P(y_i \mid x_i) \approx 1
\quad \Rightarrow \quad
-\log P(y_i \mid x_i) \approx 0
$$

If the model gives low probability to the correct class:

$$
P(y_i \mid x_i) \approx 0
\quad \Rightarrow \quad
-\log P(y_i \mid x_i) \to \infty
$$

So the loss strongly punishes confident wrong predictions and rewards confident correct predictions. This makes it very suitable for probabilistic classifiers like Logistic Regression.


---

# 2. `nbsvm`: NB-SVM-style Logistic Regression

Despite the name, our model is not a real SVM. It is **NB-SVM-style Logistic Regression**.

It combines two ideas:

```text
Naive Bayes-style word weighting
+
Logistic Regression classifier
```

The key idea is to transform the original text features using a **Naive-Bayes log-count ratio**.


## Step 1: Count feature occurrence by class

For each word/feature `j` and class `k`, count how often feature `j` appears in documents of class `k`.

Let:

```text
n_{k,j} = total count of feature j in class k
```

Then apply smoothing:

```text
p_{k,j} = (n_{k,j} + α) / Σ_j (n_{k,j} + α)
```
or

$$
p_{k,j}=\frac{n_{k,j}+\alpha}{\sum_j n_{k,j}+\alpha D}
$$

Here `α` prevents zero probabilities.


## Step 2: Compute log-count ratio

For binary NB-SVM, the classic ratio is:

```text
r_j = log( P(feature j | positive class) / P(feature j | negative class) )
```

For multiclass, a common one-vs-rest version is:

```text
r_{k,j} = log( P(feature j | class k) / P(feature j | not class k) )
```
or:  

$$
r_{k,j}=\log\frac{P(x_j \mid y=k)}{P(x_j \mid y\neq k)}
$$  

So if a word is much more common in class 2 than outside class 2, its `r_{2,j}` becomes large.


## Step 3: Reweight features

The original feature vector is transformed:

```text
x'_k = x ⊙ r_k
```

where `⊙` means element-wise multiplication.

So each feature is multiplied by how class-indicative it is.

Example:

```text
Original TF-IDF feature:
x_j

NB log-count ratio:
r_j

New feature:
x'_j = x_j × r_j
```

In

$$
x'_k = x \odot r_k
$$

the vector

$$
r_k
$$

is the **Naive-Bayes log-count-ratio vector for class $k$**.

If the original feature vector has $D$ features:

$$
x = [x_1, x_2, \dots, x_D]
$$

then

$$
r_k = [r_{k,1}, r_{k,2}, \dots, r_{k,D}]
$$

where each component is:

$$
r_{k,j}=\log\frac{P(x_j \mid y=k)}{P(x_j \mid y\neq k)}
$$

So $r_k$ contains one NB log-ratio value for every feature/word, specifically for class $k$.

The operation

$$
x'_k = x \odot r_k
$$

means **element-wise multiplication**:

$$
x'_k =
[x_1 r_{k,1},\ x_2 r_{k,2},\ \dots,\ x_D r_{k,D}]
$$

So each feature in the original text vector is multiplied by how strongly that feature is associated with class $k$.

In simple words:

$$
x
$$

is the original TF-IDF/count vector.

$$
r_k
$$

is the class-specific NB importance vector.

$$
x'_k
$$

is the reweighted feature vector for class $k$.

If a word is much more common in class $k$ than outside class $k$, then:

$$
r_{k,j} > 0
$$

so that feature becomes more supportive of class $k$.

If a word is more common outside class $k$, then:

$$
r_{k,j} < 0
$$

so that feature becomes evidence against class $k$.


## Step 4: Train Logistic Regression

After this NB weighting, a Logistic Regression model is trained:

```text
P(y = k | x') = softmax(w_k^T x' + b_k)
```
or in other words:  

$$
z_k=w_k^T x'_k+b_k
$$

$$
P(y=k \mid x')=\frac{\exp(z_k)}{\sum_{j=1}^{K}\exp(z_j)}
$$

## Regularization

The Logistic Regression part is usually L2-regularized:

$$
\min_W \left[-\sum_{i=1}^{N}\log P(y_i \mid x'_i)+\lambda \lVert W\rVert_2^2\right]
$$

Again:

$$
C=\frac{1}{\lambda}
$$

So in our grid:

```python
"C": [1, 3, 10, 30]
```

larger `C` means weaker regularization.

## Intuition

Plain Logistic Regression learns from the text features directly.

NB-SVM-style Logistic Regression first asks:

```text
Which words are especially characteristic of each class?
```

Then it gives those words stronger influence before training Logistic Regression.

That is why NB-SVM-style models often work very well for text classification.


# 3. `cnb`: Complement Naive Bayes

Complement Naive Bayes is a variant of Naive Bayes designed to work better with **imbalanced text classification**.

Multinomial Naive Bayes estimates word probabilities using the documents inside each class.

Complement Naive Bayes instead estimates weights using the **complement of each class**, meaning:

```text
not class k
```

So for class `k`, it looks at all documents that are **not** in class `k`.


## Standard Naive Bayes idea

Naive Bayes is based on Bayes’ theorem. For classification, we want to estimate the probability that a document/input $x$ belongs to class $k$:

$$
P(y = k \mid x) \propto P(y = k) P(x \mid y = k)
$$

Where:

$P(y = k \mid x)$ means the probability that the label is class $k$, given the input document $x$.

$P(y = k)$ is the prior probability of class $k$. It tells us how common class $k$ is before looking at the document.

$P(x \mid y = k)$ is the likelihood of observing document $x$, assuming the document belongs to class $k$.

$\propto$ means “proportional to.” We use it because the full Bayes formula also contains $P(x)$ in the denominator, but $P(x)$ is the same for all classes, so it does not affect the final class ranking.


For text classification, the multinomial assumption says the document probability can be written as a product over all features/words:

$$
P(x \mid y = k) = \prod_{j=1}^{D} P(\text{feature } j \mid y = k)^{x_j}
$$

Where:

$D$ is the total number of features, usually the vocabulary size.

$j$ indexes each feature or word.

$x_j$ is the value of feature $j$ in document $x$. For example, it can be the count or TF-IDF value of word $j$.

$P(\text{feature } j \mid y = k)$ is the probability of feature/word $j$ appearing in class $k$.

$\prod_{j=1}^{D}$ means multiply over all features from $1$ to $D$.

The exponent $x_j$ means that if a word appears more often in the document, it has more influence on the likelihood.  

Taking the logarithm converts the product into a sum:

$$
\log P(y = k \mid x) \propto \log P(y = k) + \sum_{j=1}^{D} x_j \log P(\text{feature } j \mid y = k)
$$

Where:

$\log P(y = k \mid x)$ is the log posterior score for class $k$.

$\log P(y = k)$ is the log prior probability of class $k$.

$\sum_{j=1}^{D}$ means sum over all features.

$x_j$ shows how much feature $j$ appears in the document.

$\log P(\text{feature } j \mid y = k)$ is the log probability of feature $j$ under class $k$.

So the model gives each class a score:

$$
\text{score}_k = \log P(y = k) + \sum_{j=1}^{D} x_j \log P(\text{feature } j \mid y = k)
$$

Then it predicts the class with the highest score:

$$
\hat{y} = \arg\max_k \text{score}_k
$$

<span style="font-size:24px; color:hotpink;">

In simple words, Naive Bayes chooses the class whose word distribution best explains the document.

</span>


## Complement Naive Bayes formula

For each class $k$, Complement Naive Bayes estimates feature probabilities from the **complement class**, meaning all classes except class $k$:

$$
P(\text{feature } j \mid y \neq k)
$$

Where:

$k$ is the target class.

$j$ is the feature or word index.

$y \neq k$ means all training samples whose label is not class $k$.

$P(\text{feature } j \mid y \neq k)$ means the probability of feature $j$ in the complement of class $k$.

With additive smoothing, the complement feature probability is estimated as:

$$
\theta_{k,j}
=
\frac{n_{\neg k,j}+\alpha}
{\sum_{j=1}^{D} n_{\neg k,j}+\alpha D}
$$

Where:

$\theta_{k,j}$ is the estimated probability of feature $j$ in the complement of class $k$.

$n_{\neg k,j}$ is the count of feature $j$ in all classes except class $k$.

$\alpha$ is the smoothing parameter.

$D$ is the total number of features.

$\sum_{j=1}^{D} n_{\neg k,j}$ is the total feature count in all classes except class $k$.

The term $\alpha D$ adjusts the denominator because smoothing adds $\alpha$ to each of the $D$ features.


Then CNB computes a weight for each feature:

$$
w_{k,j}=\log \theta_{k,j}
$$

Where:

$w_{k,j}$ is the weight of feature $j$ for class $k$.

$\log \theta_{k,j}$ is the logarithm of the complement probability of feature $j$.

A larger value of $\theta_{k,j}$ means feature $j$ is common in the complement of class $k$.  

For a document vector $x$, the complement score for class $k$ can be written as:

$$
s_k=\sum_{j=1}^{D} x_j w_{k,j}
$$

Where:

$s_k$ is the complement score for class $k$.

$x_j$ is the value of feature $j$ in the document.

$w_{k,j}$ is the complement weight of feature $j$ for class $k$.

$\sum_{j=1}^{D}$ means we sum over all features.  

Because these weights are based on the complement of class $k$, the classifier often predicts the class with the smallest complement score:

$$
\hat{y}=\arg\min_k \sum_{j=1}^{D} x_j w_{k,j}
$$

Where:

$\hat{y}$ is the predicted class.

$\arg\min_k$ means choose the class $k$ that gives the smallest score.

$\sum_{j=1}^{D} x_j w_{k,j}$ is the total complement score for class $k$.

In simple words, CNB asks:

**For which class does this document look least like the complement?**

That class is selected as the prediction.

A small technical note: some implementations change the sign or normalize the weights internally, so they may use an equivalent $\arg\max$ form instead of $\arg\min$. The idea is the same: CNB uses statistics from the complement class to make the classification more stable, especially for imbalanced text data.


## Smoothing / regularization-like effect

In our code:

```python
ComplementNB(alpha=params["alpha"])
```

$alpha$ is the additive smoothing parameter. The smoothed probability can be written as:

$$
\theta_{k,j}=
\frac{\text{count}+\alpha}
{\text{total count}+\alpha D}
$$

Where:

$\theta_{k,j}$ is the estimated probability of feature $j$ for class $k$ or its complement, depending on the Naive Bayes variant.

$\text{count}$ is the number of times feature $j$ appears in the relevant class or complement class.

$\text{total count}$ is the total number of feature occurrences in that class or complement class.

$\alpha$ is the smoothing parameter.

$D$ is the total number of features.

The term $\alpha D$ appears in the denominator because $\alpha$ is added once for each of the $D$ features.

This is not regularization in exactly the same way as L2 regularization in Logistic Regression, but it plays a similar stabilizing role.

If $\alpha$ is too small, rare words can dominate too strongly.

If $\alpha$ is larger, the probabilities become smoother and less extreme.

In our hyperparameter grid:

```python
alpha = [0.1, 0.3, 0.5, 1.0]
```

we test different smoothing strengths.


# 4. `mnb`: Multinomial Naive Bayes

Multinomial Naive Bayes is a classic model for text classification.

It assumes that documents are generated from class-specific word distributions.

For class $k$, it estimates:

$$
P(\text{feature } j \mid y=k)
$$

Where:

$k$ is the class index.

$j$ is the feature or word index.

$P(\text{feature } j \mid y=k)$ is the probability of observing feature $j$ in documents from class $k$.  

## Prior probability

First, Multinomial Naive Bayes estimates the prior probability of each class:

$$
P(y=k)=\frac{N_k}{N}
$$

Where:

$P(y=k)$ is the prior probability of class $k$.

$N_k$ is the number of training documents in class $k$.

$N$ is the total number of training documents.

The prior tells the model how common class $k$ is before looking at the document features.


## Likelihood

For each class $k$ and feature $j$, Multinomial Naive Bayes estimates:

$$
\theta_{k,j}=P(\text{feature } j \mid y=k)
$$

Where:

$\theta_{k,j}$ is the probability of feature $j$ under class $k$. 
(in CNB, $\theta_{k,j}$ was the estimated probability of feature $j$ in the complement of class $k$.)  

$P(\text{feature } j \mid y=k)$ means the probability that feature $j$ appears in a document from class $k$.

With additive smoothing, this probability is estimated as:

$$
\theta_{k,j}
=
\frac{n_{k,j}+\alpha}
{\sum_{j=1}^{D} n_{k,j}+\alpha D}
$$

Where:

$n_{k,j}$ is the count of feature $j$ in class $k$.

$\alpha$ is the smoothing parameter.

$D$ is the total number of features.

$\sum_{j=1}^{D} n_{k,j}$ is the total feature count in class $k$.

$\alpha D$ is the total smoothing added to the denominator.


## Prediction formula

For a document vector $x$, Multinomial Naive Bayes computes a score for each class $k$:

$$
\text{score}_k
=
\log P(y=k)
+
\sum_{j=1}^{D} x_j \log \theta_{k,j}
$$

Where:

$\text{score}_k$ is the score assigned to class $k$.

$\log P(y=k)$ is the log prior probability of class $k$.

$x_j$ is the value of feature $j$ in the document.

$\log \theta_{k,j}$ is the log probability of feature $j$ under class $k$.

$\sum_{j=1}^{D}$ means summing over all features.

Then the model predicts the class with the highest score:

$$
\hat{y}
=
\arg\max_k \text{score}_k
$$

Where:

$\hat{y}$ is the predicted class.

$\arg\max_k$ means choosing the class $k$ that gives the largest score.


<span style="font-size:24px; color:hotpink;">

So Multinomial Naive Bayes chooses the class whose word distribution best explains the document.

</span>

## Probability form

The original probability form is:

$$
P(y=k \mid x)
\propto
P(y=k)
\prod_{j=1}^{D}
\theta_{k,j}^{x_j}
$$

Where:

$P(y=k \mid x)$ is the posterior probability of class $k$ given document $x$.

$P(y=k)$ is the prior probability of class $k$.

$\theta_{k,j}$ is the probability of feature $j$ under class $k$.

$x_j$ is the value or count of feature $j$ in document $x$.

$\prod_{j=1}^{D}$ means multiplying over all features.

$\propto$ means “proportional to.” The denominator $P(x)$ is omitted because it is the same for all classes and does not affect the final class ranking.

In practice, we use the log form:

$$
\log P(y=k \mid x)
\propto
\log P(y=k)
+
\sum_{j=1}^{D} x_j \log \theta_{k,j}
$$

This converts multiplication into addition and avoids numerical underflow.

## Smoothing

In our code:

```python
MultinomialNB(alpha=params["alpha"])
```

$alpha$ is again the additive smoothing parameter.

Without smoothing, if a word never appears in class $k$, then:

$$
P(\text{feature } j \mid y=k)=0
$$

This would make the whole document probability zero:

$$
P(y=k \mid x)
\propto
P(y=k)
\prod_{j=1}^{D}
\theta_{k,j}^{x_j}
=
0
$$

Smoothing prevents this by ensuring that every feature has a small nonzero probability.

A larger value of $\alpha$ gives smoother probabilities.

A smaller value of $\alpha$ gives sharper, more class-specific word probabilities.


# 5. `sgd`: SGDClassifier with logistic loss

Our code uses:

```python
SGDClassifier(
    loss="log_loss",
    alpha=params["alpha"],
    max_iter=100,
    tol=1e-4,
    random_state=17
)
```

>`SGDClassifier` is a **learning algorithm/classifier implementation** in scikit-learn.
>
>More precisely:
>
> `SGDClassifier` is not a separate mathematical model by itself. It is a **training algorithm framework** that can train different linear models using **Stochastic Gradient Descent**.

This model is also a Logistic Regression-style model, but it is trained using a different optimization method.

`LogisticRegression` usually uses batch solvers such as `lbfgs`, `liblinear`, or similar optimization algorithms.

`SGDClassifier` uses **Stochastic Gradient Descent**.

<span style="font-size:24px; color:hotpink;">

This means that instead of using the whole dataset at once for each update, it updates the model gradually using individual samples or small batches of samples. 

</span>


## Model formula

The model is still linear.

For binary classification, the linear score is:

$$
z = w^T x + b
$$

Where:

$z$ is the linear score.

$w$ is the weight vector learned by the model.

$x$ is the input feature vector.

$b$ is the bias term.

$w^T x$ is the dot product between the weight vector and the input feature vector.

The probability of class $1$ is computed using the sigmoid function:

$$
P(y=1 \mid x)=\frac{1}{1+\exp(-z)}
$$

Where:

$P(y=1 \mid x)$ is the probability that the input $x$ belongs to class $1$. 

The sigmoid function converts the linear score $z$ into a probability between $0$ and $1$. 

For multiclass classification, `SGDClassifier` often uses a one-vs-rest strategy.

For each class $k$, it trains a separate binary classifier:

$$
\text{class } k \text{ vs. all other classes}
$$

Each classifier learns a separate linear score:

$$
z_k = w_k^T x + b_k
$$

Where:

$z_k$ is the linear score for class $k$.

$w_k$ is the weight vector for class $k$.

$x$ is the input feature vector.

$b_k$ is the bias term for class $k$.

The model then uses these class-specific scores to make a final class decision.


## Logistic loss

For binary classification with labels $y_i \in \{0,1\}$, the logistic loss for one training sample is:

$$
\mathcal{L}_i
=
-\left[
y_i \log(p_i)
+
(1-y_i)\log(1-p_i)
\right]
$$

Where:

$\mathcal{L}_i$ is the loss for sample $i$.

$y_i$ is the true label of sample $i$.

$p_i$ is the predicted probability that sample $i$ belongs to class $1$.

$\log(p_i)$ rewards the model when it assigns high probability to the correct positive label.

$\log(1-p_i)$ rewards the model when it assigns low probability to the positive class for a negative example.

The predicted probability is:

$$
p_i=
\frac{1}
{1+\exp(-(w^T x_i+b))}
$$

Where:

$p_i$ is the predicted probability for sample $i$.

$x_i$ is the feature vector of sample $i$.

$w$ is the learned weight vector.

$b$ is the bias term.

For multiclass one-vs-rest classification, this binary logistic loss is applied separately for each class.


## Regularization

By default, `SGDClassifier` uses L2 regularization unless the `penalty` parameter is changed.

The regularized objective is approximately:

$$
\min_w
\left[
\frac{1}{N}
\sum_{i=1}^{N}
\mathcal{L}_i
+
\alpha \lVert w \rVert_2^2
\right]
$$

Where:

$\min_w$ means the model searches for the best weight vector $w$.

$N$ is the number of training samples.

$\mathcal{L}_i$ is the logistic loss for sample $i$.

$\frac{1}{N}\sum_{i=1}^{N}\mathcal{L}_i$ is the average logistic loss over the training set.

$\alpha$ is the regularization strength.

$\lVert w \rVert_2^2$ is the squared L2 norm of the weight vector.

The term $\alpha \lVert w \rVert_2^2$ penalizes large weights and helps reduce overfitting.

In our code, the tested values are:

```python
alpha = [1e-6, 3e-6, 1e-5, 3e-5]
```
> The values for `alpha` were chosen on a logarithmic scale in a small range suitable for high-dimensional sparse text features. In `SGDClassifier`, `alpha` directly controls the strength of L2 regularization. Since text classification uses many sparse TF-IDF features, very large values such as 1 or 100 would over-regularize the model, shrink the weights toward zero, and cause underfitting. Therefore, the search was limited to small values such as $10^{-6}$ to $3 \times 10^{-5}$, which provide reasonable regularization while still allowing the model to learn discriminative word patterns.

Here, $\alpha$ directly controls regularization strength.

Unlike Logistic Regression’s $C$, here:

$$
\text{larger } \alpha \Rightarrow \text{stronger regularization}
$$

$$
\text{smaller } \alpha \Rightarrow \text{weaker regularization}
$$

So:

$$
\alpha = 10^{-6}
\Rightarrow
\text{weakest regularization}
$$

$$
\alpha = 3 \times 10^{-5}
\Rightarrow
\text{strongest regularization}
$$


# Main difference between `lr` and `sgd`

Both `lr` and `sgd` are logistic models, but they are optimized differently.

| Model | Objective                         | Optimization                |
| ----- | --------------------------------- | --------------------------- |
| `lr`  | Logistic loss + L2 regularization | Batch solver                |
| `sgd` | Logistic loss + L2 regularization | Stochastic Gradient Descent |

So mathematically, they are similar.

However, computationally, they are different.

`LogisticRegression` usually uses a solver that works more globally on the optimization problem.

`SGDClassifier` updates the model step by step using samples or mini-batches.

This makes `SGDClassifier` especially useful for large sparse text datasets, because it can train efficiently on high-dimensional feature spaces.


# Summary table

| Model   | Type                               | Main idea                                                                  | Regularization / smoothing                         |
| ------- | ---------------------------------- | -------------------------------------------------------------------------- | -------------------------------------------------- |
| `lr`    | Linear discriminative model        | Uses $P(y=k \mid x)=\text{softmax}(w_k^T x+b_k)$                           | L2 regularization controlled by $C$                |
| `nbsvm` | NB-weighted Logistic Regression    | Uses $x'=x \odot r$, then trains Logistic Regression                       | NB smoothing + L2 regularization controlled by $C$ |
| `cnb`   | Generative Naive Bayes variant     | Uses complement probabilities such as $P(\text{feature } j \mid y \neq k)$ | Additive smoothing controlled by $\alpha$          |
| `mnb`   | Generative Naive Bayes             | Uses $\log P(y=k)+\sum_j x_j \log \theta_{k,j}$                            | Additive smoothing controlled by $\alpha$          |
| `sgd`   | Logistic Regression trained by SGD | Uses logistic loss with linear scores                                      | L2 regularization controlled by $\alpha$           |


# Very simple intuition

`lr` learns:

```text
Which words push the sentence toward each class?
```

`nbsvm` learns:

```text
First, find words that are strongly class-indicative using Naive Bayes;
then train Logistic Regression on those reweighted features.
```

`cnb` learns:

```text
What does each class not look like?
```

This is often helpful when classes are imbalanced.

`mnb` learns:

```text
Which words are most likely under each class?
```

`sgd` learns:

```text
A Logistic Regression-style classifier, but using faster iterative updates.
```

In our stacked model, all five are useful because they make different kinds of mistakes. The meta-learner then learns how much to trust each one.

</details>

In [68]:
# tune, stack, and evaluate
best_base_params = {}
base_tuning_results = []

for kind in BASE_MODELS:
    best_params, results = tune_base_model(
        kind,
        X_tr,
        y_part,
        BASE_PARAM_GRIDS[kind],
        n_folds=5,
        seed=17
    )

    best_base_params[kind] = best_params
    base_tuning_results.append(results)

base_tuning_results = pd.concat(base_tuning_results, ignore_index=True)

print("Best base-model parameters:")
display(pd.DataFrame([
    {"model": kind, **params}
    for kind, params in best_base_params.items()
]))

oof_stack, holdout_stack, stack_feature_names = build_stack_features(
    X_train=X_tr,
    y_train=y_part,
    X_test=X_va,
    best_params_by_model=best_base_params,
    n_folds=5,
    seed=17
)

best_meta_C, meta_tuning_results = tune_meta_C(
    oof_stack,
    y_part,
    C_grid=(0.03, 0.1, 0.3, 1, 3, 10, 30),
    n_folds=5,
    seed=17
)

meta = LogisticRegression(C=best_meta_C, max_iter=3000)
meta.fit(oof_stack, y_part)

proba_holdout = align_proba(meta, holdout_stack) # holdout_stack.shape is (n_samples_valid, n_classes * n_models) and contains the predicted probabilities from each base model for each class, averaged across CV folds. By passing holdout_stack to align_proba, we get the final predicted probabilities for each class on the holdout set, which can then be evaluated against the true labels y_valid using log loss, accuracy, and F1 score. The meta-learner has learned to combine the predictions of the base models in a way that optimizes performance on the training split, and now we are assessing how well this combination generalizes to the holdout set.
# proba_holdout.shape is (n_samples_valid, n_classes) and contains the predicted probabilities for each class on the holdout set after being processed by the meta-learner. Each column in proba_holdout corresponds to a class in CLASSES, and each row corresponds to a sample in the holdout set. The values in proba_holdout represent the meta-learner's final predicted probabilities for each class, which are derived from the stacking features that capture the predictions of all base models. These probabilities can be used to compute log loss, accuracy, and F1 score against the true labels y_valid to evaluate the performance of the stacked model on unseen data.

valid_predictions = CLASSES[np.argmax(proba_holdout, axis=1)] # valid_predictions.shape is (n_samples_valid,) and contains the predicted class labels for each sample in the holdout set. By taking the argmax of proba_holdout along axis=1, we get the index of the class with the highest predicted probability for each sample. We then use these indices to select the corresponding class labels from CLASSES, resulting in valid_predictions which contains the final predicted class labels (1, 2, or 3) for each sample in the holdout set. These predicted labels can be compared to the true labels y_valid to compute accuracy and F1 score, while proba_holdout can be used to compute log loss.

cm = confusion_matrix(y_valid, valid_predictions, labels=CLASSES)

print(
    f"Stacked model with tuned base models     "
    f"log loss: {log_loss(y_valid, proba_holdout, labels=CLASSES):.4f}"
    f"   accuracy: {accuracy_score(y_valid, valid_predictions):.4f}"
    f"   macro-F1: {f1_score(y_valid, valid_predictions, average='macro'):.4f}"
)

LR     params={'C': 1}   inner-CV log loss: 0.4853
LR     params={'C': 3}   inner-CV log loss: 0.4213
LR     params={'C': 10}   inner-CV log loss: 0.3932
LR     params={'C': 30}   inner-CV log loss: 0.3966

Best LR params: {'C': 10}   best inner-CV log loss: 0.3932

NBSVM  params={'C': 1}   inner-CV log loss: 0.5954
NBSVM  params={'C': 3}   inner-CV log loss: 0.4924
NBSVM  params={'C': 10}   inner-CV log loss: 0.4179
NBSVM  params={'C': 30}   inner-CV log loss: 0.3861

Best NBSVM params: {'C': 30}   best inner-CV log loss: 0.3861

CNB    params={'alpha': 0.1}   inner-CV log loss: 0.3736
CNB    params={'alpha': 0.3}   inner-CV log loss: 0.3963
CNB    params={'alpha': 0.5}   inner-CV log loss: 0.4198
CNB    params={'alpha': 1.0}   inner-CV log loss: 0.4732

Best CNB params: {'alpha': 0.1}   best inner-CV log loss: 0.3736

MNB    params={'alpha': 0.1}   inner-CV log loss: 0.3919
MNB    params={'alpha': 0.3}   inner-CV log loss: 0.3962
MNB    params={'alpha': 0.5}   inner-CV log loss: 0.42

,model,C,alpha
0,lr,10.0,NaN
1,nbsvm,30.0,NaN
2,cnb,NaN,0.10000
3,mnb,NaN,0.10000
4,sgd,NaN,0.00001


META LR C=0.03  inner-CV log loss: 0.3482
META LR C=0.1   inner-CV log loss: 0.3449
META LR C=0.3   inner-CV log loss: 0.3434
META LR C=1     inner-CV log loss: 0.3429
META LR C=3     inner-CV log loss: 0.3429
META LR C=10    inner-CV log loss: 0.3429
META LR C=30    inner-CV log loss: 0.3430

Best META LR C: 3.0   best inner-CV log loss: 0.3429

Stacked model with tuned base models     log loss: 0.3504   accuracy: 0.8687   macro-F1: 0.8689


<details>
<summary><span style="color:red">About this cell:</span></summary>  

This block is the **final workflow** of our stacking experiment:

```text
1. Tune each base model
2. Build stacking features
3. Tune the meta-learner
4. Train the final meta-learner
5. Evaluate on the holdout validation set
```

# 1. Store best base-model parameters

```python
best_base_params = {}
base_tuning_results = []
```

`best_base_params` will store the best hyperparameters for each base model.

For example:

```python
{
    "lr": {"C": 10},
    "nbsvm": {"C": 3},
    "cnb": {"alpha": 0.3},
    "mnb": {"alpha": 0.5},
    "sgd": {"alpha": 1e-5}
}
```

`base_tuning_results` stores the full tuning results for all parameter values, not only the best ones.


# 2. Tune every base model

```python
for kind in BASE_MODELS:
    best_params, results = tune_base_model(
        kind,
        X_tr,
        y_part,
        BASE_PARAM_GRIDS[kind],
        n_folds=5,
        seed=17
    )
```

This loop goes through all base models:

```python
["lr", "nbsvm", "cnb", "mnb", "sgd"]
```

For each model, it calls:

```python
tune_base_model(...)
```

This function tests different hyperparameters using 5-fold cross-validation.

For example, for `lr`, it tests:

```python
C = [1, 3, 10, 30]
```

For `sgd`, it tests:

```python
alpha = [1e-6, 3e-6, 1e-5, 3e-5]
```

The model is scored using log loss:

$$
\text{log loss}
=
-\sum_{i=1}^{N}
\log P(y_i \mid x_i)
$$


<span style="font-size:24px; color:hotpink;">

Lower log loss means better probability predictions.

</span>

# 3. Save the best parameters

```python
best_base_params[kind] = best_params
base_tuning_results.append(results)
```

For each model, the best hyperparameters are saved in `best_base_params`.

The full tuning table is appended to `base_tuning_results`.

So after the loop, we have:

```python
best_base_params
```

containing only the best setting for each model, and:

```python
base_tuning_results
```

containing all tried settings and their CV scores. 

# 4. Combine all base tuning results

```python
base_tuning_results = pd.concat(base_tuning_results, ignore_index=True)
```

Each call to `tune_base_model` returns a separate DataFrame.

This line combines them into one big DataFrame.

So instead of having separate result tables for `lr`, `nbsvm`, `cnb`, etc., we get one table containing all base-model tuning results.  


# 5. Display best base-model parameters

```python
print("Best base-model parameters:")
display(pd.DataFrame([
    {"model": kind, **params}
    for kind, params in best_base_params.items()
]))
```

This prints a clean table showing the best hyperparameters for each base model.

For example:

| model |   C |   alpha |
| ----- | --: | ------: |
| lr    |  10 |     NaN |
| nbsvm |   3 |     NaN |
| cnb   | NaN |     0.3 |
| mnb   | NaN |     0.5 |
| sgd   | NaN | 0.00001 |

This helps us see what values were selected before stacking.
 

# 6. Build stacking features

```python
oof_stack, holdout_stack, stack_feature_names = build_stack_features(
    X_train=X_tr,
    y_train=y_part,
    X_test=X_va,
    best_params_by_model=best_base_params,
    n_folds=5,
    seed=17
)
```

This is the main stacking step.

It creates two new feature matrices:

```python
oof_stack
holdout_stack
```

## `oof_stack`

`oof_stack` is the meta-learner training matrix.

Its shape is:

$$
N_{\text{train}} \times 15
$$

because we have:

$$
5 \text{ base models} \times 3 \text{ classes} = 15 \text{ features}
$$

Each row contains the out-of-fold class probabilities produced by the base models.

Example row:

```text
[
  lr_p_1, lr_p_2, lr_p_3,
  nbsvm_p_1, nbsvm_p_2, nbsvm_p_3,
  cnb_p_1, cnb_p_2, cnb_p_3,
  mnb_p_1, mnb_p_2, mnb_p_3,
  sgd_p_1, sgd_p_2, sgd_p_3
]
```

Important point:

`oof_stack` is built using **out-of-fold predictions**, so every training sample is predicted by a base model that did not train on that sample.

This prevents leakage.


## `holdout_stack`

`holdout_stack` is the meta-learner input for the validation/holdout set.

Its shape is:

$$
N_{\text{valid}} \times 15
$$

For the holdout set, each base model is trained across the 5 folds and its probabilities are averaged.

So this is fold averaging:

$$
P_{\text{holdout}}
=
\frac{1}{5}
\sum_{f=1}^{5}
P_{\text{holdout}}^{(f)}
$$

This gives a more stable holdout prediction for each base model.


# 7. Tune the meta-learner

```python
best_meta_C, meta_tuning_results = tune_meta_C(
    oof_stack,
    y_part,
    C_grid=(0.03, 0.1, 0.3, 1, 3, 10, 30),
    n_folds=5,
    seed=17
)
```

Now the code tunes the Logistic Regression meta-learner.

The meta-learner does not train on the original TF-IDF features.

It trains on:

```python
oof_stack
```

So the meta-learner learns from the probability outputs of the base models.

It tests different values of `C`:

```python
0.03, 0.1, 0.3, 1, 3, 10, 30
```

Again, `C` controls L2 regularization:

$$
C = \frac{1}{\lambda}
$$

So:

```text
larger C  → weaker regularization
smaller C → stronger regularization
```

The function returns:

```python
best_meta_C
```

which is the best regularization value for the meta Logistic Regression.

# 8. Train the final meta-learner

```python
meta = LogisticRegression(C=best_meta_C, max_iter=3000)
meta.fit(oof_stack, y_part)
```

Now the final meta-model is trained using the best `C`.

Its input is:

```python
oof_stack
```

Its target is:

```python
y_part
```

So it learns:

```text
base-model probabilities → final class label
```

In other words, it learns how to combine the five base models.

# 9. Predict probabilities on the holdout set

```python
proba_holdout = align_proba(meta, holdout_stack)
```

The trained meta-model predicts probabilities for the holdout set.

The input is:

```python
holdout_stack
```

not the original validation features.

So the prediction pipeline is:

```text
X_va
→ base models
→ holdout_stack
→ meta model
→ final probabilities
```

`align_proba` makes sure the probability columns are ordered according to:

```python
CLASSES = [1, 2, 3]
```

So:

```python
proba_holdout[:, 0] = probability of class 1
proba_holdout[:, 1] = probability of class 2
proba_holdout[:, 2] = probability of class 3
```

# 10. Convert probabilities into class predictions

```python
valid_predictions = CLASSES[np.argmax(proba_holdout, axis=1)]
```

`proba_holdout` contains probabilities.

For each validation sample, this line chooses the class with the highest probability.

For example, if one row is:

```python
[0.10, 0.75, 0.15]
```

then:

```python
np.argmax(...) = 1
```

and since:

```python
CLASSES = [1, 2, 3]
```

index `1` corresponds to class `2`.

So the predicted label is:

```python
2
```

Mathematically:

$$
\hat{y}
=
\arg\max_k P(y=k \mid x)
$$

# 11. Compute the confusion matrix

```python
cm = confusion_matrix(y_valid, valid_predictions, labels=CLASSES)
```

This compares the true validation labels:

```python
y_valid
```

with the predicted labels:

```python
valid_predictions
```

The confusion matrix shows how many examples were classified correctly or incorrectly for each class.

For example, for 3 classes, it produces a 3-by-3 matrix:

$$
3 \times 3
$$

Rows usually represent true classes, and columns represent predicted classes.

# 12. Print final evaluation metrics

```python
print(
    f"Stacked model with tuned base models     "
    f"log loss: {log_loss(y_valid, proba_holdout, labels=CLASSES):.4f}"
    f"   accuracy: {accuracy_score(y_valid, valid_predictions):.4f}"
    f"   macro-F1: {f1_score(y_valid, valid_predictions, average='macro'):.4f}"
)
```

This prints three final metrics on the holdout validation set.

## Log loss

```python
log_loss(y_valid, proba_holdout, labels=CLASSES)
```

Log loss evaluates the quality of predicted probabilities.

$$
\text{log loss}
=
-\frac{1}{N}
\sum_{i=1}^{N}
\log P(y_i \mid x_i)
$$

Lower is better.

It rewards confident correct predictions and strongly penalizes confident wrong predictions.

## Accuracy

```python
accuracy_score(y_valid, valid_predictions)
```

Accuracy measures the proportion of correct predictions:

$$
\text{accuracy}
=
\frac{\text{number of correct predictions}}
{\text{total number of predictions}}
$$

Higher is better.

## Macro-F1

```python
f1_score(y_valid, valid_predictions, average='macro')
```

Macro-F1 computes the F1 score separately for each class, then averages them equally.

$$
\text{Macro-F1}
=
\frac{1}{K}
\sum_{k=1}^{K}
F1_k
$$

This is useful when class balance matters, because each class contributes equally.

# In simple words

This code first finds the best version of each base model. Then it uses those tuned base models to create probability-based features. These features are used to train a second-level Logistic Regression model. Finally, the stacked model is evaluated once on the holdout validation set using log loss, accuracy, and macro-F1.

The key idea is:

```text
Instead of manually averaging the five base models,
the meta-model learns how to combine them.
```
</details>

The tuned stacked model achieved **0.3504 log loss**, **0.8687 accuracy**, and **0.8689 macro-F1** on the holdout set. This is the strongest classical model so far. Compared with the tuned TF-IDF 3-model average, the stacked model improves both accuracy and log loss, with the most important gain coming from log loss.

The base-model search showed that different classifiers preferred different levels of regularization or smoothing. Logistic Regression performed best with **C=10**, NB-SVM-style Logistic Regression performed best with **C=30**, Complement Naive Bayes and Multinomial Naive Bayes both preferred **alpha=0.1**, and SGD Logistic Regression performed best with **alpha=1e-5**. This supports the idea that the base models are complementary rather than identical.

The Logistic Regression meta-learner selected **C=3**, with an inner-CV log loss of **0.3429**. Its holdout log loss of **0.3504** is reasonably close to the inner-CV estimate, which suggests that the stacking setup generalized well and did not only memorize the training split.

Overall, stacking improved the probability estimates more effectively than a flat average. The accuracy gain is moderate, but the lower log loss is important because log loss is the competition metric. This makes the tuned stacked model the strongest classical sparse-text pipeline in this notebook.


#### A note on calibration

The stacked model uses a Logistic Regression meta-learner trained on out-of-fold base-model probabilities. Because the meta-learner is trained directly on probability outputs, it can learn how much to trust each base model and each class probability. In this notebook, the stacked model improves validation log loss compared with the flat average, which suggests that the probability estimates are already better calibrated for the competition metric. I therefore do not add a separate calibration model.


### Kaggle submission

For the final submission, I refit the TF-IDF vectorizers on the full training data and generate predictions for the official test set. I use the same stacking setup as before: base-model hyperparameters are selected with cross-validation on the full training data, out-of-fold base-model probabilities are used to train the Logistic Regression meta-learner, and test-set probabilities are averaged across folds for each base model.

The printed level-2 OOF score evaluates the meta-learner on the already-built stacking features. In other words, it estimates how well the meta-learner combines the out-of-fold probability outputs of the base models. This score is useful as a sanity check, but it is not a full nested cross-validation estimate of the entire pipeline, because the full TF-IDF, base-model tuning, stacking-feature construction, and meta-model tuning steps are not repeated inside an outer validation loop.

The final CSV is saved in Kaggle's required column order: `id`, `EAP`, `HPL`, `MWS`.



In [69]:
# Refit TF-IDF on the full training data.
wv_full, cv_full = build_tfidf(train_texts["text"])

X_full = tfidf_features(wv_full, cv_full, train_texts["text"])
X_test = tfidf_features(wv_full, cv_full, test_texts["text"])

y_full = train_texts["author_code"].values

# Tune base-model hyperparameters on the full training data.
full_best_base_params = {}
full_base_tuning_results = []

for kind in BASE_MODELS:
    best_params, results = tune_base_model(
        kind,
        X_full,
        y_full,
        BASE_PARAM_GRIDS[kind],
        n_folds=5,
        seed=17
    )

    full_best_base_params[kind] = best_params
    full_base_tuning_results.append(results)

full_base_tuning_results = pd.concat(full_base_tuning_results, ignore_index=True)

print("Best full-data base-model parameters:")
display(pd.DataFrame([
    {"model": kind, **params}
    for kind, params in full_best_base_params.items()
]))

# Build full-data stacking features.
# oof_full is used to train the meta-learner.
# test_stack is the fold-averaged base-model prediction matrix for the official test set.
oof_full, test_stack, stack_feature_names = build_stack_features(
    X_train=X_full,
    y_train=y_full,
    X_test=X_test,
    best_params_by_model=full_best_base_params,
    n_folds=5,
    seed=17
)

# Tune the meta-learner on the full-data stacking features.
best_full_meta_C, full_meta_tuning_results = tune_meta_C(
    oof_full,
    y_full,
    C_grid=(0.03, 0.1, 0.3, 1, 3, 10, 30),
    n_folds=5,
    seed=17
)

# Compute a level-2 OOF estimate for the chosen meta-learner.
def meta_oof_proba(oof_stack, y, C, n_folds=5, seed=17):
    y = np.asarray(y)
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)

    oof_meta = np.zeros((oof_stack.shape[0], len(CLASSES)))

    for tr_idx, va_idx in skf.split(oof_stack, y):
        meta = LogisticRegression(C=C, max_iter=3000)
        meta.fit(oof_stack[tr_idx], y[tr_idx])
        oof_meta[va_idx] = align_proba(meta, oof_stack[va_idx]) # oof_meta.shape is (n_samples, n_classes) and contains the predicted probabilities from the meta-learner for each class, computed in an out-of-fold manner. For each fold, we train the meta-learner on the training portion of the stacking features (oof_stack[tr_idx]) and then predict probabilities on the validation portion (oof_stack[va_idx]). By filling oof_meta with these out-of-fold predictions, we get an unbiased estimate of the meta-learner's performance on the training data, which can be used to evaluate the log loss or other metrics without data leakage.

    return oof_meta


level2_oof = meta_oof_proba(
    oof_full,
    y_full,
    C=best_full_meta_C,
    n_folds=5,
    seed=17
)

level2_pred = CLASSES[np.argmax(level2_oof, axis=1)]

print(
    f"Level-2 OOF estimate, meta-learner only     "
    f"log loss: {log_loss(y_full, level2_oof, labels=CLASSES):.4f}"
    f"   accuracy: {accuracy_score(y_full, level2_pred):.4f}"
    f"   macro-F1: {f1_score(y_full, level2_pred, average='macro'):.4f}"
)

# Train the final meta-learner on all full-data stacking features.
meta_final = LogisticRegression(C=best_full_meta_C, max_iter=3000)
meta_final.fit(oof_full, y_full)

proba_test = align_proba(meta_final, test_stack)

# Make sure probabilities are valid for log loss and Kaggle submission.
proba_test = np.clip(proba_test, 1e-15, 1 - 1e-15)
proba_test = proba_test / proba_test.sum(axis=1, keepdims=True)

submission = pd.DataFrame({
    "id": test_texts.index,
    "EAP": proba_test[:, 0],   # class 1
    "HPL": proba_test[:, 2],   # class 3
    "MWS": proba_test[:, 1],   # class 2
})

submission_path = OUTPUT_DIR / "spooky_submission.csv"
submission.to_csv(submission_path, index=False)

print("Wrote", submission_path, submission.shape)
submission.head()

# oof_full.shape # (19579, 15) = (n_samples_train, n_classes * n_models)

LR     params={'C': 1}   inner-CV log loss: 0.4502
LR     params={'C': 3}   inner-CV log loss: 0.3902
LR     params={'C': 10}   inner-CV log loss: 0.3651
LR     params={'C': 30}   inner-CV log loss: 0.3699

Best LR params: {'C': 10}   best inner-CV log loss: 0.3651

NBSVM  params={'C': 1}   inner-CV log loss: 0.5452
NBSVM  params={'C': 3}   inner-CV log loss: 0.4496
NBSVM  params={'C': 10}   inner-CV log loss: 0.3817
NBSVM  params={'C': 30}   inner-CV log loss: 0.3564

Best NBSVM params: {'C': 30}   best inner-CV log loss: 0.3564

CNB    params={'alpha': 0.1}   inner-CV log loss: 0.3483
CNB    params={'alpha': 0.3}   inner-CV log loss: 0.3730
CNB    params={'alpha': 0.5}   inner-CV log loss: 0.3949
CNB    params={'alpha': 1.0}   inner-CV log loss: 0.4425

Best CNB params: {'alpha': 0.1}   best inner-CV log loss: 0.3483

MNB    params={'alpha': 0.1}   inner-CV log loss: 0.3684
MNB    params={'alpha': 0.3}   inner-CV log loss: 0.3763
MNB    params={'alpha': 0.5}   inner-CV log loss: 0.40

,model,C,alpha
0,lr,10.0,NaN
1,nbsvm,30.0,NaN
2,cnb,NaN,0.100000
3,mnb,NaN,0.100000
4,sgd,NaN,0.000003


META LR C=0.03  inner-CV log loss: 0.3222
META LR C=0.1   inner-CV log loss: 0.3187
META LR C=0.3   inner-CV log loss: 0.3171
META LR C=1     inner-CV log loss: 0.3167
META LR C=3     inner-CV log loss: 0.3167
META LR C=10    inner-CV log loss: 0.3167
META LR C=30    inner-CV log loss: 0.3167

Best META LR C: 30.0   best inner-CV log loss: 0.3167

Level-2 OOF estimate, meta-learner only     log loss: 0.3167   accuracy: 0.8830   macro-F1: 0.8834
Wrote outputs/spooky_submission.csv (8392, 4)


,id,EAP,HPL,MWS
0,id02310,0.022533,0.006092,0.971375
1,id24541,0.970184,0.013136,0.016679
2,id00134,0.011922,0.982015,0.006063
3,id27757,0.753014,0.198553,0.048433
4,id04081,0.922217,0.035808,0.041975


<details>
<summary><span style="color:red">About this cell:</span></summary>  

This block is the **final training and submission stage**.

Earlier, we trained and evaluated the stacked model on a train/validation split:

```text
X_tr → training part
X_va → holdout validation part
```

Now, after deciding that the stacking approach works, this code uses **all available labeled training data** to train the final model and generate predictions for the official test set.


# 1. Refit TF-IDF on full training data

```python
wv_full, cv_full = build_tfidf(train_texts["text"])
```

This rebuilds the TF-IDF/vectorizer objects using the **entire training dataset**, not only the previous partial training split.

That is important because for final submission, we want to use all available training text to learn the vocabulary.

Then:

```python
X_full = tfidf_features(wv_full, cv_full, train_texts["text"])
X_test = tfidf_features(wv_full, cv_full, test_texts["text"]) # If test_texts["text"] contains words or characters that were not seen in the training set, the fitted TF-IDF vectorizer simply ignores them.
```

`X_full` is the feature matrix for all training texts.

`X_test` is the feature matrix for the official test texts.

Both are transformed using the same fitted TF-IDF objects:

```text
training text → X_full
test text     → X_test
```

Then:

```python
y_full = train_texts["author_code"].values
```

This extracts the true labels for the full training set.


# 2. Tune base models on the full training data

```python
full_best_base_params = {}
full_base_tuning_results = []
```

These will store the best hyperparameters and tuning results for the full-data run.

Then:

```python
for kind in BASE_MODELS:
    best_params, results = tune_base_model(
        kind,
        X_full,
        y_full,
        BASE_PARAM_GRIDS[kind],
        n_folds=5,
        seed=17
    )
```

This loops over all base models:

```python
["lr", "nbsvm", "cnb", "mnb", "sgd"]
```

For each model, it performs 5-fold cross-validation on the **full training data** and selects the hyperparameters with the best inner-CV log loss.

For example:

```text
lr    → best C
nbsvm → best C
cnb   → best alpha
mnb   → best alpha
sgd   → best alpha
```

Then:

```python
full_best_base_params[kind] = best_params
full_base_tuning_results.append(results)
```

stores the selected parameters and the full tuning results.


# 3. Combine and display tuning results

```python
full_base_tuning_results = pd.concat(full_base_tuning_results, ignore_index=True)
```

This combines all base-model tuning results into one DataFrame.

Then:

```python
display(pd.DataFrame([
    {"model": kind, **params}
    for kind, params in full_best_base_params.items()
]))
```

prints a clean table showing the best full-data hyperparameters.

For example, it might display something like:

| model |   C |   alpha |
| ----- | --: | ------: |
| lr    |  10 |     NaN |
| nbsvm |   3 |     NaN |
| cnb   | NaN |     0.3 |
| mnb   | NaN |     0.5 |
| sgd   | NaN | 0.00001 |


# 4. Build full-data stacking features

```python
oof_full, test_stack, stack_feature_names = build_stack_features(
    X_train=X_full,
    y_train=y_full,
    X_test=X_test,
    best_params_by_model=full_best_base_params,
    n_folds=5,
    seed=17
)
```

This creates two important matrices:

```python
oof_full
test_stack
```

## `oof_full`

`oof_full` is the level-1 feature matrix for training the meta-learner.

Its shape is:

$$
N_{\text{train}} \times 15
$$

because we have:

$$
5 \text{ base models} \times 3 \text{ classes} = 15 \text{ features}
$$

Each row contains out-of-fold predicted probabilities from the base models.

Example row:

```text
[
  lr_p_1, lr_p_2, lr_p_3,
  nbsvm_p_1, nbsvm_p_2, nbsvm_p_3,
  cnb_p_1, cnb_p_2, cnb_p_3,
  mnb_p_1, mnb_p_2, mnb_p_3,
  sgd_p_1, sgd_p_2, sgd_p_3
]
```

Important: each training example gets base-model probabilities from models that did **not** train on that example. That prevents direct training leakage into the meta-learner.


## `test_stack`

`test_stack` is the level-1 feature matrix for the official test set.

Its shape is:

$$
N_{\text{test}} \times 15
$$

For the test set, each base model is trained across 5 folds and predicts the test set each time. Then the probabilities are averaged:

$$
P_{\text{test}}
=
\frac{1}{5}
\sum_{f=1}^{5}
P_{\text{test}}^{(f)}
$$

For **each base model**, the process is:

```text
Split full training data into 5 folds.
```

Then for fold 1:

```text
Train on folds 2, 3, 4, 5
Predict probabilities for the official test set
```

For fold 2:

```text
Train on folds 1, 3, 4, 5
Predict probabilities for the official test set
```

And so on until fold 5.

So for one base model, each test sample gets **5 probability predictions**:

$$
P_{\text{test}}^{(1)}, P_{\text{test}}^{(2)}, P_{\text{test}}^{(3)}, P_{\text{test}}^{(4)}, P_{\text{test}}^{(5)}
$$

Then we average them:

$$
P_{\text{test}}
=
\frac{1}{5}
\sum_{f=1}^{5}
P_{\text{test}}^{(f)}
$$

So:

**5 trained versions of the same base model → 5 probability predictions per test sample → average them into one final probability vector per test sample for that base model.**

Because we have **5 base models**, this happens separately for:

```text
lr
nbsvm
cnb
mnb
sgd
```

So in total, the code trains:

$$
5 \text{ base models} \times 5 \text{ folds} = 25
$$

base-model fold versions.

Each base model contributes 3 averaged class probabilities to `test_stack`.

So the final `test_stack` has:

$$
5 \times 3 = 15
$$

columns.

In simple terms:

```text
test text
→ 5 fold-trained LR models → averaged LR probabilities
→ 5 fold-trained NB-SVM models → averaged NB-SVM probabilities
→ 5 fold-trained CNB models → averaged CNB probabilities
→ 5 fold-trained MNB models → averaged MNB probabilities
→ 5 fold-trained SGD models → averaged SGD probabilities
→ 15-dimensional test_stack row
→ meta-learner
→ final prediction
```

And importantly: the official test labels are never used. The test set is only passed through the trained models to generate probabilities.  

So `test_stack` contains fold-averaged base-model probabilities for the official test data.


# 5. Tune the meta-learner on full-data stacking features

```python
best_full_meta_C, full_meta_tuning_results = tune_meta_C(
    oof_full,
    y_full,
    C_grid=(0.03, 0.1, 0.3, 1, 3, 10, 30),
    n_folds=5,
    seed=17
)
```

Now the code tunes the Logistic Regression meta-learner using `oof_full`.

The meta-learner is trained on base-model probabilities, not raw TF-IDF features.

It tries different values of `C`:

```python
0.03, 0.1, 0.3, 1, 3, 10, 30
```

In Logistic Regression:

$$
C = \frac{1}{\lambda}
$$

So:

```text
larger C  → weaker regularization
smaller C → stronger regularization
```

The function returns the best value:

```python
best_full_meta_C
```

# 6. Define `meta_oof_proba`

```python
def meta_oof_proba(oof_stack, y, C, n_folds=5, seed=17): # oof_stack.shape is (n_samples_train, n_classes * n_models)
```

This helper function computes an **out-of-fold estimate for the meta-learner**.

It does this:

```text
Split oof_stack into 5 folds
Train meta-learner on 4 folds
Predict the remaining fold
Repeat until every sample has an OOF meta prediction
```

Inside:

```python
oof_meta = np.zeros((oof_stack.shape[0], len(CLASSES)))
```

This creates a matrix to store meta-level predicted probabilities.

Its shape is:

$$
N_{\text{train}} \times 3
$$

Then:

```python
for tr_idx, va_idx in skf.split(oof_stack, y):
    meta = LogisticRegression(C=C, max_iter=3000)
    meta.fit(oof_stack[tr_idx], y[tr_idx])
    oof_meta[va_idx] = align_proba(meta, oof_stack[va_idx]) # oof_meta.shape is (n_samples, n_classes)
```

For each fold, it trains a Logistic Regression meta-model on part of `oof_stack` and predicts the held-out part.

Finally:

```python
return oof_meta
```

returns OOF probabilities from the meta-learner.


# 7. Compute level-2 OOF predictions

```python
level2_oof = meta_oof_proba(
    oof_full,
    y_full,
    C=best_full_meta_C,
    n_folds=5,
    seed=17
)
```

This gives an approximate cross-validated performance estimate for the second-level model.

Then:

```python
level2_pred = CLASSES[np.argmax(level2_oof, axis=1)]
```

converts probabilities into class labels.

For each sample, it chooses the class with the largest predicted probability:

$$
\hat{y}_i
=
\arg\max_k P(y_i=k \mid x_i)
$$

# 8. Print level-2 OOF metrics

```python
print(
    f"Level-2 OOF estimate, meta-learner only     "
    f"log loss: {log_loss(y_full, level2_oof, labels=CLASSES):.4f}"
    f"   accuracy: {accuracy_score(y_full, level2_pred):.4f}"
    f"   macro-F1: {f1_score(y_full, level2_pred, average='macro'):.4f}"
)
```

This prints three metrics using the level-2 OOF predictions.

## Log loss

$$
\text{log loss}
=
-\frac{1}{N}
\sum_{i=1}^{N}
\log P(y_i \mid x_i)
$$

Lower is better.

## Accuracy

$$
\text{accuracy}
=
\frac{\text{number of correct predictions}}
{\text{total number of predictions}}
$$

Higher is better.

## Macro-F1

$$
\text{Macro-F1}
=
\frac{1}{K}
\sum_{k=1}^{K}
F1_k
$$

Higher is better.

One technical note: this is a useful internal estimate, but it is not as clean as the earlier holdout validation score. The earlier holdout score is still the better evidence for generalization, because it evaluates the entire pipeline on data that was not used during model selection.

<details>
<summary><span style="color:red">More on this:</span></summary>  

By **“the earlier holdout validation score”**, I mean the score we got in the previous experiment where we split the original training data into two parts:

```text
training split  → X_tr, y_part
validation split → X_va, y_valid
```

Then we trained/tuned the stacked model using only:

```text
X_tr, y_part
```

and evaluated it once on:

```text
X_va, y_valid
```

That score came from this part:

```python
proba_holdout = align_proba(meta, holdout_stack)
valid_predictions = CLASSES[np.argmax(proba_holdout, axis=1)]

print(
    f"Stacked model with tuned base models     "
    f"log loss: {log_loss(y_valid, proba_holdout, labels=CLASSES):.4f}"
    f"   accuracy: {accuracy_score(y_valid, valid_predictions):.4f}"
    f"   macro-F1: {f1_score(y_valid, valid_predictions, average='macro'):.4f}"
)
```

So **holdout validation score** means:

> the score on the separate validation set that was held out from training and model tuning.

For example, if we used a 70/30 split:

```text
70% training data → used for fitting, CV tuning, stacking
30% validation data → kept aside and used only for final evaluation
```

That 30% validation score is what I called the **earlier holdout validation score**.

It is cleaner evidence for generalization because the validation texts were not used to tune the base models, build the meta-learner, or choose hyperparameters.

By contrast, the later:

```text
Level-2 OOF estimate
```

is computed after we refit everything on the **full training data**. It is useful, but it is more of an internal cross-validation sanity check, not the same as evaluating the full pipeline on a completely held-out validation split.


</details> 


# 9. Train final meta-learner on all stacking features

```python
meta_final = LogisticRegression(C=best_full_meta_C, max_iter=3000)
meta_final.fit(oof_full, y_full)
```

<details>
<summary><span style="color:red">

Is `best_full_meta_C` selected through internal cross-validation on `oof_full`, rather than from a completely untouched holdout set?:</span></summary>  

Right. `best_full_meta_C` is **not obtained from a completely held-out validation split**.

It is obtained by cross-validation on:

```python
oof_full #oof_full.shape = (19579, 15) = (n_samples_train, n_classes * n_models)
```

where `oof_full` contains the out-of-fold base-model probability features for the **full training data**.

So the process is:

```text
full training data
→ build base-model OOF probability features
→ oof_full
→ cross-validate Logistic Regression meta-learner on oof_full
→ choose best_full_meta_C
```

So `best_full_meta_C` is selected using **internal cross-validation**, not a separate untouched holdout set.

That is why this score:

```text
Level-2 OOF estimate, meta-learner only
```

is a useful sanity check, but not as clean as a real holdout validation score.

A real holdout setup would be:

```text
train split → tune base models, build stack, tune meta C
holdout split → evaluate once, without using it for model selection
```

But our final Kaggle submission setup is different:

```text
all labeled training data → tune everything with CV → train final stack → predict official test set
```

This is normal for Kaggle submissions. Once we have chosen the modeling approach using earlier validation experiments, it is common to refit and tune on the full training set before generating the final test predictions.


</details>

Now the final meta-model is trained on all full-data stacking features.

It learns:

```text
base-model probabilities → final author class
```

This is the final second-level model used for test prediction.


# 10. Predict probabilities for the official test set

```python
proba_test = align_proba(meta_final, test_stack)
```

The final meta-model predicts class probabilities for the official test set.

The input is not raw text. The input is `test_stack`, which contains fold-averaged base-model predictions.

The full prediction pipeline is:

```text
test_texts
→ TF-IDF features
→ base models
→ test_stack
→ final meta-learner
→ proba_test
```

`proba_test` has shape:

$$
N_{\text{test}} \times 3
$$

Each row contains:

```text
probability of class 1
probability of class 2
probability of class 3
```

# 11. Clip and renormalize probabilities

```python
proba_test = np.clip(proba_test, 1e-15, 1 - 1e-15)
proba_test = proba_test / proba_test.sum(axis=1, keepdims=True)
```

This makes sure the probabilities are valid and safe for log loss.

Clipping prevents exact zeros or ones.

That matters because log loss contains:

$$
\log P(y_i \mid x_i)
$$

If a model predicts probability exactly zero for the true class, then:

$$
\log(0)
$$

is undefined, and the loss becomes infinite.

So this line:

```python
np.clip(proba_test, 1e-15, 1 - 1e-15)
```

forces probabilities to stay inside this range:

$$
10^{-15}
\leq
p
\leq
1-10^{-15}
$$

Then:

```python
proba_test = proba_test / proba_test.sum(axis=1, keepdims=True)
```

renormalizes each row so that the class probabilities sum to 1:

$$
\sum_{k=1}^{3} P(y=k \mid x)=1
$$

# 12. Create the Kaggle submission file

```python
submission = pd.DataFrame({
    "id": test_texts.index,
    "EAP": proba_test[:, 0],   # class 1
    "HPL": proba_test[:, 2],   # class 3
    "MWS": proba_test[:, 1],   # class 2
})
```

This creates the final submission DataFrame.

The `id` column comes from the test set index.

The probability columns are arranged according to the competition’s required author order:

```text
EAP, HPL, MWS
```

But our internal class order is:

```python
CLASSES = [1, 2, 3]
```

and our mapping is:

```text
class 1 → EAP
class 2 → MWS
class 3 → HPL
```

That is why the columns are assigned as:

```python
"EAP": proba_test[:, 0]   # class 1
"HPL": proba_test[:, 2]   # class 3
"MWS": proba_test[:, 1]   # class 2
```

This part is very important. If the class-to-author mapping is wrong, the Kaggle score will be bad even if the model is good.

# 13. Save the submission

```python
submission_path = OUTPUT_DIR / "spooky_submission.csv"
submission.to_csv(submission_path, index=False)
```

This saves the submission file as:

```text
spooky_submission.csv
```

Then:

```python
print("Wrote", submission_path, submission.shape)
submission.head()
```

prints the file path and shape, then displays the first few rows.

# Overall meaning

This block takes the model design that worked on validation and retrains it using all available labeled data. It rebuilds TF-IDF on the full training set, tunes base models with cross-validation, builds full-data stacking features, tunes the Logistic Regression meta-learner, trains the final stack, predicts probabilities for the official test set, fixes probability formatting, and writes the Kaggle submission file.

The high-level pipeline is:

```text
Full training text
→ TF-IDF
→ tuned base models
→ OOF stacking features (By “OOF stacking features”, I mean the out-of-fold predicted probabilities produced by the base models, which become the input features for the meta-learner.)
→ tuned Logistic Regression meta-learner
→ official test probabilities
→ Kaggle submission CSV
```

</details>

The final stacked model produced the best results in the notebook. On the full training data, the level-2 out-of-fold estimate reached **0.3167 log loss**, **0.8830 accuracy**, and **0.8834 macro-F1**. Because this estimate is computed from out-of-fold stacking features built on the full training set, it uses a different evaluation setup from the earlier 70/30 holdout experiment. Therefore, it should be treated mainly as a sanity check for the meta-learner rather than as a score directly comparable to the holdout validation rows. Within this setup, the result suggests that the Logistic Regression meta-learner combines the base-model probability outputs effectively.

The base-model tuning results show that different model families preferred different hyperparameter settings. Logistic Regression performed best with **C=10**, NB-SVM-style Logistic Regression with **C=30**, Complement Naive Bayes and Multinomial Naive Bayes with **alpha=0.1**, and SGD Logistic Regression with **alpha=3e-6**. For the meta-learner, several values of **C** gave very similar rounded log-loss values, with **C=30** selected as the best value by the unrounded cross-validation score. These results are consistent with the idea that the stack benefits from combining diverse sparse-text classifiers.

After refitting the full stacked pipeline and submitting the generated probability file to Kaggle, the model achieved a **private score of 0.30414** and a **public score of 0.33621**. These leaderboard scores are reasonably close to the level-2 OOF estimate, with the private score slightly better and the public score slightly worse. This is a useful sign that the validation setup was reasonably informative, although the level-2 OOF score is still not a fully nested cross-validation estimate of the entire pipeline. Overall, the final stacked model is the strongest classical sparse-text pipeline in this project, with the main improvement coming from better probabilistic predictions rather than only higher classification accuracy.


<details>
<summary><span style="color:red">More on "This is a useful sign that the validation setup was reasonably informative, although the level-2 OOF score is still not a fully nested cross-validation estimate of the entire pipeline.":</span></summary>  

It means this:

Our **level-2 OOF log loss** was:

$$
0.3167
$$

<details>
<summary><span style="color:red">How exactly we got this log loss:</span></summary>  

We got the **level-2 OOF log loss = 0.3167** from evaluating the **meta-learner** with cross-validation on the stacking features.

The process was:

1. First, we trained the base models in 5-fold CV and created:

```python
oof_full
```

<details>
<summary><span style="color:red">How:</span></summary>  

In that stage, we did this part:

```python
oof_full, test_stack, stack_feature_names = build_stack_features(
    X_train=X_full,
    y_train=y_full,
    X_test=X_test,
    best_params_by_model=full_best_base_params,
    n_folds=5,
    seed=17
)
```

More exactly, for **each base model** — `lr`, `nbsvm`, `cnb`, `mnb`, `sgd` — we split the full training data into 5 folds.

For one base model, the process was:

```text
Fold 1:
train on folds 2,3,4,5
predict probabilities for fold 1

Fold 2:
train on folds 1,3,4,5
predict probabilities for fold 2

Fold 3:
train on folds 1,2,4,5
predict probabilities for fold 3

Fold 4:
train on folds 1,2,3,5
predict probabilities for fold 4

Fold 5:
train on folds 1,2,3,4
predict probabilities for fold 5
```

So every training sample received predicted probabilities from a model that **did not train on that sample**.

For each sample, each base model produced 3 probabilities:

$$
P(y=1 \mid x),\ P(y=2 \mid x),\ P(y=3 \mid x)
$$

Since we had 5 base models, each training sample got:

$$
5 \times 3 = 15
$$

probability features.

These were stored in:

```python
oof_full
```

So:

```python
oof_full.shape = (number_of_training_samples, 15)
```

In simple words:

**At this stage, we converted the original TF-IDF training data into a new training matrix made of out-of-fold probability predictions from the base models. This new matrix became the input data for the Logistic Regression meta-learner.**

At the same time, for the official test set, each fold-trained base model also predicted the test samples, and those 5 predictions were averaged. Those averaged test probabilities were stored in:

```python
test_stack
```

</details>

This matrix contains **out-of-fold base-model probabilities** for every training sample.

Its shape is:

$$
N_{\text{train}} \times 15
$$

because:

$$
5 \text{ base models} \times 3 \text{ class probabilities} = 15
$$

So each row of `oof_full` looks like:

$$
[
p_{\text{lr},1}, p_{\text{lr},2}, p_{\text{lr},3},
\dots,
p_{\text{sgd},1}, p_{\text{sgd},2}, p_{\text{sgd},3}
]
$$

2. Then we tuned the meta-learner and selected:

```python
best_full_meta_C = 30.0
```

3. Then `meta_oof_proba(...)` split `oof_full` into 5 folds again. For each fold:

```text
train Logistic Regression meta-learner on 4 folds of oof_full
predict probabilities for the remaining fold
```

So every training sample got a **level-2 OOF prediction** from a meta-model that did not train on that sample.

4. These predictions were stored in:

```python
level2_oof
```

Its shape is:

$$
N_{\text{train}} \times 3
$$

Each row contains the final meta-learner probabilities for the three authors/classes.

5. Finally, we computed log loss between the true labels and these level-2 OOF probabilities:

```python
log_loss(y_full, level2_oof, labels=CLASSES)
```

Mathematically:

$$
\text{log loss}
=
-\frac{1}{N}
\sum_{i=1}^{N}
\log P(y_i \mid x_i)
$$

That gave:

$$
0.3167
$$

So, in simple words:

**The 0.3167 score comes from cross-validating the Logistic Regression meta-learner on the OOF probability features produced by the base models. It measures how well the meta-learner combines the base-model predictions on the full training set.**


</details>

Our Kaggle scores were:

$$
\text{private score}=0.30414
$$

$$
\text{public score}=0.33621
$$

These numbers are in a similar range. So our internal estimate was not completely misleading. That is why I said:

**“the validation setup was reasonably informative”**

Meaning:

> The internal validation/OOF results gave a useful rough idea of how the model would perform on Kaggle.

But the second part says:

**“although the level-2 OOF score is still not a fully nested cross-validation estimate of the entire pipeline.”**

Meaning:

> Even though the OOF score is useful, it is not a perfect unbiased evaluation of the whole model-building process.

Why? Because a fully nested CV would repeat **everything** inside outer folds:

```text
TF-IDF fitting
base-model hyperparameter tuning
base-model OOF prediction
meta-model tuning
final evaluation
```

<details>
<summary><span style="color:red">More on this:</span></summary>  

Think of **fully nested CV** as asking:

> What would happen if I repeated my entire modeling process from zero on several train/validation splits?

Not just the meta-learner. **Everything.**

In our current level-2 OOF estimate, we already have:

```python
oof_full
```

That means the base-model stacking features have already been created. Then we only cross-validate the meta-learner on those features:

```text
already-built oof_full
→ CV Logistic Regression meta-learner
→ level-2 OOF score
```

So the level-2 OOF score mainly answers:

> Given these stacking features, how well does the meta-learner combine them?

But a **fully nested CV** would do something stricter.

For example, suppose we use 5 outer folds:

```text
Outer fold 1:
  hold out 20% of training data
  use remaining 80% to do EVERYTHING:
    fit TF-IDF
    tune base models
    build base-model OOF predictions
    tune meta-learner
    train final stack
  evaluate once on the held-out 20%

Outer fold 2:
  hold out a different 20%
  repeat EVERYTHING from scratch
  evaluate on that held-out 20%

...
```

So in fully nested CV, the held-out outer fold is not used for:

```text
TF-IDF vocabulary fitting
base-model tuning
meta-model tuning
choosing C
building training stacking features
```

It is only used at the very end for evaluation.

That is why it is “cleaner.”

Our level-2 OOF score is still useful, but it does **not** repeat the whole pipeline from scratch inside each outer fold. It only cross-validates the meta-learner after the stacking features already exist.

A simple comparison:

```text
Our level-2 OOF:
base stacking features already built
→ CV meta-learner
→ score
```

```text
Fully nested CV:
raw text
→ fit TF-IDF
→ tune base models
→ build stacking features
→ tune meta-learner
→ evaluate on untouched outer fold
→ repeat for each outer fold
```

So the key difference is:

**Our level-2 OOF score evaluates the second-level model. Fully nested CV would evaluate the entire modeling pipeline from raw text to final prediction.**

</details>

But our level-2 OOF score mainly evaluates the **meta-learner on already-created stacking features**. It does not fully repeat the whole pipeline inside an outer validation loop.

So a simpler version would be:

**The Kaggle scores are close to the level-2 OOF score, which suggests that the internal OOF estimate was useful. However, this OOF score should still be treated as an approximate sanity check, not as a fully independent validation score for the entire pipeline.**


</details>


## Error analysis: where does the best validated model still fail?

The stacked model gives the strongest overall validation result, but aggregate metrics do not show which authors are still being confused. To inspect this, I use the holdout predictions from the stacked model and examine the confusion matrix, per-author recall, the most common true-to-predicted confusions, and several high-confidence mistakes. This helps show whether the remaining errors are concentrated in one weaker author class or whether they come from stylistic overlap between specific authors.


> This error analysis uses the holdout validation predictions from the stacked model before the final full-data refit. After refitting the pipeline on all labeled samples, there is no independent labeled validation set left for inspecting mistakes. Therefore, the confusion matrix, recall values, and high-confidence mistakes are based on the earlier holdout validation split.


In [70]:
AUTHORS = {
    1: "EAP",
    2: "MWS",
    3: "HPL"
}

# Recompute the confusion matrix explicitly to avoid relying on an older variable.
cm = confusion_matrix(y_valid, valid_predictions, labels=CLASSES)

cm_df = pd.DataFrame(
    cm,
    index=[f"true_{AUTHORS[c]}" for c in CLASSES],
    columns=[f"pred_{AUTHORS[c]}" for c in CLASSES]
)

print("Confusion matrix:")
display(cm_df)

print("\nPer-author recall:")
recall_rows = []

for i, c in enumerate(CLASSES):
    correct = cm[i, i]
    total = cm[i].sum() # .sum() computes the total number of samples for the true class c by summing across the row i of the confusion matrix cm. This gives us the total count of samples that actually belong to class c, regardless of how they were predicted. The variable total represents the number of true instances of class c in the holdout set, which is used as the denominator when calculating recall for that class.
    recall = correct / total if total > 0 else 0

    recall_rows.append({
        "author": AUTHORS[c],
        "correct": correct,
        "total": total,
        "recall": recall
    })

recall_df = pd.DataFrame(recall_rows)
display(recall_df)

print("\nTop confusions, true -> predicted:")
confusion_rows = []

for i, true_label in enumerate(CLASSES):
    for j, pred_label in enumerate(CLASSES):
        if i != j:
            confusion_rows.append({
                "true_author": AUTHORS[true_label],
                "predicted_author": AUTHORS[pred_label],
                "count": cm[i, j]
            })

confusion_df = (
    pd.DataFrame(confusion_rows)
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

display(confusion_df.head(6))

# Optional: inspect high-confidence mistakes.
error_mask = y_valid != valid_predictions

error_examples = valid_texts.loc[error_mask, ["text"]].copy()
error_examples["true_author"] = [AUTHORS[c] for c in np.asarray(y_valid)[error_mask]]
error_examples["predicted_author"] = [AUTHORS[c] for c in np.asarray(valid_predictions)[error_mask]]
error_examples["model_confidence"] = proba_holdout[error_mask].max(axis=1)

for idx, c in enumerate(CLASSES):
    error_examples[f"p_{AUTHORS[c]}"] = proba_holdout[error_mask, idx]

error_examples = error_examples.sort_values("model_confidence", ascending=False)

print("\nHigh-confidence mistakes:")
display(error_examples.head(10))

Confusion matrix:


,pred_EAP,pred_MWS,pred_HPL
true_EAP,2084,166,120
true_MWS,191,1558,64
true_HPL,153,77,1461



Per-author recall:


,author,correct,total,recall
0,EAP,2084,2370,0.879325
1,MWS,1558,1813,0.859349
2,HPL,1461,1691,0.863986



Top confusions, true -> predicted:


,true_author,predicted_author,count
0,MWS,EAP,191
1,EAP,MWS,166
2,HPL,EAP,153
3,EAP,HPL,120
4,HPL,MWS,77
5,MWS,HPL,64



High-confidence mistakes:


,text,true_author,predicted_author,model_confidence,p_EAP,p_MWS,p_HPL
id,,,,,,,
id11010,"I remembered, too, strange stories told about ...",EAP,HPL,0.986255,0.008658,0.005087,0.986255
id07450,As the minuteness of the parts formed a great ...,MWS,EAP,0.977138,0.977138,0.014792,0.008071
id21727,I walked the cellar from end to end.,EAP,HPL,0.976580,0.014034,0.009386,0.976580
id20648,Upon one thing I was absolutely resolved.,HPL,EAP,0.975307,0.975307,0.015645,0.009048
id02152,"""Strange you shouldn't know me, though, isn't it?",EAP,HPL,0.974455,0.016718,0.008826,0.974455
id06355,At this juncture my brain became aware of a st...,HPL,EAP,0.974169,0.974169,0.016063,0.009768
id15922,"This proved however to be a second passage, wh...",MWS,EAP,0.973924,0.973924,0.016191,0.009885
id17190,"West of this door was a single window, looking...",EAP,HPL,0.973834,0.018043,0.008123,0.973834
id26842,"Even as it was, my hair stood on end, while I ...",EAP,HPL,0.972991,0.018527,0.008482,0.972991


<details>
<summary><span style="color:red">About this cell:</span></summary>  

This code performs **error analysis** for the stacked model on the holdout validation set.

It answers questions like:

```text
- Which authors are confused most often?

- Which author has the lowest recall? (recall means out of all the samples that truly belong to a certain class, how many did our model correctly identify as that class? It is calculated as the number of true positives (correctly predicted samples for that class) divided by the total number of actual positives (all samples that truly belong to that class). A high recall indicates that the model is good at capturing most of the relevant samples for that class, while a low recall suggests that the model is missing many samples of that class.)  

- Which mistakes was the model very confident about?
```


For this **error analysis section**, we should use the results from the earlier **holdout validation experiment**, before refitting the whole pipeline on all labeled data.

That means we should use:

```python
y_valid
valid_predictions
proba_holdout
valid_texts
```

These come from the earlier validation setup:

```text
training split → used to train/tune the stacked model
holdout validation split → used to evaluate errors
```

So our error analysis answers:

> Where does the best validated stacked model fail on unseen validation data?

That is appropriate because `y_valid` contains true labels, so we can compute the confusion matrix, recall, and high-confidence mistakes.

After we refit the full pipeline on **all labeled training data**, there is no separate labeled validation set left. The official Kaggle test set has no labels available to us, so we cannot do true error analysis on it.

So the correct logic is:

```text
Holdout stacked model:
X_va + y_valid available
→ can do error analysis

Full-data Kaggle model:
X_test available, but no y_test
→ can generate submission probabilities, but cannot do error analysis
```

So: for this section, we use the earlier holdout results, not the final full-data refit results.


## 1. Map numeric labels to author names

```python
AUTHORS = {
    1: "EAP",
    2: "MWS",
    3: "HPL"
}
```

Our model uses numeric class labels:

```text
1, 2, 3
```

This dictionary converts them into readable author names:

```text
1 → EAP
2 → MWS
3 → HPL
```

So instead of displaying class `1`, the notebook displays `EAP`.


## 2. Recompute the confusion matrix

```python
cm = confusion_matrix(y_valid, valid_predictions, labels=CLASSES)

# proba_holdout.shape is (n_samples_valid, n_classes) and contains the predicted probabilities for each class on the holdout set after being processed by the meta-learner. Each column in proba_holdout corresponds to a class in CLASSES, and each row corresponds to a sample in the holdout set. The values in proba_holdout represent the meta-learner's final predicted probabilities for each class, which are derived from the stacking features that capture the predictions of all base models. These probabilities can be used to compute log loss, accuracy, and F1 score against the true labels y_valid to evaluate the performance of the stacked model on unseen data.

# valid_predictions = CLASSES[np.argmax(proba_holdout, axis=1)] # valid_predictions.shape is (n_samples_valid,) and contains the predicted class labels for each sample in the holdout set. By taking the argmax of proba_holdout along axis=1, we get the index of the class with the highest predicted probability for each sample. We then use these indices to select the corresponding class labels from CLASSES, resulting in valid_predictions which contains the final predicted class labels (1, 2, or 3) for each sample in the holdout set. These predicted labels can be compared to the true labels y_valid to compute accuracy and F1 score, while proba_holdout can be used to compute log loss.

```

This compares:

```python
y_valid
```

the true labels, with:

```python
valid_predictions
```

the predicted labels from the stacked model.

The result is a confusion matrix.

For 3 classes, it has shape:

$$
3 \times 3
$$

Rows are true classes, columns are predicted classes.

So:

```text
cm[i, j]
```

means:

> number of validation samples whose true class is class `i`, but the model predicted class `j`.

The diagonal entries are correct predictions:

```text
true EAP → predicted EAP
true MWS → predicted MWS
true HPL → predicted HPL
```

The off-diagonal entries are mistakes.

## 3. Make the confusion matrix readable

```python
cm_df = pd.DataFrame(
    cm,
    index=[f"true_{AUTHORS[c]}" for c in CLASSES],
    columns=[f"pred_{AUTHORS[c]}" for c in CLASSES]
)
```

This converts the confusion matrix into a pandas DataFrame with readable row and column names.

For example:

|          | pred_EAP | pred_MWS | pred_HPL |
| -------- | -------: | -------: | -------: |
| true_EAP |      ... |      ... |      ... |
| true_MWS |      ... |      ... |      ... |
| true_HPL |      ... |      ... |      ... |

Then:

```python
display(cm_df)
```

shows the table.

## 4. Compute per-author recall

```python
recall_rows = []
```

This creates an empty list where recall results will be stored.

Then:

```python
for i, c in enumerate(CLASSES):
    correct = cm[i, i]
    total = cm[i].sum()
    recall = correct / total if total > 0 else 0
```

For each author, the code computes recall:

$$
\text{recall}
=
\frac{\text{correct predictions for that author}}
{\text{total true examples of that author}}
$$

In the code:

```python
correct = cm[i, i]
```

means the number of correctly predicted examples for that author.

```python
total = cm[i].sum()
```

means the total number of validation examples whose true class is that author.

So recall answers:

> Of all texts truly written by this author, how many did the model correctly identify?

Then:

```python
recall_df = pd.DataFrame(recall_rows)
display(recall_df)
```

shows a table like:

| author | correct | total | recall |
| ------ | ------: | ----: | -----: |
| EAP    |     ... |   ... |    ... |
| MWS    |     ... |   ... |    ... |
| HPL    |     ... |   ... |    ... |

This helps us see whether one author is harder for the model than the others.

## 5. Find the most common confusions

```python
confusion_rows = []
```

This stores all wrong true-to-predicted author pairs.

Then:

```python
for i, true_label in enumerate(CLASSES):
    for j, pred_label in enumerate(CLASSES):
        if i != j:
```

The code loops over every true-author and predicted-author pair.

The condition:

```python
if i != j:
```

means it ignores correct predictions and keeps only mistakes.

For each mistake type, it stores:

```python
{
    "true_author": AUTHORS[true_label],
    "predicted_author": AUTHORS[pred_label],
    "count": cm[i, j]
}
```

For example:

```text
true_author = EAP
predicted_author = MWS
count = 42
```

means:

> 42 EAP texts were incorrectly predicted as MWS.

Then:

```python
confusion_df = (
    pd.DataFrame(confusion_rows)
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)
```

This sorts the confusion pairs from most common to least common.

Finally:

```python
display(confusion_df.head(6))
```

shows the top 6 confusion types.

This helps identify whether errors are symmetric or one-directional. For example, maybe the model often predicts:

```text
true HPL → predicted EAP
```

but not as often:

```text
true EAP → predicted HPL
```


## 6. Identify incorrect predictions

```python
error_mask = y_valid != valid_predictions
```

This creates a Boolean mask:

```text
True  = wrong prediction
False = correct prediction
```

So if the model predicted incorrectly for a validation sample, that row is selected.


## 7. Build a table of mistaken examples

```python
error_examples = valid_texts.loc[error_mask, ["text"]].copy()
```

This selects only the validation texts where the model made a mistake.

Then it adds the true author:

```python
error_examples["true_author"] = [AUTHORS[c] for c in np.asarray(y_valid)[error_mask]]
```

and the predicted author:

```python
error_examples["predicted_author"] = [AUTHORS[c] for c in np.asarray(valid_predictions)[error_mask]]
```

So each mistaken example now has:

```text
text
true_author
predicted_author
```

## 8. Add model confidence

```python
error_examples["model_confidence"] = proba_holdout[error_mask].max(axis=1)
```

`proba_holdout` contains the stacked model’s predicted probabilities.

For each mistaken example, this line takes the largest predicted probability.

For example, if the model predicted:

```text
EAP = 0.05
MWS = 0.90
HPL = 0.05
```

then:

```text
model_confidence = 0.90
```

If the true author was actually EAP, then this is a **high-confidence mistake**.

High-confidence mistakes are especially interesting because the model was not just wrong, it was strongly wrong.


<details>
<summary><span style="color:red">More on this:</span></summary>  


This line adds a new column called `predicted_author` to `error_examples`.

```python
error_examples["predicted_author"] = [
    AUTHORS[c] for c in np.asarray(valid_predictions)[error_mask]
]
```

Step by step:

```python
np.asarray(valid_predictions)
```

converts `valid_predictions` into a NumPy array.

`valid_predictions` contains the model’s predicted numeric labels for the validation set, for example:

```python
[1, 2, 2, 3, 1, ...]
```

Then:

```python
np.asarray(valid_predictions)[error_mask]
```

selects only the predictions where the model made a mistake.

`error_mask` was created earlier:

```python
error_mask = y_valid != valid_predictions
```

So `error_mask` is `True` only for wrong predictions.

Example:

```python
valid_predictions = [1, 2, 3, 1]
y_valid           = [1, 3, 3, 2]

error_mask        = [False, True, False, True]
```

Then:

```python
np.asarray(valid_predictions)[error_mask]
```

gives:

```python
[2, 1]
```

These are the predicted labels for only the mistaken examples.

Then this part:

```python
[AUTHORS[c] for c in ...]
```

converts numeric labels into author names using:

```python
AUTHORS = {
    1: "EAP",
    2: "MWS",
    3: "HPL"
}
```

So:

```python
[2, 1]
```

becomes:

```python
["MWS", "EAP"]
```

Finally, the result is stored as a new column:

```python
error_examples["predicted_author"]
```

So each row in `error_examples` now shows **which author the model incorrectly predicted**.

In simple words:

**This line takes the model’s predictions for only the wrong validation examples, converts the numeric class labels into author names, and saves them in the `predicted_author` column.**


</details>


## 9. Add probability columns for each author

```python
for idx, c in enumerate(CLASSES):
    error_examples[f"p_{AUTHORS[c]}"] = proba_holdout[error_mask, idx]
# idx = column index in probability matrix
# c   = actual class label
```

This adds one probability column per author.

For example:

```text
p_EAP
p_MWS
p_HPL
```

So for each wrong prediction, we can inspect the full probability distribution.

This is useful because two mistakes can be very different:

```text
Example 1:
EAP = 0.45, MWS = 0.43, HPL = 0.12
```

This is a close mistake.

```text
Example 2:
EAP = 0.02, MWS = 0.95, HPL = 0.03
```

This is a confident mistake.


## 10. Sort by confidence

```python
error_examples = error_examples.sort_values("model_confidence", ascending=False)
```

This puts the most confident wrong predictions at the top.

Then:

```python
display(error_examples.head(10))
```

shows the top 10 high-confidence mistakes.

These examples are useful for qualitative analysis. We can read them and ask:

```text
Does this text genuinely sound like another author?
Is it very short?
Does it contain unusual vocabulary?
Is the label possibly ambiguous?
Are there stylistic overlaps between authors?
```


## Overall meaning

This cell analyzes where the stacked model still fails.

It gives us three levels of error analysis:

```text
1. Confusion matrix:
   Which true/predicted author pairs occur?

2. Per-author recall:
   Which author is hardest to recover correctly?

3. High-confidence mistakes:
   Which wrong predictions was the model most confident about?
```

So this cell helps move beyond aggregate metrics like log loss and accuracy. It shows **which authors are confused**, **how often**, and **what the most serious mistakes look like**.


</details>

The error analysis shows that the stacked model performs in a relatively balanced way across the three authors. Recall is highest for **EAP (0.879)**, followed by **HPL (0.864)** and **MWS (0.859)**. Although the differences are small, the results suggest that **MWS is slightly more difficult to classify correctly** compared to the other two classes.

The most common confusion is **MWS predicted as EAP (191 cases)**, followed by **EAP predicted as MWS (166 cases)**. This indicates a strong bidirectional confusion between these two authors, suggesting overlapping lexical or stylistic patterns in the feature space. The next strongest confusion is **HPL predicted as EAP (153 cases)**, indicating that Lovecraft and Poe are also sometimes confused, likely in darker or more descriptive passages.

The high-confidence mistakes are particularly informative. In several cases, the model assigns very high probability (often above **0.97**) to an incorrect author. These errors typically occur in short or stylistically generic sentences, where TF-IDF features provide limited discriminative signal. This indicates that even a strong linear stacking model can become overconfident when the available textual evidence is sparse or ambiguous.

Overall, the error analysis shows that the model is well-balanced but still struggles with stylistic overlap between authors, particularly between **MWS and EAP**, and to a lesser extent between **HPL and EAP**. This suggests that improvements would likely come from models that capture broader context and semantics beyond sparse lexical features, such as transformer-based representations.

## Results and conclusions

| Stage                                       | Evaluation         | Accuracy | Macro-F1 | Log loss |
|--------------------------------------------|-------------------|---------:|---------:|---------:|
| Word-only VW baseline                       | 70/30 holdout     | 0.8332   | 0.8323   | not reported |
| Rich VW (words + punctuation + char n-grams)| 70/30 holdout    | 0.8553   | 0.8552   | not reported |
| Tuned TF-IDF 3-model average                | 70/30 holdout     | 0.8601   | 0.8596   | 0.3843 |
| Tuned stacked model with fold averaging     | 70/30 holdout     | 0.8687   | 0.8689   | 0.3504 |
| Final stacked model (level-2 OOF estimate)  | full training data| 0.8830   | 0.8834   | 0.3167 |

Across the 70/30 holdout experiments, performance improves consistently as the modeling pipeline becomes more expressive. The word-only VW baseline already achieves strong results, and adding punctuation and character n-gram features further improves performance, indicating that stylistic signals are highly informative for authorship attribution.

The TF-IDF-based ensemble improves probabilistic performance, particularly in terms of log loss, showing the benefit of combining multiple linear classifiers with different inductive biases.

> Inductive bias = the assumptions a model makes about the data in order to generalize. In simpler words, it is the model’s built-in “preference” for how patterns in data should look.

<details>
<summary><span style="color:red">why NB and Logistic Regression have opposite inductive biases:</span></summary>  

They are called “opposite inductive biases” because they make **different and almost reversed assumptions about how features generate the label**.

Let’s make it very clear:


**1. Naive Bayes (NB) bias**  

NB assumes:

> **Features are generated from the class**

It models:

```text
P(x | y) P(y)
```

So it asks:

> “If I already know the class, how likely is this word?”

**Key assumption:**  

* words are **conditionally independent**
* each class has its own word distribution

👉 So NB is a **generative model**


**2. Logistic Regression bias**  

Logistic Regression assumes:

> **The class is generated from the features**

It models:

```text
P(y | x)
```

So it asks:

> “Given these words, what is the probability of the class?”

### Key assumption:

* you don’t model how words are generated
* you directly learn a **decision boundary**

👉 So LR is a **discriminative model**


**Why they are “opposite”**  

| Aspect    | Naive Bayes                      | Logistic Regression           |   
| --------- | -------------------------------- | ----------------------------- |
| Direction | $P(x\| y)$                       | $P(y \| x)$                   |
| View      | generative                       | discriminative                |
| Question  | “how does class generate words?” | “how do words predict class?” |
| Bias      | strong independence assumption   | no independence assumption    |
| Behavior  | counts + probabilities           | linear boundary               |


**Intuition**  

Imagine a sentence:

> “dark castle ghost”

**NB thinks:**  

* “Which class tends to generate these words?”
* it builds a **word distribution per author**
 
**LR thinks:**  

* “Which class is best separated by these words?”  
* it builds a **separating boundary in feature space**  

**Why this leads to “opposite inductive biases”**  

Because:

* NB = **model data generation process**
* LR = **model decision rule directly**

So they “look at the problem from opposite directions”:

```text
NB: class → words
LR: words → class
```

**Why combining them works (important for your project)**  

They make different mistakes:

* NB is sensitive to independence assumption
* LR is sensitive to feature correlations

So when you combine them:

> their errors cancel out → better ensemble

**summary**  

> Naive Bayes and Logistic Regression have opposite inductive biases because NB models how classes generate features, while LR models how features directly determine the class.

----


**Why NB works surprisingly well for text**  

Even though Naive Bayes has a strong (and unrealistic) assumption:

> features are conditionally independent

it still works very well for text because:

**1. Bag-of-words fits NB perfectly**  

Text in TF-IDF / BoW is already:

* sparse
* count-based
* orderless

So NB’s assumption is not too damaging.

**2. Word presence is extremely informative**  

In authorship:

* “ghost”, “dark”, “castle” → strong signal for HPL/EAP
* “love”, “passion” → strong signal for MWS

So even simple counting works well.

**3. Independence assumption becomes “useful simplification”**  

NB assumes:

```text id="nb1"
P(x | y) = Π P(x_j | y)
```

This is false in reality, BUT:

* it reduces variance
* makes learning very stable
* works well in high-dimensional sparse spaces


**Why Logistic Regression works differently**  

Logistic Regression:

* does NOT assume independence
* directly learns:

```text id="lr1"
P(y | x)
```

So:

* better at handling correlated words
* but needs more data to be stable

**Why NB + LR combination is powerful**  

They make different errors:

| Model | Strength               | Weakness                   |
| ----- | ---------------------- | -------------------------- |
| NB    | strong with rare words | ignores correlation        |
| LR    | handles correlations   | can overfit sparse signals |

So:

```text id="combo"
NB catches strong word signals
LR corrects correlation mistakes
```

**Now: What is NB-SVM (our nbsvm model)?**

This is the key part of our notebook.

NB-SVM is:

> a hybrid of Naive Bayes + Logistic Regression

**Step 1: compute NB log-odds weights**  

For each word:

```text id="nb2"
r_j = log ( P(word_j | class) / P(word_j | not class) )
```

This measures:

> how strongly a word pushes toward a class

**Step 2: reweight input features**  

```text id="nb3"
x' = x ⊙ r
```

Meaning:

* multiply each word count by its NB importance
* “amplify important words”

**Step 3: apply Logistic Regression**  

Then LR is trained on:

```text id="nb4"
x'
```

So final model:

```text id="nb5"
NB feature weighting → LR classifier
```

**Why NB-SVM is so strong**  

It combines:

**NB advantage:**  

✔ strong word importance signal
✔ good with sparse rare words

> Naive Bayes works well on text because it treats each word as an independent vote for a class and assigns strong class-specific weights even to rare words using smoothed probability estimates.

**LR advantage:**  

✔ robust decision boundary
✔ handles correlations

So NB-SVM:

> uses NB as a feature engineering step + LR as classifier

**intuition**  

> NB-SVM works because Naive Bayes extracts “word importance signals”, and Logistic Regression learns the best decision boundary using those enhanced signals.

**Final mental model**  

```text id="final"
NB = “which words belong to which class?”
LR = “how do I separate classes?”
NB-SVM = “use NB to build better features, then use LR to classify”
```

</details>


The stacked model achieves the best performance on the holdout split, reaching **0.8687 accuracy**, **0.8689 macro-F1**, and **0.3504 log loss**. Compared with simple averaging, stacking improves both classification and probabilistic performance. The main improvement comes from the meta-learner, which learns how to combine base-model probability outputs in a data-driven way rather than using uniform weights.

For the final submission pipeline, the base models are retrained on the full labeled dataset in order to generate the final test-set predictions. However, the evaluation of the meta-learner is performed separately using cross-validation on stacking features derived from out-of-fold predictions of the base models.

Each training sample is predicted by a base model that was trained without using that sample, following a cross-validation procedure. In this setup, the base-model predictions used for meta-training are fixed (out-of-fold), and are not retrained during the meta-evaluation stage.

Therefore, this evaluation measures only the generalization performance of the meta-learner on these fixed stacking features, rather than the full end-to-end pipeline including feature extraction and base-model training.

For this reason, it should be interpreted as an internal consistency check rather than a fully nested cross-validation estimate of the complete system.

The level-2 OOF results obtained from this setup are **0.8830 accuracy**, **0.8834 macro-F1**, and **0.3167 log loss**, as shown in the results table.

### Kaggle leaderboard

| Submission                | Private score | Public score |
|--------------------------|--------------:|-------------:|
| Final stacked model      | 0.30414       | 0.33621      |

The Kaggle results are broadly consistent with the internal evaluation. The private score is slightly better than the level-2 OOF estimate, while the public score is slightly higher, but both remain in the same general range. This suggests that the validation strategy provides a reasonably reliable estimate of generalization performance, although it is not a strictly unbiased estimator of the final test distribution.

### How far classical methods go

The results demonstrate that classical sparse-text models are highly competitive for authorship attribution. Word n-grams, character n-grams, Naive Bayes variants, linear classifiers, and stacking together produce a strong pipeline without requiring deep learning methods.

The error analysis shows that remaining mistakes are not concentrated in a single weak class. Instead, errors arise primarily from stylistic overlap between authors, particularly between MWS and EAP, and to a lesser extent between HPL and EAP. These errors typically occur in short or stylistically ambiguous sentences where sparse lexical features provide limited contextual information.

A natural next step is to compare this classical pipeline with contextual models such as transformer-based architectures, to evaluate whether richer semantic representations can reduce the remaining author confusions that linear sparse-text models still struggle with.

## Appendix: foundational NLP representations

To place the main results in context, I also evaluate several foundational text representation methods on the same holdout split. The goal is not only to compare performance, but also to understand which representations are most suitable for authorship attribution.

Authorship identification is primarily a **style-based classification problem**. The most informative signals often come from word choice, function words, punctuation patterns, spelling variation, and character-level structure. For this reason, sparse count-based representations are often particularly effective.

**Count-based representations**

* **Bag of Words:** represents text using raw word or n-gram counts. It is simple but often highly effective for text classification tasks.
* **TF-IDF:** extends Bag of Words by downweighting very frequent terms and emphasizing discriminative terms. It forms the core representation used in the best-performing models in this work.
* **BM25:** a ranking-based alternative to TF-IDF. Since it is not a native classifier, I use it as a nearest-neighbour retrieval method: for each validation sentence, the most similar training sentences are retrieved using BM25 similarity, and predictions are made based on their labels.

**Embedding representations**

* **Word2Vec:** learns dense word embeddings that capture semantic relationships. Sentence representations are obtained by averaging word vectors.
* **FastText:** extends Word2Vec by incorporating character n-grams, which improves handling of rare and unseen words.

Overall, I expect sparse n-gram models to outperform averaged embedding approaches. While embeddings capture semantic meaning, authorship attribution relies heavily on surface-level stylistic signals, which can be diluted when word vectors are averaged into a single sentence representation.

> These appendix models are intended as baseline representations rather than fully optimized systems. Hyperparameters are kept simple and fixed, while the main optimized model in this notebook is the stacked TF-IDF pipeline described earlier.

In [71]:
# Appendix helpers: Bag of Words, BM25, Word2Vec, and FastText.
# gensim is optional; if it is not installed, run `pip install gensim` once before this section.
import io
import contextlib

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from gensim.models import Word2Vec, FastText # Word2Vec, FastText are embedding models from the gensim library that learn vector representations of words based on their context in a corpus. Word2Vec uses a shallow neural network to learn embeddings by predicting surrounding words (context) given a target word (skip-gram) or predicting a target word given its context (CBOW). FastText extends Word2Vec by representing words as bags of character n-grams, allowing it to generate embeddings for out-of-vocabulary words and capture subword information. In this code, these models are used to create dense vector representations of text data, which can then be used for classification tasks.


@contextlib.contextmanager
def quiet_stderr():
    """
    Suppress harmless gensim / BLAS stderr messages that can appear in some environments.
    """
    with contextlib.redirect_stderr(io.StringIO()):
        yield


def simple_tokenize(text):
    """
    Lowercase tokenizer that keeps alphabetic words and apostrophes.
    """
    return re.findall(r"[a-z']+", str(text).lower())


def evaluate_probability_model(name, y_true, proba, rows, note=""):
    """
    Store accuracy, macro-F1, and log loss for models that output probabilities.
    """
    y_true = np.asarray(y_true)
    pred = CLASSES[np.argmax(proba, axis=1)]

    rows.append({
        "representation": name,
        "accuracy": accuracy_score(y_true, pred),
        "macro_F1": f1_score(y_true, pred, average="macro"),
        "log_loss": log_loss(y_true, proba, labels=CLASSES),
        "note": note
    })

    print(
        f"{name:<28}"
        f" acc={rows[-1]['accuracy']:.4f}"
        f" macro-F1={rows[-1]['macro_F1']:.4f}"
        f" log loss={rows[-1]['log_loss']:.4f}"
    )


def class_prior_proba(y):
    """
    Return class-prior probabilities in CLASSES order.
    """
    y = np.asarray(y)
    counts = np.bincount(y, minlength=int(CLASSES.max()) + 1)[CLASSES]
    return counts / counts.sum()


appendix_results = []

<details>
<summary><span style="color:red">More on this cell:</span></summary>  

**1. What this code is doing**  

This section is building a **benchmarking framework for NLP representations** in our appendix.

Goal:  
Compare different text representations using:

* Accuracy
* Macro-F1
* Log loss

**Imports**  

```python
from gensim.models import Word2Vec, FastText
```

* loads embedding models
* Word2Vec = word meaning vectors
* FastText = Word2Vec + subword info


### 🔹 `quiet_stderr()`

```python
@contextlib.contextmanager
def quiet_stderr():
```

#### What it does:

Suppresses annoying warning messages from libraries like gensim.

👉 Why needed?

* Word2Vec / FastText often print logs
* keeps notebook clean

### 🔹 `simple_tokenize(text)`

```python
re.findall(r"[a-z']+", str(text).lower())
```

#### What it does:

* lowercases text
* keeps only:

  * letters (a–z)
  * apostrophes (')

#### Example:

```text
"I’m happy!" → ["i", "m", "happy"]
```

👉 Purpose:
Standard preprocessing for:

* Word2Vec
* FastText
* BM25


### 🔹 `evaluate_probability_model(...)`

This is our **evaluation function**

#### Input:

* true labels (`y_true`)
* predicted probabilities (`proba`)

#### Steps:

##### 1. Convert probabilities → class predictions

```python
pred = CLASSES[np.argmax(proba, axis=1)]
```

👉 chooses highest probability class


##### 2. Compute metrics

```python
accuracy_score(...)
f1_score(...)
log_loss(...)
```

So we measure:

* correctness (accuracy)
* balance across classes (F1)
* probability quality (log loss)

##### 3. Store results

```python
rows.append({...})
```

👉 builds a results table for comparison


##### 4. Print summary

```python
print(...)
```


### 🔹 `class_prior_proba(y)`

```python
counts = np.bincount(...)
```

#### What it does:

Computes:

> P(class)

Example:

| Author | probability |
| ------ | ----------- |
| EAP    | 0.4         |
| MWS    | 0.35        |
| HPL    | 0.25        |

👉 This is used for baseline / smoothing models (like NB or BM25 voting).

### 🔹 `appendix_results = []`

Just an empty list to store all model results.


## 2. What each representation means

Now the important conceptual part.

### 1. Bag of Words (BoW)

#### Idea:

> “A text is just a collection of word counts”


#### How it works:

Each document becomes a vector:

```text
["ghost", "dark", "castle"]
```

→

```text
ghost=2, dark=1, castle=1
```


#### Key properties:

✔ ignores word order
✔ uses frequency
✔ very sparse
✔ very strong for style tasks


#### Why it works well here:

Because authorship depends on:

* word choice
* function words
* repetition patterns


### 2. BM25

#### Idea:

> “Better version of TF-IDF for ranking documents”

#### Formula intuition:

BM25 scores:

* word importance
* document length normalization
* term saturation


#### Key difference from BoW:

| BoW                     | BM25              |
| ----------------------- | ----------------- |
| raw counts              | weighted counts   |
| no normalization        | length-normalized |
| used for classification | used for ranking  |


#### In our code:

We use BM25 as:

> nearest-neighbor classifier

So:

1. compute similarity
2. find closest training sentences
3. vote by author

### 3. Word2Vec

#### Idea:

> “Words have meaning based on context”


#### How it works:

Learns vector:

```text
king - man + woman ≈ queen
```

#### Key idea:

Each word → dense vector:

```text
ghost → [0.21, -0.44, 0.88, ...]
```

#### In our pipeline:

We:

1. get word vectors
2. average them into sentence vector

```text
sentence = mean(word vectors)
```

#### Problem:

Averaging causes:

* loss of word order
* loss of style signals

👉 important for authorship


### 4. FastText

#### Idea:

> “Words are made of character pieces”


#### Difference from Word2Vec:

Instead of only words:

FastText uses:

* subwords (character n-grams)

Example:

```text
"ghost" → "gh", "hos", "ost"
```

#### Advantages:

✔ handles rare words
✔ handles misspellings
✔ better generalization

#### In our results:

Still weaker because:

> semantic similarity ≠ stylistic identity


#### Final comparison

| Method       | Type          | Strength           | Weakness             |
| ------------ | ------------- | ------------------ | -------------------- |
| Bag of Words | sparse counts | captures style     | ignores meaning      |
| BM25         | retrieval     | good matching      | not classifier       |
| Word2Vec     | embeddings    | semantic meaning   | loses style          |
| FastText     | embeddings    | subword robustness | still semantic-heavy |


**Intuition**  

> Sparse methods capture stylistic patterns, while embedding methods capture semantic meaning; authorship attribution is primarily a stylistic task.

</details>

In [72]:
# Count-based family.

# 1. Bag of Words with Logistic Regression.
bow = CountVectorizer(
    ngram_range=(1, 2),
    min_df=2
)

X_bow_tr = bow.fit_transform(train_texts_part["text"])
X_bow_va = bow.transform(valid_texts["text"])

bow_lr = LogisticRegression(C=10, max_iter=3000)
bow_lr.fit(X_bow_tr, y_part)

proba_bow = align_proba(bow_lr, X_bow_va)

evaluate_probability_model(
    "Bag of Words + LR",
    y_valid,
    proba_bow,
    appendix_results,
    note="word count features with Logistic Regression"
)


# 2. BM25 as a nearest-neighbour retriever.
# BM25 is a ranking function, not a classifier. Here we convert it into a classifier
# by finding the top-K most similar training sentences and taking a weighted author vote.
bm25_vectorizer = CountVectorizer(min_df=2)

D_tr = bm25_vectorizer.fit_transform(train_texts_part["text"]).astype(np.float32)
D_va = bm25_vectorizer.transform(valid_texts["text"]).astype(np.float32)

y_part_arr = np.asarray(y_part)
n_docs, vocab_size = D_tr.shape

doc_freq = np.asarray((D_tr > 0).sum(axis=0)).ravel()
idf = np.log(1 + (n_docs - doc_freq + 0.5) / (doc_freq + 0.5))

doc_len = np.asarray(D_tr.sum(axis=1)).ravel()
avg_doc_len = doc_len.mean()

k1 = 1.5
b = 0.75

coo = D_tr.tocoo()

bm25_weights = (
    idf[coo.col]
    * (coo.data * (k1 + 1))
    / (coo.data + k1 * (1 - b + b * doc_len[coo.row] / avg_doc_len))
)

bm25_docs = sp.csr_matrix(
    (bm25_weights, (coo.row, coo.col)),
    shape=(n_docs, vocab_size)
)

query_terms = (D_va > 0).astype(np.float32).tocsr() # this line means that we are creating a sparse matrix representation of the validation documents (D_va) where each entry is 1 if the term is present in the document and 0 otherwise. The resulting query_terms matrix will have the same shape as D_va, but with binary values indicating the presence or absence of terms in each validation document. This binary representation is used to compute similarity scores between validation documents and training documents using the BM25 weighting scheme, allowing us to retrieve the most relevant training documents for each validation document based on shared terms.

K = 15
prior = class_prior_proba(y_part_arr)
proba_bm25 = np.zeros((query_terms.shape[0], len(CLASSES)), dtype=np.float32)

for start in range(0, query_terms.shape[0], 500):
    end = min(start + 500, query_terms.shape[0])

    scores = np.asarray((query_terms[start:end] @ bm25_docs.T).todense())

    k_eff = min(K, scores.shape[1])
    topk = np.argpartition(-scores, kth=k_eff - 1, axis=1)[:, :k_eff]

    for row in range(topk.shape[0]):
        neighbour_idx = topk[row]
        weights = scores[row, neighbour_idx]

        if weights.sum() <= 0:
            proba_bm25[start + row] = prior
            continue

        weighted_counts = np.bincount(
            y_part_arr[neighbour_idx],
            weights=weights,
            minlength=int(CLASSES.max()) + 1
        )[CLASSES]

        if weighted_counts.sum() <= 0:
            proba_bm25[start + row] = prior
        else:
            proba_bm25[start + row] = weighted_counts / weighted_counts.sum()

proba_bm25 = np.clip(proba_bm25, 1e-15, 1 - 1e-15)
proba_bm25 = proba_bm25 / proba_bm25.sum(axis=1, keepdims=True)

evaluate_probability_model(
    "BM25 k-NN retriever",
    y_valid,
    proba_bm25,
    appendix_results,
    note="BM25 used off-label as weighted nearest-neighbour classification"
)

Bag of Words + LR            acc=0.8073 macro-F1=0.8058 log loss=0.5636
BM25 k-NN retriever          acc=0.7692 macro-F1=0.7680 log loss=0.6697


<details>
<summary><span style="color:red">More on this cell:</span></summary>  

This is a **very important block in our appendix**, because it shows the *difference between “simple linear text models” and “retrieval-based models”*. 

### 1. Big picture: what this code is doing

This section compares:

-  ✔ 1. Bag of Words + Logistic Regression (classifier)  

- ✔ 2. BM25 k-NN (retrieval-based classifier)

Both use **count-based text representations**, but behave very differently.

### PART 1: Bag of Words + Logistic Regression  

#### Step 1: Convert text → BoW features  

```python
CountVectorizer(ngram_range=(1, 2), min_df=2)
```

##### This means:

* unigrams (words)
* bigrams (word pairs)
* ignore rare terms (`min_df=2`)


##### Example:

Sentence:

```text
"dark castle appears"
```

Becomes features like:

```text
dark
castle
appears
dark castle
castle appears
```


#### Step 2: Transform data

```python
X_bow_tr = bow.fit_transform(...)
X_bow_va = bow.transform(...)
```

* train → learn vocabulary
* validation → same vocabulary applied

#### Step 3: Train Logistic Regression

```python
LogisticRegression(C=10)
```

##### What it learns:

A weight for each n-gram:

```text
score = Σ w_i x_i 
```

This is usually written more compactly as:

```text
score = w^T x + b
```

Then:

```text
P(class | x) = 1 / (1 + exp(-score))
```

#### Step 4: Align probabilities

```python
align_proba(...)
```

Ensures class order is consistent.


#### Step 5: Evaluate

```python
evaluate_probability_model(...)
```

Computes:

* Accuracy
* Macro-F1
* Log loss

#### Interpretation

👉 This model is:

* discriminative
* linear
* stable
* strong for text style


### PART 2: BM25 k-NN Retriever

This is much more interesting.

#### Key idea

BM25 is NOT a classifier.

It is:

> a ranking function for document similarity

We convert it into a classifier using **nearest neighbors voting**.

#### Step 1: Build BoW matrix

```python
D_tr = CountVectorizer(...)
```

Same as BoW but raw counts.

#### Step 2: Compute IDF

```python
idf = log(...)
```

#### Meaning:

* rare words → high weight
* common words → low weight

#### Step 3: Document length normalization

```python
doc_len
avg_doc_len
```

Used to prevent bias toward long texts.


#### Step 4: BM25 weighting formula

```python
bm25_weights = idf * (TF part)
```

##### This is the core BM25 idea:

```text
score = TF × IDF × length normalization
```

#### Step 5: Build BM25 document matrix

```python
bm25_docs = csr_matrix(...)
```

Now each document is represented in BM25 space.


#### Step 6: Convert validation texts

```python
query_terms = (D_va > 0)
```

So validation becomes sparse binary queries.  

#### Step 7: Similarity computation

```python
scores = query_terms @ bm25_docs.T
```

👉 This computes:

> how similar each test sentence is to each training sentence


#### Step 8: Top-K nearest neighbors

```python
topk = argpartition(...)
```

We select:

> K most similar training documents

Here:

```python
K = 15
```

#### Step 9: Weighted voting

For each test sample:

```python
weighted_counts = np.bincount(...)
```

Meaning:

* each neighbor votes for its author
* weight = similarity score


#### Step 10: Convert to probabilities

```python
proba_bm25 = weighted_counts / sum
```

So final output becomes:

> probability distribution over authors  


#### Step 11: fallback cases

If similarity is zero:

```python
use prior probabilities
```

#### Step 12: final normalization

```python
clip + normalize
```

Ensures valid probability distribution.


#### Interpretation of BM25 model

This model is:

> a **retrieval-based classifier**

NOT a learned model.


### Key difference: BoW vs BM25

| Aspect      | BoW + LR             | BM25 k-NN         |
| ----------- | -------------------- | ----------------- |
| Type        | discriminative model | retrieval model   |
| learning    | learns weights       | no learning       |
| decision    | linear boundary      | nearest neighbors |
| stability   | high                 | medium            |
| probability | learned              | heuristic voting  |


#### Why BM25 performs worse

Because:

* no true learning of decision boundary
* depends on similarity structure
* sensitive to local matches only


#### Final intuition

-  ✔ BoW + LR:

> “learn global patterns of authorship”

- ✔ BM25:

> “find similar sentences and copy their labels”

**Summary**  

> Bag of Words + Logistic Regression is a learned linear classifier over sparse features, while BM25 is a retrieval-based k-NN method that converts document similarity into class probabilities through weighted voting.

</details>

In [73]:
# Embedding family.

train_tokens = [simple_tokenize(text) for text in train_texts_part["text"]]
valid_tokens = [simple_tokenize(text) for text in valid_texts["text"]]

# IDF weights are used when pooling word vectors so common words do not dominate.
idf_vectorizer = TfidfVectorizer(
    tokenizer=simple_tokenize,
    token_pattern=None,
    lowercase=False,
    min_df=2
)

idf_vectorizer.fit(train_texts_part["text"])

idf_weight = {
    word: idf_vectorizer.idf_[idx]
    for word, idx in idf_vectorizer.vocabulary_.items()
}


def document_vectors(model, tokenized_docs):
    """
    Create one document vector per text by taking an IDF-weighted average
    of the word vectors in that text.
    """
    vectors = np.zeros((len(tokenized_docs), model.vector_size), dtype=np.float32)

    for i, tokens in enumerate(tokenized_docs):
        doc_vecs = []
        doc_weights = []

        for token in tokens:
            try:
                token_vec = model.wv[token]
            except KeyError:
                continue

            doc_vecs.append(token_vec)
            doc_weights.append(idf_weight.get(token, 1.0))

        if doc_vecs:
            vectors[i] = np.average(doc_vecs, axis=0, weights=doc_weights)

    return vectors


def train_embedding_classifier(name, embedding_model):
    """
    Train Logistic Regression on pooled document vectors.
    """
    X_embed_tr = document_vectors(embedding_model, train_tokens)
    X_embed_va = document_vectors(embedding_model, valid_tokens)

    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=10, max_iter=3000)
    )

    clf.fit(X_embed_tr, y_part)
    proba = align_proba(clf, X_embed_va)

    evaluate_probability_model(
        name,
        y_valid,
        proba,
        appendix_results,
        note="IDF-weighted average of word vectors with Logistic Regression"
    )


with quiet_stderr():
    word2vec_model = Word2Vec(
        train_tokens,
        vector_size=200,
        window=5,
        min_count=2,
        workers=1,
        epochs=30,
        seed=17
    )

    fasttext_model = FastText(
        train_tokens,
        vector_size=200,
        window=5,
        min_count=2,
        workers=1,
        epochs=30,
        seed=17,
        min_n=3,
        max_n=5
    )

    train_embedding_classifier("Word2Vec + LR", word2vec_model)
    train_embedding_classifier("FastText + LR", fasttext_model)


appendix_results_df = (
    pd.DataFrame(appendix_results)
    .sort_values("log_loss", ascending=True)
    .reset_index(drop=True)
)

display(appendix_results_df)

Word2Vec + LR                acc=0.7337 macro-F1=0.7328 log loss=0.6656
FastText + LR                acc=0.7126 macro-F1=0.7111 log loss=0.7077


,representation,accuracy,macro_F1,log_loss,note
0,Bag of Words + LR,0.807286,0.805849,0.563611,word count features with Logistic Regression
1,Word2Vec + LR,0.733742,0.732832,0.665585,IDF-weighted average of word vectors with Logi...
2,BM25 k-NN retriever,0.769152,0.767960,0.669721,BM25 used off-label as weighted nearest-neighb...
3,FastText + LR,0.712632,0.711086,0.707685,IDF-weighted average of word vectors with Logi...


<details>
<summary><span style="color:red">More on this cell:</span></summary>  

This is our **Embedding family pipeline**, and it’s actually the most conceptually important “representation learning” part of our appendix.


### 1. Big picture (what this code is doing)

This section builds:

> A document representation using **word embeddings + TF-IDF weighting**, then trains a classifier on top.

Pipeline:

```text id="p8v2wq"
Text → tokens → word embeddings → weighted averaging → document vector → Logistic Regression
```

#### STEP 1: Tokenization

```python id="k3m2qn"
train_tokens = [simple_tokenize(text) for text in train_texts_part["text"]]
```

##### What happens:

Each sentence becomes a list of words:

```text id="t9m1wx"
"I saw the ghost" → ["i", "saw", "the", "ghost"]
```

#### STEP 2: IDF weighting (important idea)

```python id="idv3np"
TfidfVectorizer(...)
```

We are NOT using TF-IDF for classification here.

👉 We only use it to compute:

> how important each word is globally


##### IDF formula idea:

```text id="idf1"
idf(word) = log(N / df(word))
```
idf(word) = importance of a word in distinguishing documents  
N: Total number of documents (texts)  
df(word): Document Frequency. Means: how many documents contain this word  
ratio N / df(word) measures how rare a word is  
log(N / df(word)): This just compresses large values so they don’t explode.  

##### Meaning:

| word    | IDF  |
| ------- | ---- |
| “the”   | low  |
| “ghost” | high |


#### STEP 3: Build IDF dictionary

```python id="map1"
idf_weight = {word: idf_value}
```

So every word gets a weight.


#### STEP 4: Document vector creation (core part)

```python id="docvec"
def document_vectors(model, tokenized_docs):
```

This is the most important function.

##### What it does:

For each document:

> combine word vectors into ONE sentence vector


##### Step-by-step:

##### 1. initialize matrix

```python id="init"
vectors = zeros(num_docs, embedding_dim)
```

##### 2. loop over tokens

```python id="loop1"
for token in tokens:
```

##### 3. get word embedding

```python id="emb1"
token_vec = model.wv[token]
```

So each word becomes:

```text id="vec"
ghost → [0.2, -0.1, 0.7, ...]
```
> Word embedding numbers are learned coordinates that place words in a space where similar contexts → closer positions.

##### 4. get weight (IDF)

```python id="w1"
doc_weights.append(idf_weight.get(token, 1.0))
```

So:

* rare words → high weight
* common words → low weight

##### 5. weighted averaging

```python id="avg"
vectors[i] = weighted_average(word_vectors)
```
Final formula (VERY IMPORTANT)

Each document becomes:

```text id="final_emb"
d = (Σ idf(w_i) · v(w_i)) / Σ idf(w_i)
```

Where:

* $v(w_i)$ = word embedding
* $idf(w_i)$ = importance weight


#### STEP 5: Train classifier

```python id="clf1"
LogisticRegression(C=10)
```

We train:

> linear classifier on top of dense vectors


##### Full model:

```text id="full_model"
x_doc → embedding vector → logistic regression → probability
```

#### STEP 6: Standardization

```python id="std"
StandardScaler()
```

Why?

Because embeddings:

* may have different scales
* LR performs better with normalized inputs

#### STEP 7: Word2Vec training

```python id="w2v"
Word2Vec(...)
```

##### What it learns:

Each word vector is trained so that:

> words in similar context → close vectors


##### Example:

```text id="w2v_ex"
king ≈ queen
ghost ≈ spirit
dark ≈ shadow
```

#### STEP 8: FastText training

```python id="ft"
FastText(...)
```

##### Difference:

Instead of only words:

👉 it also learns subwords

Example:

```text id="sub"
"ghost" → "gh", "hos", "ost"
```

##### Why this matters:

✔ handles rare words
✔ handles misspellings
✔ better generalization


#### STEP 9: Training both models

```python id="train_all"
train_embedding_classifier(...)
```

We run same pipeline twice:

* Word2Vec + LR
* FastText + LR

#### STEP 10: Results table

```python id="results"
appendix_results_df
```

Sorts models by performance.

### 2. Conceptual explanation of the whole family


**A. Word2Vec**  

**Idea:**   

> “Words are defined by their context”

**learns:**  

```text id="w2v_math"
v(word) # the vector representation of a word
```

so that:

* similar context → similar vectors

**B. FastText**  

**Idea:**  

> “Words are made of character pieces”

So representation includes:

```text id="fasttext"
word = sum(subword vectors)
```

**C. Embedding document model (our system)** 
 
**Final representation:**  

```text id="doc_final"
document = weighted average of word vectors
```

Then:

```text id="clf_final"
P(y | document)
```


**Key insight**   

| Method               | Type               | Strength          | Weakness       |
| -------------------- | ------------------ | ----------------- | -------------- |
| Word2Vec             | semantic           | meaning           | loses style    |
| FastText             | semantic + subword | rare words        | still semantic |
| IDF-weighted pooling | improved semantic  | reduces stopwords | still averaged |
| BoW                  | sparse             | style signals     | no semantics   |

**Final intuition**  

> Embedding-based methods compress an entire document into a single dense vector, which preserves semantic meaning but tends to blur fine-grained stylistic patterns that are crucial for authorship attribution.

**Summary**  

> This section builds IDF-weighted averaged word embeddings using Word2Vec and FastText, and evaluates them with Logistic Regression to compare semantic representations against sparse count-based models.

</details>

<details>
<summary><span style="color:red">More on semantic type vs sparse type:</span></summary>  

**1. Sparse representations (Sparse type)**  

**Idea:**  

> Represent text by **counts of words or n-grams**

**Examples:**   

* Bag of Words
* TF-IDF
* BM25

**What the vector looks like:**  

Very large vector, mostly zeros:

```text id="sparse1"
[0, 0, 3, 0, 0, 1, 0, ...]
```

Each dimension = a word in vocabulary.


**Meaning:**  

Each number means:

> “how many times this word appears”

or

> “how important this word is”


**Key properties:**  

✔ high-dimensional  
✔ sparse (mostly zeros)  
✔ interpretable  
✔ preserves word identity  
✔ strong for style tasks  

**Why it works well for authorship:**  

Because it captures:

* word choice
* punctuation habits
* function words
* writing style

**2. Semantic representations (Dense / semantic type)**  

**Idea:**  

> Represent words by their **meaning based on context**  

**Examples:**  

* Word2Vec
* FastText
* GloVe

**What the vector looks like:**   

Small dense vector:

```text id="dense1"
[0.21, -0.44, 0.87, 0.05, ...]
```

(typically 100–300 dimensions)  

> The dimensionality of an embedding vector is a user-defined parameter that sets how many hidden features the model is allowed to use to represent word meaning.

**Meaning:**  

Each dimension encodes:

> abstract semantic properties learned from context

**Key properties:**   

✔ low-dimensional  
✔ dense (no zeros)  
✔ not interpretable per dimension  
✔ captures meaning/similarity  
✔ weaker for stylistic tasks  

**Why it works less well for authorship:**  

Because averaging embeddings:

* removes word identity
* smooths style differences
* focuses on meaning, not writing pattern

**Direct comparison**  

| Feature          | Sparse (BoW / TF-IDF)      | Semantic (Word2Vec / FastText) |
| ---------------- | -------------------------- | ------------------------------ |
| Representation   | word counts                | learned vectors                |
| Dimension        | very large                 | small (100–300)                |
| Values           | mostly zeros               | dense real numbers             |
| Captures         | style + words              | meaning                        |
| Best for         | authorship, classification | semantics, similarity          |
| Interpretability | high                       | low                            |

**One-line intuition**  

* **Sparse models:** “What words does the author use?”
* **Semantic models:** “What do the words mean?”

**In this project context**  

Our results show:

> Sparse models outperform semantic models

because:

✔ authorship = style problem  
✔ style = surface-level patterns  
✔ sparse models preserve exact usage  
✔ semantic models blur those patterns  

**Final intuition**  

> Sparse = identity of words
> Semantic = meaning of words


</details>

### Survey results

| Technique                    | Family        | Accuracy | Macro-F1 | Log loss |
| ---------------------------- | ------------- | -------: | -------: | -------: |
| Bag of Words + LR            | counts        |   0.8073 |   0.8058 |   0.5636 |
| BM25 k-NN retriever          | retrieval     |   0.7692 |   0.7680 |   0.6697 |
| Word2Vec + LR                | embeddings    |   0.7337 |   0.7328 |   0.6656 |
| FastText + LR               | embeddings    |   0.7126 |   0.7111 |   0.7077 |
| Tuned TF-IDF 3-model average | main pipeline |   0.8601 |   0.8596 |   0.3843 |
| Tuned stacked model          | main pipeline |   0.8687 |   0.8689 |   0.3504 |

Two main conclusions emerge from this appendix.

First, **count-based representations are more effective than simple pooled embeddings for authorship attribution in this setting**. Bag of Words with Logistic Regression already achieves strong performance (**0.8073 accuracy**), while Word2Vec and FastText with averaged sentence embeddings perform worse, remaining in the low 0.70 range. This reflects the fact that authorship attribution relies heavily on stylistic cues such as word choice, function words, punctuation patterns, and character-level structure, which are partially lost when embeddings are averaged.

Second, **BM25 performs reasonably well as a retrieval-based method but is not naturally suited for classification**. When used as a nearest-neighbour approach, it achieves **0.7692 accuracy**, but remains below supervised linear models. Although neighbour-based voting can be converted into probabilities for evaluation, BM25 is fundamentally designed for ranking rather than calibrated classification.

These results are consistent with the initial hypothesis stated earlier: sparse n-gram representations outperform averaged embedding-based approaches in authorship attribution. Bag of Words performs strongly, while Word2Vec and FastText lag behind due to loss of stylistic information during vector averaging.

Overall, the experiments validate that authorship attribution is better modeled using surface-level linguistic signals rather than dense semantic representations. This appendix supports the main findings of the project, where sparse count-based methods and stacked linear models achieve the best performance.

The tuned stacked model achieves the best overall performance with **0.8687 accuracy** and **0.3504 log loss**, confirming that carefully engineered sparse representations remain extremely strong baselines before considering contextual transformer-based models.